# Establishing Connections to Databases

In [1]:
bridge_sfn = '0130192'

## TIMS 'doticsqlp31' Data Source

In [2]:
import pandas as pd
import sqlalchemy as db
from sqlalchemy import create_engine, MetaData, text, inspect
from sqlalchemy.engine import URL

In [3]:
connection_string = (
    r"Driver=ODBC Driver 17 for SQL Server;"
    r"Server=doticsqlp31;"
    r"Database=TIMS;"
    r"Trusted_Connection=yes;"
)

connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})

engine = create_engine(connection_url)
conn = engine.connect()

## 'DOTWarehousePrd' Data Source

In [4]:
connection_string = (
    r"Driver=SQL Server;"
    r"Server=DOTWarehousePrd;"
    r"Database=Warehouse;"
    r"Trusted_Connection=yes;"
)
connection_url = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})
engine = create_engine(connection_url)
conn = engine.connect()

In [5]:
# Get data from 1985-1990 table - Huge, don't do this.
# hist85_90DataQuery = text("SELECT * from BMS_1985_1990")

In [6]:
# hist_bridge_data = pd.read_sql(hist85_90DataQuery, conn)
# hist_bridge_data

## Oracle Load Rating Database

In [7]:
from sqlalchemy import create_engine, text
import json

In [8]:
def load_secrets(file_path):
    with open(file_path, 'r') as file:
        user_info = json.load(file)
    return user_info

In [ ]:
from pathlib import Path

file_path = Path.home() / 'secrets.json'
dane = load_secrets(file_path)

In [ ]:
# Oracle connection string using service name instead of SID
oracle_connection_string = (
    f"oracle+cx_oracle://{dane['BRR_USN']}:{dane['BRR_PASS']}@{dane['BRR_SERVER']}:{dane['BRR_PORT']}/?service_name={dane['BRR_SERVICE']}"
)

In [11]:
# Create the engine
oracle_engine = create_engine(oracle_connection_string)

In [12]:
# Establish connection
oracle_conn = oracle_engine.connect()
print("Connection successful!")

Connection successful!


In [13]:
from sqlalchemy import inspect

# Create an inspector from the engine
inspector = inspect(oracle_engine)

# Get a list of table names
tables = inspector.get_table_names()
print("Tables in the database:")
for table in tables:
    print(table)

Tables in the database:


In [14]:
query = text("""
  SELECT DISTINCT owner
  FROM all_tables
  ORDER BY owner
""")

result = oracle_conn.execute(query)
schemas = [row[0] for row in result]
print("Accessible schemas:")

for schema in schemas:
     print(schema)

Accessible schemas:
BRIDGEWARE
CTXSYS
MDSYS
ODOTREF
SYS
SYSTEM
XDB


In [15]:
schema_list = ["BRIDGEWARE", "CTXSYS", "MDSYS", "ODOTREF", "SYS", "SYSTEM", 
               "XDB"]  # Add other schemas if you want to search deeper

for schema_name in schema_list:
    print(f"\nChecking for tables in schema: {schema_name}\n")
    query = text(f"""
    SELECT table_name
    FROM all_tables
    WHERE owner = '{schema_name}'
    """)
    result = oracle_conn.execute(query)
    tables = [row[0] for row in result]
    if tables:
        print(f"Tables in schema {schema_name}:")
        for table in tables:
            print(table)
    else:
        print(f"No tables found in schema {schema_name}.")


Checking for tables in schema: BRIDGEWARE

Tables in schema BRIDGEWARE:
ABW_BRIDGE_SUB_LRFDDSN_SETTING
ABW_MA_STRUSS_ELEM_LOSS_RANGE
ABW_MCB_SEG_DECK
ABW_MATL_CONC
ABW_MATL_PS_BAR
ABW_MATL_PS_STRAND
ABW_FL_FLOORBEAM_TRAVELWAY
ABW_FLINE_MBR
ABW_MCB_TEND_PROF_DEF_DETAIL
ABW_STL_ANGLE
ABW_STL_ANGLE_CONN
ABW_STL_BEAM_ASSEMBLY
ABW_WEB_DISTRIB_LOAD
BRIDGE
COPTIONS
MULTIMEDIA
ABW_TRUSS_LINE_MBR
ABW_TRUSS_LINE_SUPPORT
ABW_TRUSSDEF_ELEMENT_CONC_LOAD
ABW_RESULTS_CONC_LS_SUMMARY
ABW_RESULTS_CONC_XSEC_PROP
ABW_LIB_MATL_TIMBER_SAWN_ITEM
ABW_SYS_LRFD_LS
ABW_SUBSDEF_MODEL_SETTING
ABW_SUPER_BR_FORCE_SUB
ABW_CONC_CURB_SIDEWALK
ABW_CONC_RAILING
ABW_CONC_RAILING_LOC
ABW_BRIDGE_DIAPHRAGM_DEF_CON
ABW_SUBSDEF_FOUND_FK
ABW_STL_BUILTUP_IBEAM_DEF
ABW_STL_LONG_STIFF
ABW_EVENT_VEHICLE_TEMPLATE
ABW_LEG_ANAL_PT_REINF_CONC
ABW_RESULTS_CRIT_LOAD_LRFD
ABW_RESULTS_DL_ACTION
ABW_RESULTS_LL_ACTION
ABW_RESULTS_LS_SUMMARY_DETAIL
ABW_RESULTS_PS_CONC_STRESS
ABW_RESULTS_RC_SERVICE_STRESS
ABW_RESULTS_SPNG_MBR_ALT_XY
ABW_RESU

In [16]:
import pandas as pd
from sqlalchemy.sql import text

# Query to fetch the entire table
query = text("SELECT * FROM BRIDGEWARE.ABW_GIRDER_MBR")

# Execute the query and fetch the data into a DataFrame
result = oracle_conn.execute(query)
df = pd.DataFrame(result.fetchall(), columns=result.keys())

In [17]:
df

,bridge_id,struct_def_id,super_struct_mbr_id,struct_def_ref_line_id,settlement_load_case_id
0,15974,1,1,5,NaN
1,15974,1,2,6,NaN
2,15974,1,3,7,NaN
3,15974,1,4,8,NaN
4,15974,1,5,9,NaN
...,...,...,...,...,...
73403,17240,1,11,15,NaN
73404,17241,1,1,5,NaN
73405,17241,1,2,6,NaN
73406,17241,1,3,7,NaN


In [18]:
def get_table(schema, table):
    query = text(f"SELECT * FROM {schema}.{table}")
    result = oracle_conn.execute(query)
    df = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    return df

### BRIDGEWARE

#### ABW_BRIDGE_SUB_LRFDDSN_SETTING

In [19]:
df = get_table("BRIDGEWARE", "ABW_BRIDGE_SUB_LRFDDSN_SETTING")
df

,bridge_id,lrfd_design_setting_id,name,descr,bridge_lrfd_factor_id,preliminary_design_setting_ind,final_design_setting_ind,dynamic_load_allow_method_type,dla_simple_fatigue_frac_impact,dla_simple_other_ls_impact,vehicle_summary_display_type,last_change_timestamp,limit_num_dsgn_load_combo_ind,cw_num_loadcombo_axial_bending,ftg_num_loadcombo_brg_capacity,lrfd_analysis_module_guid,lrfd_spec_edition_guid
0,3332,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2016-01-29 16:55:09.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
1,3717,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2016-10-28 16:40:47.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
2,5564,1,Final Design Setting (US),Final Design Setting (US),2,F,T,42301,15.0,33.0,43502,2018-11-27 18:47:28.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
3,5478,1,Final Design Setting (US),Final Design Setting (US),2,F,T,42301,15.0,33.0,43502,2018-09-25 17:49:03.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
4,13095,1,Final Design Setting (US),Final Design Setting (US),1,F,T,42301,15.0,33.0,43502,2020-05-18 17:07:42.000,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,6F9B56BA-D0F2-48E5-B139-693A46DE3052
5,18262,1,Preliminary Design Setting (US),Preliminary Design Setting (US),26,T,F,42301,15.0,33.0,43502,2024-08-15 13:11:39.103,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,70A01C5A-21D9-4C12-9A23-1FD3431D583B


#### ABW_MCB_SEG_DECK

In [20]:
df = get_table("BRIDGEWARE", "ABW_MCB_SEG_DECK")
df

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,segment_deck_id,super_ref_line_type,super_ref_line_ref_type,super_ref_line_ref_web_id
0,3958,1,1,1,1,63803,63902,None
1,4275,1,1,1,1,63803,63902,None
2,4275,2,1,1,1,63803,63902,None
3,5236,1,1,1,1,63803,63902,None
4,3496,1,1,1,1,63801,63902,None


#### ABW_MATL_CONC

In [21]:
df = get_table("BRIDGEWARE", "ABW_MATL_CONC")
df

,bridge_id,conc_id,name,descr,si_or_us_type,comp_strength_28,initial_comp_strength,thermal_exp_coeff,density_dl,density_e,...,shear_factor,std_initial_mod_of_elast,last_change_timestamp,shrinkage_coef,lrfd_mod_of_elast,lrfd_initial_mod_of_elast,splitting_tensile_strength,lrfd_max_aggregate_size,std_mod_of_rupture,lrfd_mod_of_rupture
0,752,1,Class S OH 1951-1980,Class S Cement Concrete,10401,27.579028,NaN,0.000011,2402.809922,2322.716258,...,1.0,NaN,2011-03-14 18:18:48.000000,None,25125.523576,NaN,None,NaN,NaN,3.308500
1,776,1,Class C (US),Class C cement concrete,10401,27.579032,NaN,0.000011,2402.809922,2322.716258,...,1.0,NaN,2010-10-06 13:19:31.000000,None,25125.523576,NaN,None,NaN,NaN,3.308500
2,777,1,Class C (US),Class C cement concrete,10401,27.579032,NaN,0.000011,2402.809922,2322.716258,...,1.0,NaN,2010-10-06 13:24:15.000000,None,25125.523576,NaN,None,NaN,NaN,3.308500
3,781,1,f'c 4000,Based on year of construction,10401,27.579028,24.131650,0.000011,2402.809922,2322.716258,...,1.0,NaN,2010-10-12 18:51:58.000000,None,25125.511010,NaN,None,NaN,NaN,NaN
4,789,1,f'c 3000,Based on year of construction,10401,20.684271,17.236892,0.000011,2402.809922,2322.716258,...,1.0,NaN,2019-01-08 14:49:16.000000,None,21759.330818,NaN,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16112,18507,1,Class S OH 1951-1980,Class S Cement Concrete,10401,27.579028,NaN,0.000011,2402.809922,2322.716258,...,1.0,0.000000,2024-09-19 13:56:13.996000,None,27486.282900,0.000000,None,NaN,NaN,3.309483
16113,18508,1,Class S OH 1931-1950,Class S Cement Concrete,10401,20.684271,NaN,0.000011,2402.809922,2322.716258,...,1.0,0.000000,2024-07-02 16:46:33.742000,None,25125.523578,0.000000,None,NaN,NaN,3.308500
16114,18524,1,PS 7.0 ksi 5.5 Release,release 5500psi,10401,48.263299,37.921163,0.000011,2402.809922,2402.809922,...,1.0,30999.246508,2025-05-30 15:22:19.582129,None,34970.207504,30999.241157,None,NaN,NaN,4.378035
16115,18524,2,Deck Concrete 4.5 ksi,Deck Concrete,10401,31.026407,NaN,0.000011,2402.809922,2322.716258,...,1.0,0.000000,2025-05-30 15:22:19.581337,None,28575.664911,0.000000,None,NaN,NaN,3.510237


#### ABW_MATL_PS_BAR

In [22]:
df = get_table("BRIDGEWARE", "ABW_MATL_PS_BAR")
df

,bridge_id,ps_bar_id,name,descr,bar_diameter,bar_area,bar_type,ultimate_tensile_strength,yield_strength,modulus_of_elasticity,unit_load_per_length,epoxy_coated_ind
0,3430,1,Tie Rod,"1"" Type I Plain Bar",25.4,503.2248,55201,744.633756,689.475700,206842.71,3.973465,F
1,4383,1,"1"" Type I","1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081518,206842.71,3.973465,F
2,13098,1,"1"" Type I","1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081518,206842.71,3.973465,F
3,3914,1,None,None,NaN,NaN,55201,NaN,NaN,NaN,NaN,F
4,3583,1,None,None,NaN,NaN,55201,NaN,NaN,NaN,NaN,F
5,5484,1,Tie Rod,"1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081518,206842.71,3.973465,F
6,6351,1,"1"" Type I","1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081518,206842.71,3.973465,F
7,12916,1,Tie Rod,"1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081518,206842.71,3.973465,F
8,16051,1,"1"" Type I","1"" Type I Plain Bar",25.4,503.2248,55201,1034.213550,879.081517,206842.71,3.973465,F


#### ABW_MATL_PS_STRAND

In [23]:
df = get_table("BRIDGEWARE", "ABW_MATL_PS_STRAND")
df

,bridge_id,matl_ps_strand_id,name,descr,si_or_us_type,strand_type,strand_diameter,strand_area,weight_per_length,ultimate_tensile_strength,yield_strength,mod_of_elast,transfer_length_lrfd,transfer_length_std,epoxy_coated_ind,last_change_timestamp
0,20,1,"1/2"" (7W-270) LR","Low relaxation 1/2""/Seven Wire/fpu = 270",10401,34301,12.70,98.70948,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2003-01-01 12:00:00.000000
1,21,1,"1/2"" (7W-270) LR","Low relaxation 1/2""/Seven Wire/fpu = 270",10401,34301,12.70,98.70948,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2003-01-01 12:00:00.000000
2,22,1,"1/2"" (7W-270) LR","Low relaxation 1/2""/Seven Wire/fpu = 270",10401,34301,12.70,98.70948,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2003-01-01 12:00:00.000000
3,23,1,"1/2"" (7W-270)","1/2""/Seven Wire/fpu = 270",10401,34301,12.70,98.70948,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2003-01-01 12:00:00.000000
4,19464,1,"1/2"" (7W-270) SR","Stress relieved 1/2""/Seven Wire/fpu = 270",10401,34302,12.70,98.70948,0.773858,1861.58439,1582.346731,196500.5745,762.0,635.0,F,2024-09-13 20:54:07.337984
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2978,20424,1,"1/2"" (7W-270) LR","Low relaxation 1/2""/Seven Wire/fpu = 270",10401,34301,12.70,98.70948,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2019-05-08 21:18:25.000000
2979,20427,1,"1/2"" (7W-270) SR","Stress relieved 1/2""/Seven Wire/fpu = 270",10401,34302,12.70,98.70948,0.773858,1861.58439,1582.346731,196500.5745,762.0,635.0,F,2024-10-04 00:15:56.696000
2980,20427,2,"1/2"" (7W-270K) LR 0.167 sq inch","LR 1/2""/Seven Wire/fpu = 270 special",10401,34301,12.70,107.74172,0.773858,1861.58439,1675.425951,196500.5745,762.0,635.0,F,2024-10-04 00:10:11.477000
2981,20428,1,"1/2"" (7W-270) SR","Stress relieved 1/2""/Seven Wire/fpu = 270",10401,34302,12.70,98.70948,0.773858,1861.58439,1582.346731,196500.5745,762.0,635.0,F,2025-04-08 15:32:50.657000


#### ABW_FL_FLOORBEAM_TRAVELWAY

In [24]:
df = get_table("BRIDGEWARE", "ABW_FL_FLOORBEAM_TRAVELWAY")
df

,bridge_id,struct_def_id,super_struct_mbr_id,travelway_id,dist_left_girder_travelway,travelway_width
0,8322,4,1,1,5.449885,10.05840
1,8322,4,1,2,17.489485,10.05840
2,11883,2,4,1,-0.533400,9.55039
3,13078,2,2,1,0.914400,0.99060


#### ABW_MCB_TEND_PROF_DEF_DETAIL

In [25]:
df = get_table("BRIDGEWARE", "ABW_MCB_TEND_PROF_DEF_DETAIL")
df

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,tendon_profile_id,detail_id,profile_type,inflect_point_left_percent,inflect_point_low_percent,inflect_point_right_percent,...,vert_offset_left_end_dist,vert_offset_le_meas_from_type,vert_offset_left_dist,vert_offset_l_meas_from_type,vert_offset_low_dist,vert_offset_lo_meas_from_type,vert_offset_right_dist,vert_offset_r_meas_from_type,vert_offset_right_end_dist,vert_offset_re_meas_from_type
0,5236,1,1,1,1,11,57101,NaN,NaN,NaN,...,215.9,57201,None,57201,101.6,57202,None,57201,215.9,57201
1,3958,1,1,1,1,4,57102,NaN,NaN,NaN,...,444.5,57201,None,57201,812.8,57202,None,57201,444.5,57201
2,4275,1,1,1,1,7,57103,NaN,40.0,10.0,...,855.0,57201,None,57201,380.0,57202,None,57201,584.0,57201
3,4275,1,1,1,2,1,57101,10.0,50.0,10.0,...,584.0,57201,None,57201,380.0,57202,None,57201,584.0,57201
4,4275,1,1,1,2,2,57101,10.0,50.0,10.0,...,584.0,57201,None,57201,380.0,57202,None,57201,584.0,57201
5,4275,1,1,1,2,3,57101,10.0,50.0,10.0,...,584.0,57201,None,57201,380.0,57202,None,57201,584.0,57201
6,4275,1,1,1,2,4,57104,10.0,60.0,NaN,...,584.0,57201,None,57201,380.0,57202,None,57201,855.0,57201
7,4275,2,1,1,1,7,57103,NaN,40.0,10.0,...,855.0,57201,None,57201,425.0,57202,None,57201,584.0,57201
8,4275,2,1,1,1,8,57101,10.0,50.0,10.0,...,584.0,57201,None,57201,425.0,57202,None,57201,584.0,57201
9,4275,2,1,1,1,9,57101,10.0,50.0,10.0,...,584.0,57201,None,57201,425.0,57202,None,57201,584.0,57201


#### ABW_STL_ANGLE

In [26]:
df = get_table("BRIDGEWARE", "ABW_STL_ANGLE")
df

,bridge_id,stl_shape_id,angle_size_1,angle_size_2,angle_thick,nominal_weight_or_mass,k,area,ixx,yxx,iyy,xyy,rzz,zztantheta
0,18682,2,152.4,152.4,9.5250,22.174016,NaN,2825.8008,6.409964e+06,41.148,6.409964e+06,41.1480,30.2260,NaN
1,18994,3,101.6,101.6,7.9502,12.203150,NaN,1548.3840,1.527569e+06,28.194,1.527569e+06,28.1940,19.8374,NaN
2,116,1,76.2,76.2,7.9502,9.077953,NaN,1148.3848,6.243471e+05,21.844,6.243471e+05,21.8440,14.8082,NaN
3,16402,3,76.2,76.2,7.9375,9.077953,NaN,1148.3848,6.285095e+05,21.971,6.285095e+05,21.9710,14.9606,NaN
4,18682,1,152.4,101.6,7.9502,15.328347,NaN,1954.8348,4.745038e+06,48.260,1.719036e+06,23.0632,22.1996,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8178,19148,4,101.6,101.6,7.9502,12.203150,NaN,1548.3840,1.527569e+06,28.194,1.527569e+06,28.1940,19.8374,NaN
8179,19149,1,127.0,127.0,9.5250,18.304725,NaN,2354.8340,3.646187e+06,34.798,3.646187e+06,34.7980,25.0444,NaN
8180,19149,2,101.6,101.6,9.5250,14.584252,NaN,1845.1576,1.798120e+06,28.702,1.798120e+06,28.7020,19.7866,NaN
8181,19162,3,76.2,76.2,7.9502,9.077953,NaN,1148.3848,6.243471e+05,21.844,6.243471e+05,21.8440,14.8082,NaN


#### ABW_STL_BEAM_ASSEMBLY

In [27]:
df = get_table("BRIDGEWARE", "ABW_STL_BEAM_ASSEMBLY")
df

,bridge_id,struct_def_id,spng_mbr_def_id,stl_assembly_id,stl_component_id,dist
0,17084,2,1,6,24,18.739104
1,17084,2,2,1,2,0.000000
2,17084,2,2,2,11,18.281599
3,17084,2,2,6,29,18.281599
4,17084,2,3,1,3,0.000000
...,...,...,...,...,...,...
231206,16597,1,4,5,29,29.260800
231207,16597,1,4,6,30,7.315200
231208,16597,1,4,7,31,29.260800
231209,16597,1,5,1,32,0.000000


#### COPTIONS

In [28]:
df = get_table("BRIDGEWARE", "COPTIONS")
df

,optionname,optionval,defaultval,helpid,description
0,MULTISERVER,C:\,C:\,1580,This specifies the directory path to multimedi...


#### MULTIMEDIA

In [29]:
df = get_table("BRIDGEWARE", "MULTIMEDIA")
df

,docrefkey,sequence,context,fileloc,fileref,filetype,agencytype,status,reportflag,userkey1,userkey2,userkey3,userkey4,userkey5,createdatetime,createuserkey,modtime,userkey,notes
0,6701213::15A2EBD9-5B6A-4298-AA40-E393E2E520A1,1,Virtis/Opis,X:\POR-43-1375\,DSC_1690.JPG,JPG,D,None,N,None,None,None,None,None,2006-06-21 09:16:00,VOU,None,None,None


#### ABW_TRUSS_LINE_MBR

In [30]:
df = get_table("BRIDGEWARE", "ABW_TRUSS_LINE_MBR")
df

,bridge_id,struct_def_id,super_struct_mbr_id,truss_location_type,length,truss_spacing,settlement_load_case_id,deck_crack_control_param_z,deck_exposure_factor
0,12853,6,1,31801,None,None,None,None,None


#### ABW_LIB_MATL_TIMBER_SAWN_ITEM

In [31]:
df = get_table("BRIDGEWARE", "ABW_LIB_MATL_TIMBER_SAWN_ITEM")
df

,timber_matl_id,sawn_timber_matl_item_id,timber_commercial_grade_id,timber_size_classification_id,timber_grading_rule_agency_id,unit_asd_bending_stress,unit_asd_tens_stress_parallel,unit_asd_shear_stress_parallel,unit_asd_comp_stress_perp,unit_asd_comp_stress_parallel,mod_of_elast,density,unit_lrfd_bending_stress,unit_lrfd_tens_stress_parallel,unit_lrfd_shear_stres_parallel,unit_lrfd_comp_stress_perp,unit_lrfd_comp_stress_parallel,lrfd_mod_of_elast,asd_notes,lrfd_notes
0,6,2,4,4,1,6.894757,3.964485,0.758423,6.101860,6.377650,9652.6598,800.93664,6.894757,3.964485,1.516847,6.101860,6.377650,9652.660,None,None
1,6,3,7,4,1,6.722388,3.964485,0.758423,6.101860,4.998699,8963.1841,800.93664,6.722388,3.964485,1.516847,6.101860,4.998699,8963.185,None,None
2,6,4,8,8,1,11.031611,6.550019,0.723949,6.101860,6.550019,8963.1841,800.93664,11.031610,6.550019,1.413425,6.101860,6.550019,8963.185,None,None
3,6,5,4,8,1,9.307922,4.653961,0.723949,6.101860,5.515806,8963.1841,800.93664,9.307922,4.653961,1.413425,6.101860,5.515806,8963.185,None,None
4,6,6,7,8,1,6.032912,2.930272,0.723949,6.101860,3.447379,6894.7570,800.93664,6.032912,2.930272,1.413425,6.101860,3.447378,6894.757,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,12,20,4,9,6,5.515806,3.792116,0.448159,2.309744,4.309223,8273.7084,800.93664,5.515806,3.792116,0.861845,2.309744,4.309223,8273.708,None,None
148,12,21,7,9,6,2.413165,1.551320,0.448159,2.309744,1.551320,6894.7570,800.93664,3.275010,2.240796,0.861845,2.309744,2.930272,6894.757,None,None
149,13,1,8,4,1,6.894757,3.964485,0.517107,2.895798,6.205281,10342.1355,800.93664,6.894757,3.964485,0.999740,2.895798,6.205281,10342.140,None,None
150,13,2,4,4,1,4.998699,2.930272,0.517107,2.895798,4.998699,9652.6598,800.93664,4.998699,2.930272,0.999740,2.895798,4.998699,9652.660,None,None


#### ABW_SYS_LRFD_LS

In [32]:
df = get_table("BRIDGEWARE", "ABW_SYS_LRFD_LS")
df

,sys_lrfd_ls_id,name,descr
0,1,STRENGTH-I,STRENGTH-I
1,2,STRENGTH-II,STRENGTH-II
2,3,STRENGTH-III,STRENGTH-III
3,4,STRENGTH-IV,STRENGTH-IV
4,5,STRENGTH-V,STRENGTH-V
5,6,SERVICE-I,SERVICE-I
6,7,SERVICE-II,SERVICE-II
7,8,SERVICE-III,SERVICE-III
8,9,SERVICE-IV,SERVICE-IV
9,10,FATIGUE-I,FATIGUE-I


#### ABW_SUBSDEF_MODEL_SETTING

In [33]:
df = get_table("BRIDGEWARE", "ABW_SUBSDEF_MODEL_SETTING")
df

,bridge_id,struct_def_id,model_setting_id,default_ind,num_equal_element_cap_span,num_equal_element_col_seg,footing_element_type,use_cracked_moi_ind,cracked_moi_top_col_wall_pct,cracked_moi_bot_col_wall_pct,use_long_stl_reinf_ei_calc_ind,prov_moment_rel_col_cap_ind
0,18675,5,1,None,NaN,NaN,43901,None,None,None,None,None
1,4904,3,1,None,NaN,NaN,43901,None,None,None,None,None
2,7533,2,1,None,NaN,NaN,43901,None,None,None,None,None
3,7534,3,1,None,NaN,NaN,43901,None,None,None,None,None
4,5564,3,1,None,1.0,1.0,43901,F,None,None,None,F
5,3717,2,1,None,1.0,1.0,43901,F,None,None,None,F
6,5564,4,1,None,NaN,NaN,43901,None,None,None,None,None
7,5564,5,1,None,NaN,NaN,43901,None,None,None,None,None
8,5564,6,1,None,NaN,NaN,43901,None,None,None,None,None
9,6370,2,1,None,NaN,NaN,43901,None,None,None,None,None


#### ABW_CONC_CURB_SIDEWALK

In [34]:
df = get_table("BRIDGEWARE", "ABW_CONC_CURB_SIDEWALK")
df

,bridge_id,struct_def_id,curb_id,load_case_id,use_type,offset_ref_type,conc_id,bridge_ref_line_id,straight_ind,offset_start,offset_end,bot_width_start,top_width_start,bot_width_end,top_width_end,avg_thick,conc_density,measured_to_left_face_ind,pedestrian_load
0,13690,1,2,1.0,20102,20202,1,None,None,-0.050810,-0.050810,2006.6,None,None,None,234.95000,None,F,NaN
1,13763,1,1,1.0,20102,20201,1,None,None,0.000000,0.000000,990.6,None,None,None,320.67500,None,T,NaN
2,13763,1,2,1.0,20102,20202,1,None,None,0.000000,0.000000,990.6,None,None,None,320.67500,None,F,NaN
3,16749,1,1,2.0,20101,20202,1,None,None,0.050800,0.050800,3962.4,None,None,None,203.20000,None,F,NaN
4,16928,4,1,2.0,20101,20201,4,None,None,0.000000,0.000000,2133.6,None,None,None,203.20000,None,T,3591.019792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1173,18584,2,2,2.0,20101,20202,1,None,None,0.304800,0.304800,1524.0,None,None,None,233.36250,None,F,NaN
1174,19381,1,1,2.0,20101,20202,1,None,None,0.000000,0.000000,1574.8,None,None,None,160.33750,None,F,NaN
1175,16731,1,2,1.0,20101,20202,3,None,None,0.304800,1.828800,1524.0,None,None,None,203.20000,None,F,3.591020
1176,16731,1,1,1.0,20101,20201,3,None,None,0.304800,1.828800,1524.0,None,None,None,203.20000,None,T,3.591020


#### ABW_CONC_RAILING

In [35]:
df = get_table("BRIDGEWARE", "ABW_CONC_RAILING")
df

,bridge_id,conc_railing_id,name,descr,si_or_us_type,barrier_type,conc_id,x1,x2,x3,x4,x5,y1,y2,y3,y4,dist_to_cg,additional_load,conc_density,last_change_timestamp
0,1200,1,Barrier,Barrier,10401,26804,None,457.200,NaN,NaN,NaN,NaN,914.4,NaN,NaN,NaN,228.6,9.340107,NaN,2012-02-01 21:16:58.000
1,1201,1,Pipe Rail,Hand Rail,10401,26804,None,50.800,NaN,NaN,NaN,NaN,609.6,NaN,NaN,NaN,25.4,0.145939,NaN,2012-02-03 16:55:59.000
2,1274,1,"OH-Deflector Parapet Type 42""",based upon plan detail,10401,26801,None,228.600,50.800,177.8,NaN,NaN,0.0,736.6,254.00,76.2,NaN,NaN,2402.809922,2017-12-01 14:34:50.000
3,1269,1,LEFT PARAPET,None,10401,26801,None,228.600,76.200,228.6,NaN,NaN,0.0,431.8,330.20,114.3,0.0,0.656726,2402.809922,2012-04-06 12:02:47.000
4,1269,2,SIDEWALK BARRIER,None,10401,26801,None,304.800,0.000,0.0,NaN,NaN,0.0,685.8,400.05,0.0,0.0,0.612945,2402.809922,2012-04-06 12:02:47.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8586,20555,2,Single Slope Median Barrier 57 inches,Single Slope Median Barrier 57 inches\r\nStd D...,10401,26801,None,257.175,276.225,0.0,NaN,NaN,0.0,1447.8,0.00,0.0,NaN,NaN,2402.809922,2024-07-25 20:31:33.480
8587,20556,1,"42"" Single Slope Concrete Bridge Railing",Section B-B Std. Dwg. SBR-1-20 (7-19-2024),10401,26801,None,254.000,203.200,0.0,NaN,NaN,0.0,1066.8,0.00,0.0,NaN,NaN,2402.809922,2024-07-31 17:12:20.527
8588,20556,2,"57"" Single Slope Concrete Median",Section G-G Std. Dwg. SBR-2-20 (7-19-2024),10401,26801,None,257.175,276.225,0.0,NaN,NaN,0.0,1447.8,0.00,0.0,NaN,NaN,2402.809922,2024-07-31 17:11:03.513
8589,20556,3,"Modified Deflector Parapet 50"" (Median)","Modified Deflector Parapet 50"" (Median)",10401,26801,None,127.000,50.800,254.0,NaN,NaN,0.0,889.0,330.20,50.8,NaN,NaN,2402.809922,2024-08-28 12:22:36.327


#### ABW_CONC_RAILING_LOC

In [36]:
df = get_table("BRIDGEWARE", "ABW_CONC_RAILING_LOC")
df

,bridge_id,struct_def_id,conc_railing_location_id,load_case_id,offset_ref_type,stl_railing_id,conc_railing_id,bridge_ref_line_id,conc_railing_offset_start,conc_railing_offset_end,steel_railing_offset,straight_ind,conc_railing_face_left_ind,stl_railing_face_left_ind,measured_to_front_face_ind
0,3158,1,1,1.0,20201,None,1,None,0.0,0.0,None,None,F,None,F
1,3158,1,2,1.0,20202,None,1,None,0.0,0.0,None,None,T,None,F
2,3158,2,1,1.0,20201,None,1,None,0.0,0.0,None,None,F,None,F
3,3158,2,2,1.0,20202,None,1,None,0.0,0.0,None,None,T,None,F
4,3159,1,1,1.0,20201,None,1,None,0.0,0.0,None,None,F,None,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16811,18065,1,2,4.0,20202,None,1,None,0.0,0.0,None,None,T,None,F
16812,18066,1,1,4.0,20201,None,1,None,0.0,0.0,None,None,F,None,F
16813,18066,1,2,4.0,20202,None,1,None,0.0,0.0,None,None,T,None,F
16814,18072,1,1,4.0,20201,None,1,None,0.0,0.0,None,None,F,None,F


#### ABW_BRIDGE_DIAPHRAGM_DEF_CON

In [37]:
df = get_table("BRIDGEWARE", "ABW_BRIDGE_DIAPHRAGM_DEF_CON")
df

,bridge_id,diaphragm_def_id,connection_id,name,support_type,y_dist,meas_from_type
0,2505,2,4,B,52401,NaN,52501.0
1,2505,3,1,A,52401,50.8,52501.0
2,2505,3,2,B,52401,50.8,52501.0
3,2505,3,3,C,52401,50.8,52502.0
4,2505,3,4,D,52401,50.8,52502.0
...,...,...,...,...,...,...,...
36101,20430,1,126,A,52401,355.6,52501.0
36102,20430,1,127,B,52401,355.6,52501.0
36103,20430,1,128,C,52401,254.0,52502.0
36104,20430,1,129,D,52401,254.0,52502.0


#### ABW_SUBSDEF_FOUND_FK

In [38]:
df = get_table("BRIDGEWARE", "ABW_SUBSDEF_FOUND_FK")
df

,bridge_id,struct_def_id,subsdef_found_id,as_built_found_alt_id,current_found_alt_id
0,7533,2,1,NaN,NaN
1,7534,3,1,NaN,NaN
2,18675,5,1,1.0,1.0
3,4904,3,1,1.0,1.0
4,5564,3,1,1.0,1.0
5,3717,2,1,1.0,1.0
6,5564,4,1,1.0,1.0
7,5564,5,1,1.0,1.0
8,5564,6,1,1.0,1.0
9,6370,2,1,NaN,NaN


#### ABW_STL_BUILTUP_IBEAM_DEF

In [39]:
df = get_table("BRIDGEWARE", "ABW_STL_BUILTUP_IBEAM_DEF")
df

,bridge_id,struct_def_id,spng_mbr_def_id
0,8528,2,1
1,8528,2,2
2,8528,2,3
3,8528,2,4
4,8528,2,5
...,...,...,...
519,7870,3,7
520,7960,1,1
521,7960,1,2
522,8145,2,1


#### ABW_STL_LONG_STIFF

In [40]:
df = get_table("BRIDGEWARE", "ABW_STL_LONG_STIFF")
df

,bridge_id,struct_def_id,stl_component_id,stl_angle_shape_id,web_weld_id,length,width,vert_dist,vert_dist_by_web_fraction,dist_ref_type,thick,short_leg_attachment_ind,stl_long_stiff_type,last_change_timestamp,long_stiff_side_type
0,12769,2,252,NaN,1.0,33680.400000,152.4,609.6,NaN,27002,15.875,None,22201,2019-11-06 18:37:22.000,NaN
1,12769,2,253,NaN,1.0,38252.400000,152.4,609.6,NaN,27001,15.875,None,22201,2019-11-06 18:37:22.000,NaN
2,12769,2,254,NaN,1.0,33680.400000,152.4,609.6,NaN,27002,15.875,None,22201,2019-11-06 18:37:22.000,NaN
3,12769,2,255,NaN,1.0,33171.384000,152.4,609.6,NaN,27001,15.875,None,22201,2019-11-06 18:37:22.000,NaN
4,4760,7,57,NaN,NaN,20624.800102,152.4,NaN,20.0,27004,9.525,None,22201,2017-12-11 15:34:32.000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3068,20329,1,918,NaN,NaN,16857.085920,114.3,NaN,20.0,27004,9.525,None,22201,2024-12-17 16:23:59.110,57301.0
3069,20329,1,919,NaN,NaN,17512.131600,114.3,NaN,20.0,27004,9.525,None,22201,2024-12-17 16:23:59.110,57301.0
3070,20329,1,920,NaN,NaN,17512.131600,114.3,NaN,20.0,27004,9.525,None,22201,2024-12-17 16:23:59.110,57301.0
3071,20329,1,921,NaN,NaN,17512.131600,114.3,NaN,20.0,27004,9.525,None,22201,2024-12-17 16:23:59.110,57301.0


#### ABW_EVENT_VEHICLE_TEMPLATE

In [41]:
df = get_table("BRIDGEWARE", "ABW_EVENT_VEHICLE_TEMPLATE")
df

,template_id,template_vehicle_id,vehicle_id,inventory_ind,operating_ind,design_ind,permit_ind,fatigue_ind,scale_factor,single_lane_ind,...,lrfr_operating_ind,legal_pair_ind,lrfr_fatigue_ind,legal_live_load_override_ind,legal_ll_override_factor,lrfr_legal_haul_ind,asr_lfr_permit_inventory_ind,asr_lfr_permit_operating_ind,asr_lfr_legal_inventory_ind,asr_lfr_legal_operating_ind
0,95,12,44,F,T,F,F,F,1.0,F,...,F,F,F,F,0.0,F,F,F,F,F
1,95,13,7,F,T,F,F,F,1.0,F,...,F,F,F,F,NaN,F,F,F,F,F
2,95,14,9,F,T,F,F,F,1.0,F,...,F,F,F,F,NaN,F,F,F,F,F
3,95,15,8,F,T,F,F,F,1.0,F,...,F,F,F,F,NaN,F,F,F,F,F
4,96,1,994,F,T,F,F,F,1.0,F,...,F,F,F,F,0.0,F,F,F,F,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448,107,14,9,F,T,F,F,F,1.0,F,...,F,F,F,F,NaN,F,F,F,F,F
449,106,14,9,F,F,F,F,F,1.0,F,...,F,F,F,F,NaN,F,F,F,F,T
450,106,15,8,F,F,F,F,F,1.0,F,...,F,F,F,F,NaN,F,F,F,F,T
451,107,1,994,F,T,F,F,F,1.0,F,...,F,F,F,F,0.0,F,F,F,F,F


#### ABW_RESULTS_DL_ACTION

In [42]:
df = get_table("BRIDGEWARE", "ABW_RESULTS_DL_ACTION")
df

,spng_mbr_alt_event_id,point_id,dl_action_id,dead_load_case_id,side_type,moment,axial,shear,reaction,x_defl,y_defl,top_flange_moment,bot_flange_moment,torsion
0,862,22,4,2193,58301,9.741623,0.0,-105.200300,NaN,0.0,-8.372860,0.0,0.0,NaN
1,862,23,4,2193,58301,-61.609489,0.0,-114.672080,NaN,0.0,-6.758536,0.0,0.0,NaN
2,862,24,7,2193,58302,-178.612077,0.0,-128.704060,NaN,0.0,-4.459940,0.0,0.0,NaN
3,862,24,8,2193,58303,-178.612077,0.0,-128.704060,NaN,0.0,-4.459940,0.0,0.0,NaN
4,862,25,4,2193,58301,-305.082018,0.0,-142.323982,NaN,0.0,-2.483374,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
332215,1149,14,6,2956,58301,187.127317,0.0,-3.601851,NaN,0.0,-1.291407,0.0,0.0,NaN
332216,1149,15,6,2956,58301,186.531070,0.0,-3.733869,NaN,0.0,-1.286499,0.0,0.0,NaN
332217,1149,16,6,2956,58301,162.097363,0.0,-7.323108,NaN,0.0,-1.089735,0.0,0.0,NaN
332218,1149,17,11,2956,58302,146.280424,0.0,-8.906086,NaN,0.0,-0.966901,0.0,0.0,NaN


#### ABW_RESULTS_LL_ACTION

In [43]:
df = get_table("BRIDGEWARE", "ABW_RESULTS_LL_ACTION")
df

,spng_mbr_alt_event_id,point_id,ll_action_id,stage_id,ll_vehicle_id,ll_vehicle_type,moment_pos,moment_neg,axial_pos,axial_neg,...,concurrent_moment_pos,concurrent_moment_neg,concurrent_shear_pos,concurrent_shear_neg,top_flange_moment_pos,top_flange_moment_neg,bot_flange_moment_pos,bot_flange_moment_neg,torsion_pos,torsion_neg
0,854,75,3,3,2,32802,351.517908,-1380.899441,0.0,0.0,...,241.242298,351.517908,-11.257291,44.223030,0.0,0.0,0.0,0.0,None,None
1,854,76,3,3,2,32802,349.517570,-1373.041331,0.0,0.0,...,271.251192,349.517570,-11.257291,44.223030,0.0,0.0,0.0,0.0,None,None
2,854,77,3,3,2,32802,623.043067,-1284.337721,0.0,0.0,...,602.539143,561.016654,251.212485,44.223030,0.0,0.0,0.0,0.0,None,None
3,854,78,3,3,2,32802,819.880821,-1227.466170,0.0,0.0,...,801.635219,752.368391,242.180532,44.223030,0.0,0.0,0.0,0.0,None,None
4,854,79,3,3,2,32802,948.704476,-1188.441926,0.0,0.0,...,931.429974,879.619589,235.715187,44.223030,0.0,0.0,0.0,0.0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441657,1149,182,41,5,9,32802,2254.976259,-518.627849,0.0,0.0,...,1973.563138,1973.563228,-54.183619,44.034948,0.0,0.0,0.0,0.0,None,None
441658,1149,183,41,5,9,32802,2145.790084,-686.530944,0.0,0.0,...,1730.951198,1968.366702,-101.842643,-44.176474,0.0,0.0,0.0,0.0,None,None
441659,1149,184,41,5,9,32802,2129.029885,-697.565551,0.0,0.0,...,1708.794975,1957.535332,-104.926198,-44.176474,0.0,0.0,0.0,0.0,None,None
441660,1149,185,41,5,9,32802,2127.889277,-698.283458,0.0,0.0,...,1707.308907,1956.779835,-105.126913,-44.176387,0.0,0.0,0.0,0.0,None,None


#### ABW_RESULTS_SPNG_MBR_ALT_XY

In [44]:
df = get_table("BRIDGEWARE", "ABW_RESULTS_SPNG_MBR_ALT_XY")
df

,spng_mbr_alt_event_id,report_id,spng_mbr_alt_xy_id,point_id,y
0,27,20,69,1738,-8.267467e+00
1,27,20,70,1739,-6.288441e+00
2,27,20,71,1741,-8.604776e-01
3,27,20,72,1742,-1.896382e-01
4,27,20,73,1743,-6.406247e-12
...,...,...,...,...,...
21238,825,1,25,25,-1.144289e+00
21239,825,1,26,26,-3.785004e-01
21240,825,1,27,27,-3.366784e-01
21241,825,1,28,28,-2.683945e-01


#### ABW_RIVET_DEF

In [45]:
df = get_table("BRIDGEWARE", "ABW_RIVET_DEF")
df

,bridge_id,rivet_def_id,name,descr,rivet_type,undriven_rivet_diameter,hole_diameter,fu,last_change_timestamp
0,10820,1,"7/8"" Rivet",None,58504,22.225,22.2250,NaN,2019-07-17 17:12:22
1,10866,1,"7/8"" Rivets","Original 7/8"" Rivets for Spans 1-5",58501,22.225,23.8125,413.685420,2019-07-17 18:48:39
2,4898,1,Rivet,None,58501,NaN,NaN,124.105626,2017-12-21 16:50:18
3,7545,1,"Standard 7/8"" Rivet-ASTM A141",Rivets for 1956 constructed steel plate girder...,58501,22.225,25.4000,358.527364,2019-04-26 12:31:09
4,12858,1,Rivets,Used for built up sections,58504,25.400,25.4000,413.685420,2019-12-23 19:36:21
5,13078,1,LAW7 Rivets,None,58504,22.225,25.4000,344.737850,2020-04-23 15:13:21
6,12768,1,"7/8"" Rivets",None,58501,22.225,23.8125,399.895906,2019-11-06 18:35:52


#### ABW_FLOORSYS_UNIT_GEOMETRY

In [46]:
df = get_table("BRIDGEWARE", "ABW_FLOORSYS_UNIT_GEOMETRY")
df

,bridge_id,struct_def_id,unit_geometry_id,unit_num,floorsys_stringer_group_def_id,mirror_stringer_group_def_type,ref_floor_beam_mbr_loc_id,unit_ref_type,dist_start_stringer_group_def,current_mbr_alt_name_template,existing_mbr_alt_name_template
0,18126,2,1,1,1.0,38301,11.0,38401,NaN,None,None
1,18126,2,2,2,1.0,38301,12.0,38402,NaN,None,None
2,18126,2,3,3,1.0,38301,13.0,38402,NaN,None,None
3,18126,2,4,4,1.0,38301,14.0,38402,NaN,None,None
4,18126,2,5,5,1.0,38301,15.0,38402,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...
566,20260,7,12,12,9.0,38301,NaN,38402,0.0,None,None
567,20260,7,13,13,10.0,38301,NaN,38402,0.0,None,None
568,20260,7,14,14,11.0,38301,NaN,38402,0.0,None,None
569,20260,7,15,15,12.0,38301,NaN,38402,0.0,None,None


#### ABW_FLRBM_STRINGER_DL_REACTION

In [47]:
df = get_table("BRIDGEWARE", "ABW_FLRBM_STRINGER_DL_REACTION")
df

,bridge_id,struct_def_id,floor_beam_mbr_id,dl_reaction_id,stringer_mbr_id,stage_id,user_defined_reaction,override_computed_ind,up_to_date_ind
0,12957,2,75,87,46,1,NaN,F,F
1,12957,2,75,88,46,2,NaN,F,F
2,12957,2,75,89,47,1,NaN,F,F
3,12957,2,75,90,47,2,NaN,F,F
4,12957,2,76,1,12,1,NaN,F,F
...,...,...,...,...,...,...,...,...,...
21927,20416,18,19,9,9,1,NaN,None,F
21928,20416,18,19,10,9,2,NaN,None,F
21929,20416,18,20,1,5,1,NaN,None,F
21930,20416,18,20,2,5,2,NaN,None,F


#### ABW_MULTIMEDIA

In [48]:
df = get_table("BRIDGEWARE", "ABW_MULTIMEDIA")
df

,bridge_id,multimedia_id,context,fileloc,fileref,filetype,agency_type_param_id,status,reportflag,userkey1,userkey2,userkey3,userkey4,userkey5,notes,creation_event_id,last_modified_event_id
0,20530,2,BrDR,R:\BridgeManagement\LRDMS\Data Files\0804002\P...,P3290304.JPG,JPG File,137,None,N,None,None,None,None,None,None,1205562.0,1205563.0
1,16912,1,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,BrR Library Ohio 2022-10-19.brlx,BRLX File,137,None,N,None,None,None,None,None,None,NaN,NaN
2,16912,2,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,2022 BrR Rating Templates 2022-10.brsx,BRSX File,137,None,N,None,None,None,None,None,None,NaN,NaN
3,16912,3,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,BrDR System Data 2022-9-27.brsx,BRSX File,137,None,N,None,None,None,None,None,None,NaN,NaN
4,20530,1,BrDR,R:\BridgeManagement\LRDMS\Data Files\0804002\P...,P3290238.JPG,JPG File,137,None,N,None,None,None,None,None,None,1205560.0,1205561.0
5,85,1,Virtis/Opis,X:\POR-43-1375\,DSC_1690.JPG,JPG,137,None,N,None,None,None,None,None,None,NaN,NaN
6,16731,1,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,BrR Library Ohio 2022-10-19.brlx,BRLX File,137,None,N,None,None,None,None,None,None,NaN,NaN
7,16731,2,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,2022 BrR Rating Templates 2022-10.brsx,BRSX File,137,None,N,None,None,None,None,None,None,NaN,NaN
8,16731,3,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,BrDR System Data 2022-9-27.brsx,BRSX File,137,None,N,None,None,None,None,None,None,NaN,NaN


#### ABW_LIB_MATL_SOIL

In [49]:
df = get_table("BRIDGEWARE", "ABW_LIB_MATL_SOIL")
df

,soil_id,name,descr,library_type,si_or_us_type,soil_unit_load,sat_soil_unit_load,lrfd_lrfr_at_rest_lep_coeff,lrfd_lrfr_active_lep_coeff,lrfd_lrfr_passive_lep_coeff,lfr_max_lat_soil_press,lfr_min_lat_soil_press,lrfr_at_rest_lep_coeff
0,1,Standard Soil 1,Standard Soil 1,22901,10401,1922.247938,2002.341602,0.5,0.33,3.0,961.123969,480.561984,None
1,2,Standard Soil 2,Standard Soil 2,22901,10401,1922.247938,2162.528930,0.5,0.33,0.5,480.561984,480.561984,None


#### ABW_LIB_MATL_ALUMINUM

In [50]:
df = get_table("BRIDGEWARE", "ABW_LIB_MATL_ALUMINUM")
df

,aluminum_id,name,descr,library_type,si_or_us_type,yield_strength,tens_strength,thermal_exp_coeff,density,mod_of_elast
0,1,H34,Corrugated metal (Grade 3004-H34),22901,10401,165.474168,213.737467,0.000023,2803.278242,68947.57
1,2,H32,Corrugated metal (Grade 3004-H32),22901,10401,137.895140,186.158439,0.000023,2803.278242,68947.57
2,3,Structural Plate 0.100-0.175,"Structural plate (thickness 0.100""-0.175"")",22901,10401,165.474168,241.316495,0.000023,2803.278242,68947.57
3,4,Structural Plate 0.176-0.250,"Structural plate (thickness 0.176""-0.250"")",22901,10401,165.474168,234.421738,0.000023,2803.278242,68947.57


#### ABW_MATL_TIMBER

In [51]:
df = get_table("BRIDGEWARE", "ABW_MATL_TIMBER")
df

,bridge_id,timber_id,name,descr,si_or_us_type,matl_timber_type,density,grading_method_type,last_change_timestamp
0,16762,1,Douglas Fir-Larch,Douglas Fir-Larch,10401,35801,800.936641,36401,2023-09-25 19:34:28.837
1,20225,1,Southern Pine SP 47,SP 47 Glulam Panels,10401,35801,800.936641,36401,2025-05-01 14:53:14.740
2,20225,4,Dummy Deck,No additional load.,10401,35801,0.000000,36401,2025-05-01 14:53:13.573
3,806,1,Wooden Deck (2X4),Southern Pine,10401,35801,800.936640,36401,2010-11-16 20:19:25.000
4,806,2,Wooden Beams,Douglas Fir-Larch,10401,35801,800.936640,36401,2010-11-16 20:19:25.000
...,...,...,...,...,...,...,...,...,...
82,20521,4,Dummy Deck,No additional load.,10401,35801,0.000000,36401,2025-05-01 14:53:13.573
83,20522,1,Southern Pine SP 47,SP 47 Glulam Panels,10401,35801,800.936641,36401,2025-07-01 10:37:03.590
84,20522,4,Dummy Deck,No additional load.,10401,35801,0.000000,36401,2025-05-01 14:53:13.573
85,20523,1,Southern Pine SP 47,SP 47 Glulam Panels,10401,35801,800.936641,36401,2025-05-01 14:53:14.740


#### ABW_MATL_SOIL

In [52]:
df = get_table("BRIDGEWARE", "ABW_MATL_SOIL")
df

,bridge_id,soil_id,name,descr,si_or_us_type,soil_unit_load,sat_soil_unit_load,lrfd_lrfr_at_rest_lep_coeff,lrfd_lrfr_active_lep_coeff,lrfd_lrfr_passive_lep_coeff,lfr_max_lat_soil_press,lfr_min_lat_soil_press,lrfr_at_rest_lep_coeff
0,16829,1,Standard Soil 1,Standard Soil 1,10401,1922.247938,2002.341602,0.5,0.33,3.0,961.123969,480.561984,NaN
1,16893,1,Standard Soil 1,Standard Soil 1,10401,1922.247938,2002.341602,0.5,0.33,3.0,961.123969,480.561984,NaN
2,16894,1,Standard Soil 1,Standard Soil 1,10401,1922.247938,2002.341602,0.5,0.33,3.0,961.123969,480.561984,NaN
3,16899,1,Standard Soil 1,Standard Soil 1,10401,1922.247938,2002.341602,0.5,0.33,3.0,961.123969,480.561984,NaN
4,16900,1,Standard Soil 1,Standard Soil 1,10401,1922.247938,2002.341602,0.5,0.33,3.0,961.123969,480.561984,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1259,20436,1,Standard Soil 1,Standard Soil 1,10401,1922.247938,2002.341602,0.5,0.33,3.0,961.123969,480.561984,NaN
1260,20440,1,Standard Soil 1,Standard Soil 1,10401,1922.247937,2002.341602,0.5,0.33,3.0,961.123969,480.561984,NaN
1261,20441,1,Standard Soil 1,Standard Soil 1,10401,1922.247937,2002.341602,0.5,0.33,3.0,961.123969,480.561984,NaN
1262,20516,1,Standard Soil 1,Standard Soil 1,10401,1922.247937,2002.341602,0.5,0.33,3.0,961.123969,480.561984,NaN


#### ABW_METAL_PIPE_CULVERT_DEF

In [53]:
df = get_table("BRIDGEWARE", "ABW_METAL_PIPE_CULVERT_DEF")
df

,bridge_id,struct_def_id,culvert_def_id,metal_pipe_culvert_struct_type,ncspa_struct_category_type,dd_backfill_material_type,dd_relative_compaction,lrfr_cons_dd_plastic_mom_ind,lrfr_cons_mult_load_ln_ind,lfr_cons_dd_plastic_mom_ind,...,water_height,clear_roadway_width,matl_stl_structural_id,matl_aluminum_id,pavement_reduct_factor,pavement_reduct_fact_comment,wearing_surface_density,wearing_surface_thickness,thickness_field_measured_ind,section_properties_gage
0,17123,1,1,62201,62301,62403,95.0,F,T,F,...,0.0000,13.4112,1.0,NaN,NaN,None,0.000000,0.0,F,12.0
1,17124,1,1,62201,62301,62403,95.0,F,T,F,...,0.0000,13.4112,1.0,NaN,NaN,None,0.000000,0.0,F,12.0
2,18649,5,1,62203,62303,62403,90.0,T,T,T,...,NaN,35.9664,2.0,NaN,NaN,None,NaN,NaN,F,5.0
3,18650,5,1,62203,62303,62403,90.0,T,T,T,...,NaN,7.9248,2.0,NaN,NaN,None,NaN,NaN,F,5.0
4,19388,1,1,62201,62303,62403,90.0,T,T,T,...,0.6858,13.4112,2.0,NaN,NaN,None,2322.716258,63.5,F,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,20327,1,1,62203,62303,62403,90.0,F,T,F,...,NaN,14.6304,1.0,NaN,NaN,None,2322.716258,76.2,F,1.0
60,20327,2,1,62203,62303,62403,90.0,F,T,F,...,NaN,14.6304,1.0,NaN,NaN,None,2322.716258,76.2,F,1.0
61,20422,3,1,62201,62301,62403,90.0,F,T,F,...,0.0000,7.3152,1.0,NaN,NaN,None,2322.716258,76.2,F,10.0
62,20389,2,1,62203,62303,62403,90.0,T,T,T,...,NaN,6.4008,2.0,NaN,NaN,None,NaN,NaN,F,8.0


#### ABW_CULVERT

In [54]:
df = get_table("BRIDGEWARE", "ABW_CULVERT")
df

,bridge_id,bridge_design_alt_id,culvert_id,name,descr,dist,offset,angle,start_station,last_modified_event_id,creation_event_id,last_change_timestamp,vehicle_path_long_increment
0,16829,1,1,Existing,None,NaN,NaN,0.0,NaN,None,None,2023-10-05 16:33:17.881000,NaN
1,16893,1,1,Culvert,None,NaN,NaN,NaN,NaN,None,None,NaT,1.2192
2,16894,1,1,MIA-41-1085,None,NaN,NaN,NaN,NaN,None,None,NaT,1.2192
3,16899,1,1,SHE-29-02.620,None,NaN,NaN,NaN,NaN,None,None,2023-11-14 17:39:08.113000,1.2192
4,16900,1,1,SHE-29-10.33,None,NaN,NaN,NaN,NaN,None,None,NaT,1.2192
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1220,20497,1,1,5710316 One Cell,None,NaN,NaN,0.0,NaN,None,None,2025-06-09 21:15:53.863000,NaN
1221,20499,1,2,6105246 Two Cell,None,NaN,NaN,0.0,NaN,None,None,2025-06-29 20:42:53.703000,NaN
1222,20506,1,1,RC Box Culvert (deterioration),None,NaN,NaN,0.0,NaN,None,None,2025-01-29 14:28:34.120000,NaN
1223,20539,1,2,3201694,None,NaN,NaN,0.0,NaN,None,None,2025-07-01 19:32:06.295000,NaN


#### ABW_SYS_LRFR_LOADING

In [55]:
df = get_table("BRIDGEWARE", "ABW_SYS_LRFR_LOADING")
df

,sys_lrfr_loading_id,name,descr
0,1,DC,DC
1,2,DW,DW
2,3,LL,LL


#### ABW_SYS_LRFR_LOAD_GROUP

In [56]:
df = get_table("BRIDGEWARE", "ABW_SYS_LRFR_LOAD_GROUP")
df

,sys_lrfr_load_group_id,name,descr
0,1,Inventory,Inventory
1,2,Operating,Operating
2,3,Legal Load,Legal Load
3,4,Permit Load,Permit Load
4,5,Dead Load,Dead Load


#### ABW_SYS_REPORT

In [57]:
df = get_table("BRIDGEWARE", "ABW_SYS_REPORT")
df

,sys_report_id,descr,report_keyword_1_id,report_keyword_2_id,report_keyword_3_id,report_keyword_4_id,report_keyword_5_id,report_keyword_6_id,stored_unit_id,si_unit_id,us_unit_id,data_type
0,1,Girder Weight Deflection Report,None,None,None,None,None,None,24,24,22,10203
1,2,Prestressing Loads Deflection Report,None,None,None,None,None,None,24,24,22,10203
2,3,Final Deflection Report,None,None,None,None,None,None,24,24,22,10203


#### ABW_SYS_REPORT_KEYWORD

In [58]:
df = get_table("BRIDGEWARE", "ABW_SYS_REPORT_KEYWORD")
df

,report_keyword_id,keyword,descr


#### ABW_SYS_SUBTYPE_TABLES

In [59]:
df = get_table("BRIDGEWARE", "ABW_SYS_SUBTYPE_TABLES")
df

,subtype_table_id,table_name,type_define,type_identifier,discriminator_col
0,155,abw_bmdef_anal_pt_stl,TYP_ANALPT_ANPTSTL,ANALPT.ANPTSTL,None
1,156,abw_floorline_floor_beam_mbr,TYP_SUPSTMBR_FLNFBMBR,SUPSTMBR.FLNFBMBR,None
2,159,abw_floorsys_floor_beam_mbr,TYP_SUPSTMBR_FSYSFBMBR,SUPSTMBR.FSYSFBMBR,None
3,160,abw_floorsys_stringer_mbr,TYP_SUPSTMBR_FSYSSTRMBR,SUPSTMBR.FSYSSTRMBR,None
4,161,abw_floorsys_struct_def,*,*,floorsys_struct_def_type
...,...,...,...,...,...
221,350,abw_lib_metal_pipe_culvert,*,*,lib_metal_pipe_culvert_type
222,351,abw_metal_box_culvert_def,TYP_CULVERTDEF_METALBOX,CULVERTDEF.METALBOX,None
223,352,abw_lib_matl_timber_glulam,TYP_MATLTIMBER_GLULAM,MATLTIMBER.GLULAM,None
224,353,abw_matl_timber_glulam,TYP_MATLTIMBER_GLULAM,MATLTIMBER.GLULAM,None


#### ABW_SUPER_FR_FORCE_SUB

In [60]:
df = get_table("BRIDGEWARE", "ABW_SUPER_FR_FORCE_SUB")
df

,bridge_id,sub_struct_def_id,fr_force_set_id,span_location_type


#### ABW_SUPER_LL_PATTERN_SUB

In [61]:
df = get_table("BRIDGEWARE", "ABW_SUPER_LL_PATTERN_SUB")
df

,bridge_id,struct_def_id,ll_pattern_id,load_pattern_number,descr


#### ABW_SUPER_LOAD_SCENARIO

In [62]:
df = get_table("BRIDGEWARE", "ABW_SUPER_LOAD_SCENARIO")
df

,bridge_id,struct_def_id,load_scenario_id,name,descr
0,355,1,1,Default,created by BARS Import
1,356,1,1,Default,created by BARS Import
2,357,1,1,Default,created by BARS Import
3,395,1,1,Default,created by BARS Import
4,17766,1,1,Load Scenario 1,Load Scenario created for new SuperStructure D...
...,...,...,...,...,...
14856,16720,1,1,Load Scenario 1,Load Scenario created for new SuperStructure D...
14857,16721,1,1,Load Scenario 1,Load Scenario created for new SuperStructure D...
14858,16722,1,1,Load Scenario 1,Load Scenario created for new SuperStructure D...
14859,16723,1,1,Load Scenario 1,Load Scenario created for new SuperStructure D...


#### ABW_SUPER_LOAD_SCENARIO_ITEM

In [63]:
df = get_table("BRIDGEWARE", "ABW_SUPER_LOAD_SCENARIO_ITEM")
df

,bridge_id,struct_def_id,load_scenario_id,load_scenario_item_id,load_case_id
0,355,1,1,1,1
1,356,1,1,1,1
2,357,1,1,1,1
3,395,1,1,1,1
4,17084,1,1,8,2
...,...,...,...,...,...
48321,9375,2,1,4,4
48322,13939,1,1,1,1
48323,13939,1,1,2,2
48324,13939,1,1,3,3


#### ABW_SUPER_PED_LL_REACT_SUB

In [64]:
df = get_table("BRIDGEWARE", "ABW_SUPER_PED_LL_REACT_SUB")
df

,bridge_id,sub_struct_def_id,ped_ll_set_id,span_location_type,use_override_values_ind
0,13291,3,1,42403,None
1,13291,4,1,42403,F
2,13291,5,1,42403,None
3,13291,6,1,42403,None


#### ABW_PIER_WSHFT_FLEX_REINF

In [65]:
df = get_table("BRIDGEWARE", "ABW_PIER_WSHFT_FLEX_REINF")
df

,bridge_id,struct_def_id,wall_shaft_id,reinf_set_id,start_dist,straight_length,reinf_pattern_def_id,hook_at_start_ind,hook_at_end_ind,developed_at_start_ind,developed_at_end_ind,reinf_set_num,follows_profile_ind,head_at_start_ind,head_at_end_ind


#### ABW_PIER_WSHFT_RECT_XSECT

In [66]:
df = get_table("BRIDGEWARE", "ABW_PIER_WSHFT_RECT_XSECT")
df

,bridge_id,struct_def_id,wall_shaft_id,wall_shaft_xsection_id,width,depth


#### ABW_PIER_WSHFT_REINF_PATT_DET

In [67]:
df = get_table("BRIDGEWARE", "ABW_PIER_WSHFT_REINF_PATT_DET")
df

,bridge_id,struct_def_id,wall_shaft_id,reinf_pattern_def_id,reinf_bar_id,rebar_id,stl_reinf_id,bundle_type,x_coordinate,y_coordinate


#### ABW_SUPER_STRUCT_DIAPH_MBR

In [68]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_DIAPH_MBR")
df

,bridge_id,struct_def_id,super_struct_mbr_id,left_spng_mbr_id,right_spng_mbr_id,left_spng_mbr_dist,right_spng_mbr_dist,diaph_def_id


#### ABW_SUPER_STRUCT_LOADING_PATH

In [69]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_LOADING_PATH")
df

,bridge_id,bridge_design_alt_id,super_struct_id,loading_path_id,nsg_vehicle_path_type,nsg_vehicle_cen_line_loc,adjacent_vehicle_path_type,adjacent_vehicle_cen_line_loc
0,6319,1,1,1,45702,NaN,45705,None
1,2440,2,2,1,45702,NaN,45705,None
2,2257,2,2,1,45702,NaN,45705,None
3,6350,1,1,1,45702,NaN,45705,None
4,6351,1,1,1,45702,NaN,45705,None
...,...,...,...,...,...,...,...,...
10554,19862,1,1,1,45702,NaN,45705,None
10555,19864,2,1,1,45702,NaN,45705,None
10556,19878,2,1,1,45702,NaN,45705,None
10557,19879,1,126,1,45702,NaN,45705,None


#### ABW_PIER_COL_REINF_PATTERN_DEF

In [70]:
df = get_table("BRIDGEWARE", "ABW_PIER_COL_REINF_PATTERN_DEF")
df

,bridge_id,struct_def_id,pier_column_id,reinf_pattern_def_id,name,bundled_bars_ind
0,5564,3,1,1,Typical,F
1,5564,3,2,1,Typical,F
2,5564,3,3,1,Typical,F
3,5564,4,1,1,Typical,F
4,5564,4,2,1,Typical,F
5,5564,4,3,1,Typical,F
6,5564,5,1,1,Typical,F
7,5564,5,2,1,Typical,F
8,5564,5,3,1,Typical,F
9,5564,6,1,1,Typical,F


#### ABW_STL_TRUSS_XSECTION_CPLATE

In [71]:
df = get_table("BRIDGEWARE", "ABW_STL_TRUSS_XSECTION_CPLATE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,element_xsection_id,cover_plate_id,width,thick,relative_position,stl_structural_id,side_weld_id,end_weld_id,bolt_rivet_hole_size,bolt_rivet_hole_effective_num,bolt_rivet_eff_area_deduction
0,4034,1,5,2,1,152.4,9.525,1,1,None,None,None,0.0,None
1,4034,1,5,2,2,152.4,9.525,-1,1,None,None,None,0.0,None


#### ABW_STL_WEB_COVER_PLATE

In [72]:
df = get_table("BRIDGEWARE", "ABW_STL_WEB_COVER_PLATE")
df

,bridge_id,struct_def_id,stl_component_id,length,relative_position,depth_start,depth_end,depth_variation_type


#### ABW_STL_WEB_LOSS_RANGE

In [73]:
df = get_table("BRIDGEWARE", "ABW_STL_WEB_LOSS_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,range_id,percent_thick_loss
0,16762,3,1,1,5.0
1,18036,1,1,2,10.4
2,18036,1,2,1,10.4
3,18036,1,2,2,10.4
4,20151,2,5,3,10.0
...,...,...,...,...,...
696,20457,2,1,1,20.0
697,20457,2,1,2,20.0
698,20457,2,4,1,20.0
699,20457,2,4,4,20.0


#### ABW_STL_WEB_PLATE

In [74]:
df = get_table("BRIDGEWARE", "ABW_STL_WEB_PLATE")
df

,bridge_id,struct_def_id,stl_component_id,end_weld_id,length,depth_start,depth_end,depth_variation_type
0,1753,1,4,NaN,85.3440,1549.400,1549.400,26202
1,1753,2,4,NaN,85.3440,1625.600,1625.600,26202
2,1753,1,18,NaN,85.3440,1549.400,1549.400,26202
3,1753,2,17,NaN,85.3440,1625.600,1625.600,26202
4,19507,1,17,NaN,68.8848,1320.800,1320.800,26202
...,...,...,...,...,...,...,...,...
25955,2394,3,165,NaN,10.2870,719.074,719.074,26202
25956,2394,3,166,NaN,14.5542,719.074,719.074,26202
25957,2394,3,167,NaN,12.1158,719.074,719.074,26202
25958,2394,3,168,NaN,2.4384,719.074,1473.200,26201


#### ABW_STL_XSECTION

In [75]:
df = get_table("BRIDGEWARE", "ABW_STL_XSECTION")
df

,bridge_id,struct_def_id,spng_mbr_def_id,xsection_id,name,xsection_type,web_depth,web_thick,flng_thick_bot,flng_thick_top,...,tdeck_tributary_width,tdeck_nominal_thick,tdeck_nominal_width,lfd_timber_ll_dist_dist_width,shear_flex_moisture_cond_type,bearing_moisture_cond_type,stiffness_moisture_cond_type,timber_deck_type,deck_load_case_id,deck_loadcase_engineassign_ind
0,20260,6,25,1,FB36,31303,1371.600,12.700,NaN,NaN,...,None,None,None,None,36101.0,36101.0,36101.0,36301.0,NaN,None
1,20260,6,25,2,FB36 - Post-Tensioning Rod,31303,1371.600,12.700,NaN,NaN,...,None,None,None,None,36101.0,36101.0,36101.0,36301.0,NaN,None
2,20260,6,26,1,FB38,31303,1371.600,12.700,NaN,NaN,...,None,None,None,None,36101.0,36101.0,36101.0,36301.0,NaN,None
3,20260,6,27,1,FB15,31303,1371.600,12.700,NaN,NaN,...,None,None,None,None,36101.0,36101.0,36101.0,36301.0,NaN,None
4,20260,6,27,2,FB15 - Post-Tensioning Rod,31303,1371.600,12.700,NaN,NaN,...,None,None,None,None,36101.0,36101.0,36101.0,36301.0,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8097,20416,14,4,8,Section 6A,31303,3350.514,15.875,NaN,NaN,...,None,None,None,None,36101.0,36101.0,36101.0,36301.0,NaN,T
8098,20416,14,4,9,Section 6B,31303,3217.164,15.875,NaN,NaN,...,None,None,None,None,36101.0,36101.0,36101.0,36301.0,NaN,T
8099,20416,14,4,10,Section 7A,31303,3217.164,15.875,NaN,NaN,...,None,None,None,None,36101.0,36101.0,36101.0,36301.0,NaN,T
8100,20416,14,4,11,Section 7B,31303,3048.000,15.875,NaN,NaN,...,None,None,None,None,36101.0,36101.0,36101.0,36301.0,NaN,T


#### ABW_ANALYSIS_EVENT

In [76]:
df = get_table("BRIDGEWARE", "ABW_ANALYSIS_EVENT")
df

,event_id,analysis_method_type,analysis_event_type,structural_analysis_type,nsg_vehicle_path_type,nsg_vehicle_cen_line_loc,adjacent_vehicle_path_type,adjacent_vehicle_cen_line_loc,std_ll_ss_df_compute_type,lrfr_permit_lane_load,lrfr_exc_permit_lane_load_ind,lane_impact_loading_type,genpref_template_id,adjacent_veh_live_load_fact,td_fea_analysis_option_type
0,1202217,32606,32501,46401,45701.0,None,45701.0,None,48201,NaN,F,50201,None,None,59104.0
1,1205031,32606,32501,46401,45701.0,None,45701.0,None,48201,NaN,F,50201,None,None,59104.0
2,1192182,32606,32501,46401,45701.0,None,45701.0,None,48201,NaN,F,50201,None,None,59104.0
3,1203372,32606,32501,46401,45701.0,None,45701.0,None,48201,NaN,F,50201,None,None,59104.0
4,21093,32602,32501,46401,NaN,None,NaN,None,48201,NaN,None,50201,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,1164879,32606,32501,46401,45701.0,None,45701.0,None,48201,NaN,None,50201,None,None,59104.0
196,1164898,32606,32501,46401,45701.0,None,45701.0,None,48201,NaN,None,50201,None,None,59104.0
197,1201211,32602,32501,46401,45701.0,None,45701.0,None,48201,NaN,None,50201,None,None,59104.0
198,1184257,32606,32501,46401,45701.0,None,45701.0,None,48201,NaN,F,50201,None,None,59104.0


#### ABW_STL_THREADED_BAR

In [77]:
df = get_table("BRIDGEWARE", "ABW_STL_THREADED_BAR")
df

,bridge_id,stl_component_id,struct_def_id,diameter,num_bars,end_config_type


#### ABW_STL_TRANS_STIFF_ANGLE

In [78]:
df = get_table("BRIDGEWARE", "ABW_STL_TRANS_STIFF_ANGLE")
df

,bridge_id,struct_def_id,stl_component_id,stl_shape_id,vert_direction_ind,short_leg_attachment_ind,num_bolts,dist_top_bolt,dist_bot_bolt,dist_from_top_flnge,dist_from_bot_flnge,bolt_def_id
0,18338,2,1,3,None,T,NaN,None,None,None,None,NaN
1,18351,1,1,4,None,T,NaN,None,None,None,None,NaN
2,18351,1,3,5,None,T,NaN,None,None,None,None,NaN
3,20266,1,54,12,None,T,NaN,None,None,None,None,NaN
4,20266,1,56,13,None,T,NaN,None,None,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
458,20260,7,11,13,None,T,NaN,None,None,None,None,NaN
459,20260,7,12,10,None,T,NaN,None,None,None,None,NaN
460,20260,7,13,26,None,F,NaN,None,None,None,None,NaN
461,20260,7,14,12,None,T,NaN,None,None,None,None,NaN


#### ABW_STL_TRANS_STIFF_GNRL_RANGE

In [79]:
df = get_table("BRIDGEWARE", "ABW_STL_TRANS_STIFF_GNRL_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,stiff_range_id,dist,length,max_spacing,stl_component_id


#### ABW_STL_TRANS_STIFF_LOC_RANGE

In [80]:
df = get_table("BRIDGEWARE", "ABW_STL_TRANS_STIFF_LOC_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,stiff_range_id,stl_component_id,dist,spacing,num_stiff
0,18944,2,1,1,2.0,3.274448,0.000000,1.0
1,18944,2,1,2,2.0,3.274448,4500.000000,4.0
2,18944,2,1,3,2.0,21.274448,3500.000000,3.0
3,572,1,1,1,NaN,0.000000,635.000048,1.0
4,572,1,1,2,NaN,0.635000,1219.200000,1.0
...,...,...,...,...,...,...,...,...
223193,20525,2,2,30,1.0,26.103762,2124.075000,1.0
223194,20525,2,2,31,1.0,30.351912,1416.050000,2.0
223195,20525,2,2,32,1.0,34.600062,2124.075000,1.0
223196,20525,2,2,33,1.0,38.848212,1416.050000,2.0


#### ABW_CHECKED_OUT_BRIDGE

In [81]:
df = get_table("BRIDGEWARE", "ABW_CHECKED_OUT_BRIDGE")
df

,bridge_check_out_id,bridge_id,event_id


#### ABW_STL_TRANS_STIFF_PLATE

In [82]:
df = get_table("BRIDGEWARE", "ABW_STL_TRANS_STIFF_PLATE")
df

,bridge_id,struct_def_id,stl_component_id,top_flng_weld_id,bot_flng_weld_id,web_weld_id,width,dist_from_top_flng,dist_from_bot_flng,out_clip_length_top,out_clip_length_bot,ins_clip_horz_length_top,ins_clip_vert_length_top,ins_clip_horz_length_bot,ins_clip_vert_length_bot,thick
0,3650,1,78,1.0,1.0,1.0,127.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.5250
1,116,2,1,NaN,NaN,NaN,190.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.8750
2,16304,2,1,NaN,NaN,NaN,114.3,NaN,NaN,0.0,0.0,25.4,63.5,25.4,63.5,12.7000
3,16324,2,1,NaN,NaN,NaN,139.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.5250
4,16324,2,2,NaN,NaN,NaN,139.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.5250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8222,20532,3,4,NaN,NaN,NaN,152.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.0500
8223,20532,3,89,NaN,NaN,NaN,127.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.9375
8224,20533,4,1,NaN,NaN,NaN,152.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.7000
8225,20533,4,2,NaN,NaN,NaN,190.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.0500


#### ABW_ADV_CONC_TENDON_RANGE

In [83]:
df = get_table("BRIDGEWARE", "ABW_ADV_CONC_TENDON_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,tendon_range_id,tendon_profile_id,stage_id
0,18358,1,1,12,1,None
1,18358,1,1,13,3,None
2,18358,1,1,14,4,None
3,18358,1,1,16,9,None
4,18358,1,3,2,1,None
5,18358,1,3,3,2,None
6,18358,1,3,4,3,None
7,18358,1,3,6,7,None
8,19736,1,1,21,1,None
9,19736,1,1,22,2,None


#### ABW_DECK_PANEL_RATING_RESULTS

In [84]:
df = get_table("BRIDGEWARE", "ABW_DECK_PANEL_RATING_RESULTS")
df

,deck_panel_analysis_event_id,rating_results_summary_id,vehicle_id,vehicle_type,design_method_type,rating_success_type,inv_capacity,opr_capacity,post_capacity,safe_capacity,...,inv_limit_state,opr_limit_state,post_limit_state,safe_limit_state,lane_loading_type,impact_loading_type,results_flag_type,span_num,range_left_dist,range_right_dist


#### ABW_DEF_ANAL_OPTION_SUBS_LOAD

In [85]:
df = get_table("BRIDGEWARE", "ABW_DEF_ANAL_OPTION_SUBS_LOAD")
df

,bridge_id,bridge_design_alt_id,vehicle_id,sub_struct_loading_id,vehicle_type,axle_load


#### ABW_DESIGNRATIO_RESULT

In [86]:
df = get_table("BRIDGEWARE", "ABW_DESIGNRATIO_RESULT")
df

,spng_mbr_alt_event_id,point_id,rating_results_id,vehicle_id,vehicle_type,design_method_type,design_capcity_type,design_capacity_force,design_capacity_stress,design_capacity_moment,...,fatigue_capacity_force,design_design_ratio,permit_design_ratio,fatigue_design_ratio,design_limit_state,permit_limit_state,fatigue_limit_state,design_article_name,permit_article_name,fatigue_article_name


#### ABW_DIAPH_LOC

In [87]:
df = get_table("BRIDGEWARE", "ABW_DIAPH_LOC")
df

,bridge_id,struct_def_id,diaph_loc_id,left_spng_mbr_id,right_spng_mbr_id,left_spng_mbr_dist,right_spng_mbr_dist,num_spaces,spacing,diaph_def_id,diaph_weight,diaphragm_def_id,spacing_reference_type,curved_sys_left_spacing,curved_sys_right_spacing
0,19771,1,10,3,4,5.972215,5.910283,5.0,6.0960,None,1.610256,2.0,54803,NaN,NaN
1,19771,1,12,4,5,0.000000,0.000000,1.0,0.0000,None,NaN,NaN,54803,NaN,NaN
2,19771,1,13,4,5,5.910283,5.848350,1.0,0.0000,None,1.258847,1.0,54803,NaN,NaN
3,19771,1,14,4,5,5.910283,5.848350,5.0,6.0960,None,1.258847,1.0,54803,NaN,NaN
4,19771,1,22,1,2,36.576080,36.514148,1.0,3.0480,None,1.258847,1.0,54803,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287325,20413,2,118,1,2,40.017192,39.514272,2.0,1.2192,None,1.601360,4.0,54803,NaN,NaN
287326,20414,1,1,1,2,0.000000,0.000000,1.0,NaN,None,113.918965,1.0,54803,NaN,NaN
287327,20414,1,2,1,2,6.790563,5.809107,1.0,0.0000,None,2.757898,2.0,54803,NaN,NaN
287328,20414,1,3,1,2,6.790563,5.809107,2.0,6.7056,None,2.757898,2.0,54803,NaN,NaN


#### ABW_MBR_ALT_SUPPORT

In [88]:
df = get_table("BRIDGEWARE", "ABW_MBR_ALT_SUPPORT")
df

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,mbr_alt_support_id,support_id,bearing_dist_left,bearing_dist_right,timber_brg_area_length,timber_brg_area_width
0,1147,1,1,1,594,2157,190.5,190.5,NaN,NaN
1,1147,1,1,1,595,2158,NaN,NaN,NaN,NaN
2,19464,2,1,1,1,1,NaN,NaN,NaN,NaN
3,19464,2,1,1,2,2,NaN,NaN,NaN,NaN
4,19464,2,2,1,1,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
32743,20554,1,1,1,2,2,NaN,NaN,NaN,NaN
32744,20554,1,2,1,1,1,NaN,NaN,NaN,NaN
32745,20554,1,2,1,2,2,NaN,NaN,NaN,NaN
32746,20554,1,3,3,1,1,NaN,NaN,NaN,NaN


#### ABW_BMDEF_ANAL_PT_STL

In [89]:
df = get_table("BRIDGEWARE", "ABW_BMDEF_ANAL_PT_STL")
df

,bridge_id,struct_def_id,spng_mbr_def_id,anal_pt_id,web_angle,trans_stiff_spacing,trans_stiff_thick,trans_stiff_width,trans_stiff_stl_id,tens_field_action_ind,...,bearing_stiff_attachment_type,long_stiff_vert_dist_ref_type,override_diaph_ind,dist_diaph_left,dist_diaph_right,diaph_at_point_ind,trans_stiff_type,override_long_stiff_ind,override_trans_stiff_ind,override_bearing_stiff_ind
0,3192,1,3,1,None,None,None,None,1,F,...,26901,27001,F,None,None,F,33401,F,F,F
1,3192,1,3,2,None,None,None,None,1,F,...,26901,27001,F,None,None,F,33401,F,F,F
2,3192,1,3,3,None,None,None,None,1,F,...,26901,27001,F,None,None,F,33401,F,F,F
3,3373,1,3,1,None,None,None,None,1,F,...,26901,27001,F,None,None,F,33401,F,F,F
4,3373,1,3,2,None,None,None,None,1,F,...,26901,27001,F,None,None,F,33401,F,F,F
5,3373,1,3,3,None,None,None,None,1,F,...,26901,27001,F,None,None,F,33401,F,F,F


#### ABW_METAL_BOX_CULVERT_DEF

In [90]:
df = get_table("BRIDGEWARE", "ABW_METAL_BOX_CULVERT_DEF")
df

,bridge_id,struct_def_id,culvert_def_id,span,rise,rc,rh,delta,d,l,...,section_properties_mp_haunch,mp_crown_adjustment_factor,mp_haunch_adjustment_factor,stl_structural_id,aluminum_id,pavement_reduct_factor,pavement_reduct_fact_comment,wearing_surface_density,wearing_surface_thickness,thickness_field_measured_ind
0,18086,2,1,2.920999,1.244590,7.315200,0.914400,1.047198,1.219200,0.914400,...,47.284600,100.0,100.0,NaN,2.0,100.0,None,None,None,None
1,16049,3,1,6.096000,1.828800,6.096000,1.524000,1.047198,1.219200,0.914400,...,112.540017,100.0,100.0,3.0,NaN,100.0,None,None,None,None
2,16049,3,4,4.876800,1.828800,6.096000,1.524000,1.047198,1.219200,0.914400,...,89.364780,100.0,100.0,3.0,1.0,100.0,None,None,None,None
3,19886,3,3,6.654790,2.158990,7.556510,0.768349,1.221730,0.819150,0.577850,...,112.540017,100.0,100.0,1.0,NaN,100.0,None,None,None,None
4,20539,2,1,5.740390,1.727302,7.556501,0.965198,1.221730,0.471488,0.395288,...,114.675163,95.0,95.0,1.0,NaN,NaN,None,None,None,None


#### ABW_MBR_ALT_TIMBER_DECK_RANGE

In [91]:
df = get_table("BRIDGEWARE", "ABW_MBR_ALT_TIMBER_DECK_RANGE")
df

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,deck_range_id,dist,length,timber_deck_type,deck_timber_id,deck_total_thick,...,adj_deck_volume_factor,adj_deck_flat_use_factor,adj_deck_rep_use_factor,adj_deck_load_sharing_factor,adj_deck_load_dur_factor,shear_flex_moisture_cond_type,bearing_moisture_cond_type,stiffness_moisture_cond_type,deck_nominal_thick,deck_nominal_width
0,16762,3,1,1,1,None,5.79120,36301.0,NaN,NaN,...,None,None,NaN,None,None,36101,36101,36101,NaN,NaN
1,16762,3,2,1,1,None,5.79120,36301.0,NaN,NaN,...,None,None,NaN,None,None,36101,36101,36101,NaN,NaN
2,20225,1,1,1,1,None,7.81050,36301.0,4.0,0.0,...,None,None,NaN,None,None,36101,36101,36101,0.0,0.0
3,25,1,2,1,1,None,5.18160,NaN,NaN,NaN,...,None,None,1.15,None,None,36101,36101,36101,NaN,NaN
4,363,1,1,1,1,None,NaN,36301.0,1.0,NaN,...,None,None,NaN,None,None,36101,36101,36101,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,19163,9,3,1,1,None,16.45920,36301.0,NaN,NaN,...,None,None,NaN,None,None,36101,36101,36101,NaN,NaN
179,19163,9,4,1,1,None,16.45920,36301.0,NaN,NaN,...,None,None,NaN,None,None,36101,36101,36101,NaN,NaN
180,20521,1,1,1,1,None,6.70560,36301.0,4.0,0.0,...,None,None,NaN,None,None,36101,36101,36101,0.0,0.0
181,20522,1,1,1,1,None,11.78558,36301.0,4.0,0.0,...,None,None,NaN,None,None,36101,36101,36101,0.0,0.0


#### ABW_MBR_ALT_TRUSS_LINE_SUPPORT

In [92]:
df = get_table("BRIDGEWARE", "ABW_MBR_ALT_TRUSS_LINE_SUPPORT")
df

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,support_id,truss_line_support_id,bearing_dist_left,bearing_dist_right,timber_brg_area_length,timber_brg_area_width


#### ABW_MBR_COLUMN_STIFFNESS

In [93]:
df = get_table("BRIDGEWARE", "ABW_MBR_COLUMN_STIFFNESS")
df

,bridge_id,struct_def_id,super_struct_mbr_id,column_stiffness_id,column_mbr_id,percent_stiffness


#### ABW_MBR_CONCENT_LOAD

In [94]:
df = get_table("BRIDGEWARE", "ABW_MBR_CONCENT_LOAD")
df

,bridge_id,struct_def_id,super_struct_mbr_id,mbr_concent_load_id,dist,force_x,force_y,force_z,moment_x,moment_y,moment_z,load_case_id,local_axis_ind,flexure_percent_of_load,shear_percent_of_load,description,apply_to_full_box_only_ind
0,1311,1,3,6,29.477909,NaN,1.957218,None,None,None,NaN,1.0,None,NaN,NaN,None,None
1,1311,1,2,1,3.560064,NaN,1.512395,None,None,None,NaN,1.0,None,NaN,NaN,None,None
2,1311,1,2,2,7.120128,NaN,1.512395,None,None,None,NaN,1.0,None,NaN,NaN,None,None
3,1311,1,2,3,10.683240,NaN,1.512395,None,None,None,NaN,1.0,None,NaN,NaN,None,None
4,19496,1,3,1,3.149498,NaN,1.641394,None,None,None,NaN,3.0,None,100.0,100.0,Bracket point of water pipe,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13873,20425,1,4,2,44.007329,NaN,3.164020,None,None,None,NaN,1.0,None,100.0,100.0,W36x230 splice weight,None
13874,20425,1,4,3,58.710881,NaN,1.818433,None,None,None,NaN,1.0,None,100.0,100.0,Transition splice weight,None
13875,20425,1,5,1,19.550177,NaN,1.818433,None,None,None,NaN,1.0,None,100.0,100.0,Transition splice weight,None
13876,20425,1,5,2,44.007329,NaN,3.164020,None,None,None,NaN,1.0,None,100.0,100.0,W36x230 splice weight,None


#### ABW_FRAME_COLUMN_MBR

In [95]:
df = get_table("BRIDGEWARE", "ABW_FRAME_COLUMN_MBR")
df

,bridge_id,struct_def_id,super_struct_mbr_id,support_line_id,frame_leg_def_id,name,description,angle,length,column_distance_display_type,...,y_rotation_spring_constant,z_rotation_spring_constant,x_translation_settlement,y_translation_settlement,z_translation_settlement,x_rotation_settlement,y_rotation_settlement,z_rotation_settlement,settlement_load_case_id,bent_cap_width


#### ABW_FRAME_LEG_BRACE_LOC

In [96]:
df = get_table("BRIDGEWARE", "ABW_FRAME_LEG_BRACE_LOC")
df

,bridge_id,struct_def_id,leg_brace_loc_id,right_spng_mbr_id,right_leg_id,left_spng_mbr_id,left_leg_id,left_leg_dist,right_leg_dist,num_spaces,spacing,brace_weight


#### ABW_FRAME_LEG_MBR_DEF

In [97]:
df = get_table("BRIDGEWARE", "ABW_FRAME_LEG_MBR_DEF")
df

,bridge_id,struct_def_id,frame_leg_def_id,frame_leg_mbr_def_type,name,descr,additional_self_weight,conn_self_weight_percent,creation_event_id,last_modified_event_id,comment_id


#### ABW_FRAME_MBR_LEG

In [98]:
df = get_table("BRIDGEWARE", "ABW_FRAME_MBR_LEG")
df

,bridge_id,struct_def_id,super_struct_mbr_id,leg_id,name,descr,frame_joint_support_line_id,fline_joint_node_id,angle,length,...,x_rotation_spring_constant,y_rotation_spring_constant,z_rotation_spring_constant,x_translation_settlement,y_translation_settlement,z_translation_settlement,x_rotation_settlement,y_rotation_settlement,z_rotation_settlement,bent_cap_width


#### ABW_MCB_SEG_CONC_CURB_SIDEWALK

In [99]:
df = get_table("BRIDGEWARE", "ABW_MCB_SEG_CONC_CURB_SIDEWALK")
df

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,curb_id,load_case_id,use_type,offset_ref_type,conc_id,straight_ind,...,offset_end,bot_width_start,top_width_start,bot_width_end,top_width_end,avg_thick,conc_density,measured_to_left_face_ind,pedestrian_load,bridge_ref_line_id
0,4275,2,1,1,1,2,20102,20202,2,None,...,0.0,6058.0,None,None,None,190.0,None,F,None,None


#### ABW_DECK_PANEL_EVENT_ERRORS

In [100]:
df = get_table("BRIDGEWARE", "ABW_DECK_PANEL_EVENT_ERRORS")
df

,deck_panel_analysis_event_id,error_id,component_guid,component_error_id


#### ABW_FLNG_LAT_BEND_STRESS_SUP

In [101]:
df = get_table("BRIDGEWARE", "ABW_FLNG_LAT_BEND_STRESS_SUP")
df

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,flng_lat_bend_stress_id,flng_lat_bend_stress_sup_id,support_num,girder_reaction_adj_factor
0,16479,1,1,1,1,1,1,None
1,16479,1,1,1,1,2,2,None
2,16479,1,1,1,1,3,3,None
3,16479,1,1,1,2,1,1,None
4,16479,1,1,1,2,2,2,None
...,...,...,...,...,...,...,...,...
17664,20566,1,8,1,1,2,2,None
17665,20566,1,8,1,2,1,1,None
17666,20566,1,8,1,2,2,2,None
17667,20566,1,8,1,3,1,1,None


#### ABW_STRGRP_LL_DISTFACT_RANGE

In [102]:
df = get_table("BRIDGEWARE", "ABW_STRGRP_LL_DISTFACT_RANGE")
df

,bridge_id,struct_def_id,floorsys_stringer_group_def_id,template_id,range_id,dist,length,single_lane_ll_distfactor,multi_lane_ll_distfactor,action_type


#### ABW_STRINGER_DL_REACT_DETAIL

In [103]:
df = get_table("BRIDGEWARE", "ABW_STRINGER_DL_REACT_DETAIL")
df

,bridge_id,struct_def_id,floor_beam_mbr_id,dl_reaction_id,dl_reaction_detail_id,dl_case_id,computed_reaction,results_timestamp
0,4760,2,30,1,2,2,NaN,NaT
1,4760,2,30,2,2,7,NaN,NaT
2,4760,2,31,1,1,1,NaN,NaT
3,4760,2,31,2,1,8,NaN,NaT
4,4760,2,31,1,2,2,NaN,NaT
...,...,...,...,...,...,...,...,...
43736,20416,12,64,47,3,4,0.0,2025-06-04 14:59:42.133
43737,20416,12,64,48,1,5,0.0,2025-06-04 14:59:42.133
43738,20416,12,64,48,2,10,0.0,2025-06-04 14:59:42.133
43739,20416,12,64,48,3,8,0.0,2025-06-04 14:59:42.133


#### ABW_CONC_BMDEF_SECTION_PROFILE

In [104]:
df = get_table("BRIDGEWARE", "ABW_CONC_BMDEF_SECTION_PROFILE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,section_profile_id,start_distance,length,start_width,end_width,conc_id,modular_ratio
0,4201,1,1,1,0.0,39.624000,13411.200000,13411.200000,1,8.00
1,4056,1,1,1,0.0,23.774400,8534.400000,8534.400000,3,8.00
2,4146,1,1,1,0.0,10.839450,304.800000,304.800000,1,7.00
3,4204,1,1,1,0.0,43.586400,13411.200000,13411.200000,1,8.00
4,678,2,1,1,0.0,8.229600,11125.200000,11125.200000,2,9.00
...,...,...,...,...,...,...,...,...,...,...
3571,20434,1,1,1,0.0,43.586400,19085.911523,21266.077412,1,7.27
3572,20435,1,1,1,0.0,43.586400,13106.400000,13106.400000,1,7.27
3573,20438,1,1,1,0.0,43.586400,17373.600000,17373.600000,1,7.27
3574,20438,1,3,1,0.0,44.168343,1080.674674,3665.437449,1,7.27


#### ABW_CONC_BMDEF_VOID_RANGE

In [105]:
df = get_table("BRIDGEWARE", "ABW_CONC_BMDEF_VOID_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,void_range_id,section_pattern_id,start_distance,length
0,4980,2,1,1,1,1.2954,9.2202
1,4980,2,1,2,1,10.8204,1.9050
2,4980,2,1,3,1,15.3162,13.6398
3,4980,2,1,4,1,32.8422,1.9050
4,4980,2,1,5,1,35.0520,9.2202
5,4128,1,1,1,1,0.6096,30.7848
6,4128,1,1,2,1,32.6136,38.4048
7,4128,1,1,3,1,72.2376,30.7848
8,15346,3,1,1,2,1.0668,13.1064
9,15346,3,1,2,2,16.3068,16.9164


#### ABW_CONC_BMDEF_WEB_PROFILE

In [106]:
df = get_table("BRIDGEWARE", "ABW_CONC_BMDEF_WEB_PROFILE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,web_profile_id,start_dist,length,start_depth,end_depth,web_depth_variation_type,top_begin_width,top_end_width,bottom_begin_width,bottom_end_width
0,290,2,1,19,13.309597,0.7874,1263.6500,1144.5875,26201,None,None,None,None
1,290,2,1,20,14.096997,0.7874,1144.5875,1036.6375,26201,None,None,None,None
2,290,2,1,21,14.884397,0.7874,1036.6375,938.2125,26201,None,None,None,None
3,290,2,1,22,15.671797,0.7874,938.2125,850.9000,26201,None,None,None,None
4,290,2,1,23,16.459197,0.7874,850.9000,773.1125,26201,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4270,20531,2,1,1,0.000000,30.7086,463.5500,463.5500,26202,None,None,None,None
4271,20534,2,1,1,0.000000,46.9392,482.6000,482.6000,26202,None,None,None,None
4272,20536,2,2,1,0.000000,44.5008,622.3000,622.3000,26202,None,None,None,None
4273,20537,2,2,1,0.000000,44.5008,622.3000,622.3000,26202,None,None,None,None


#### ABW_PS_PRECAST_BEAM_DEF

In [107]:
df = get_table("BRIDGEWARE", "ABW_PS_PRECAST_BEAM_DEF")
df

,bridge_id,struct_def_id,spng_mbr_def_id,ps_precast_beam_def_type,curing_method_type,curing_time,ignore_pos_supp_mom_rating_ind,lrfd_loss_calc_method_type,lrfr_loss_calc_method_type,lfr_shear_comp_method_type,...,lrfd_cons_split_resist_art_ind,lrfr_cons_split_resist_art_ind,lrfr_ignore_tens_rate_top_ind,asr_con_deck_reinf_devlen_ind,lfr_con_deck_reinf_devlen_ind,lrfr_con_deck_reinf_devlen_ind,lrfd_con_deck_reinf_devlen_ind,lrfr_mod_mcft_size_effect_ind,stl_brng_pl_emb_st_gr_end_ind,allow_cracking_girder_ends_ind
0,2604,1,1,24112,NaN,NaN,None,49101,49101,49403.0,...,F,F,F,F,F,F,F,None,None,None
1,2604,1,2,24112,NaN,NaN,None,49101,49101,49403.0,...,F,F,F,F,F,F,F,None,None,None
2,1956,3,3,24111,NaN,NaN,None,49101,49101,49403.0,...,F,F,F,F,F,F,F,None,None,None
3,1956,3,4,24111,NaN,NaN,None,49101,49101,49403.0,...,F,F,F,F,F,F,F,None,None,None
4,1956,3,5,24111,NaN,NaN,None,49101,49101,49403.0,...,F,F,F,F,F,F,F,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16331,20542,3,6,24112,34802.0,30.0,None,49101,49101,49403.0,...,F,F,F,None,F,F,F,None,None,None
16332,20542,3,7,24112,34802.0,30.0,None,49101,49101,49403.0,...,F,F,F,None,F,F,F,None,None,None
16333,20542,3,8,24112,34802.0,30.0,None,49101,49101,49403.0,...,F,F,F,None,F,F,F,None,None,None
16334,20548,1,1,24111,34802.0,NaN,None,49101,49101,49403.0,...,F,F,F,None,F,F,F,None,None,None


#### ABW_CONC_BMDEF_WEB_WIDTH

In [108]:
df = get_table("BRIDGEWARE", "ABW_CONC_BMDEF_WEB_WIDTH")
df

,bridge_id,struct_def_id,spng_mbr_def_id,web_width_id,top_begin_width,top_end_width,bot_begin_width,bot_end_width,beam_def_support_id,start_dist,length
0,12854,3,1,1,508.000,508.000,508.000,508.000,None,0.0000,46.761502
1,909,3,2,1,1739.900,1739.900,1739.900,1739.900,None,0.0000,0.228600
2,909,3,2,2,323.850,323.850,323.850,323.850,None,0.2286,10.680700
3,909,3,2,3,1739.900,1739.900,1739.900,1739.900,None,10.9093,0.228600
4,909,3,4,1,1498.600,1498.600,1498.600,1498.600,None,0.0000,0.228600
5,909,3,4,2,419.100,419.100,419.100,419.100,None,0.2286,10.680700
6,909,3,4,3,1498.600,1498.600,1498.600,1498.600,None,10.9093,0.228600
7,909,3,3,1,1538.224,1538.224,1538.224,1538.224,None,0.0000,0.228600
8,909,3,3,2,419.100,419.100,419.100,419.100,None,0.2286,10.680700
9,909,3,3,3,1538.224,1538.224,1538.224,1538.224,None,10.9093,0.228600


#### ABW_CONC_BMDF_VOID_SEC_PAT_DET

In [109]:
df = get_table("BRIDGEWARE", "ABW_CONC_BMDF_VOID_SEC_PAT_DET")
df

,bridge_id,struct_def_id,spng_mbr_def_id,section_pattern_id,pattern_detail_id,dist_left_edge,num_spaces,void_spacing,void_diameter
0,4980,2,1,1,1,0.431800,18,431.8000,254.0
1,4128,1,1,1,1,0.508102,13,762.0000,457.2
2,15346,3,1,2,1,0.381000,17,510.9718,355.6
3,5198,1,1,1,1,0.533400,12,736.6000,508.0


#### ABW_STL_STRUCT_TEE

In [110]:
df = get_table("BRIDGEWARE", "ABW_STL_STRUCT_TEE")
df

,bridge_id,stl_shape_id,shape_type,nominal_weight_or_mass,area,tee_depth_d,stem_thick_tw,stem_area,flng_width_bf,flng_thick_tf,dist_k,dist_y_to_cg,ixx,iyy
0,17143,3,23703,18.900000,2399.9952,127.000,7.8994,NaN,118.3640,12.4714,28.7020,30.4800,3.242443e+06,1.398538e+06
1,18281,2,23701,33.484252,4232.2496,153.162,8.5090,NaN,204.4700,14.6050,27.4320,28.7020,6.909442e+06,1.040579e+07
2,18336,2,23701,36.460631,4651.6036,126.746,8.6360,NaN,254.0000,14.2240,30.1625,20.4978,4.162314e+06,1.943801e+07
3,18368,1,23701,33.484252,4232.2496,153.162,8.5090,NaN,204.4700,14.6050,27.4320,28.7020,6.909442e+06,1.040579e+07
4,18368,2,23701,61.015749,7741.9200,181.864,12.9540,NaN,256.5400,21.7170,36.8300,35.3060,1.714873e+07,3.084275e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,17737,4,23701,22.322835,2851.6072,133.096,7.6200,NaN,147.5740,12.9540,20.5740,27.9400,3.862628e+06,3.475532e+06
188,17744,4,23701,33.484252,4232.2496,153.162,8.5090,NaN,204.4700,14.6050,27.4320,28.7020,6.909442e+06,1.040579e+07
189,17744,5,23701,108.637797,13870.9400,314.960,16.5100,NaN,327.6600,27.6860,40.3860,67.5640,1.098851e+08,8.116513e+07
190,17747,3,23703,31.921654,4070.9596,190.500,10.4394,NaN,139.7254,15.7988,34.9250,51.0540,1.373564e+07,2.992704e+06


#### ABW_SUB_STRUCT_WS_LOAD

In [111]:
df = get_table("BRIDGEWARE", "ABW_SUB_STRUCT_WS_LOAD")
df

,bridge_id,struct_def_id,sub_ws_load_id,sub_struct_def_component_id,design_height_z,wind_pressure_pd,height_operator_type,comparison_height,sys_design_water_level_id,wind_pressure_pz,sys_lrfd_ls_id
0,3717,2,169,1,None,None,42702,9.144,None,None,3.0
1,3717,2,170,1,None,None,42702,9.144,None,None,5.0
2,3717,2,171,1,None,None,42702,9.144,None,None,6.0
3,3717,2,172,1,None,None,42702,9.144,None,None,9.0
4,3717,2,173,3,None,None,42702,9.144,None,None,3.0
...,...,...,...,...,...,...,...,...,...,...,...
118,13291,6,16,1,None,None,42702,9.144,None,None,NaN
119,13291,6,17,3,None,None,42702,9.144,None,None,NaN
120,13291,6,18,5,None,None,42702,9.144,None,None,NaN
121,13291,6,19,7,None,None,42702,9.144,None,None,NaN


#### ABW_SUB_STRUCT_WS_SKEW

In [112]:
df = get_table("BRIDGEWARE", "ABW_SUB_STRUCT_WS_SKEW")
df

,bridge_id,struct_def_id,sub_ws_load_id,skew_angle_id,skew_angle,calc_wind_press_pd_long,calc_wind_press_pd_trans,ovr_wind_press_pd_long,ovr_wind_press_pd_trans,calc_wind_press_pd_long_ul,...,ovr_wind_press_pd_long_ul,ovr_wind_press_pd_trans_ul,calc_wind_press_pz_long,calc_wind_press_pz_trans,ovr_wind_press_pz_long,ovr_wind_press_pz_trans,calc_wind_press_pz_long_ul,calc_wind_press_pz_trans_ul,ovr_wind_press_pz_long_ul,ovr_wind_press_pz_trans_ul
0,3717,2,169,1,0.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,3717,2,169,2,15.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,3717,2,169,3,30.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,3717,2,169,4,45.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,3717,2,169,5,60.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
610,13291,6,20,1,0.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
611,13291,6,20,2,15.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
612,13291,6,20,3,30.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
613,13291,6,20,4,45.0,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


#### ABW_SUBS_STREAM_FLOW

In [113]:
df = get_table("BRIDGEWARE", "ABW_SUBS_STREAM_FLOW")
df

,bridge_id,bridge_design_alt_id,sub_struct_id,stream_flow_info_id,sys_design_water_level_id,water_elevation,design_velocity,scour_elevation,consider_ice_dynamic_ind,consider_ice_hanging_dam_ind,consider_ice_jam_ind,consider_ice_adhesion_ind,consider_stream_flow_ind
0,14034,1,12,1,1,0.0,0.0,0.0,F,F,F,F,F
1,14034,1,12,2,2,0.0,0.0,0.0,F,F,F,F,F
2,14034,1,12,3,3,0.0,0.0,0.0,F,F,F,F,F
3,14034,1,12,4,4,0.0,0.0,0.0,F,F,F,F,F
4,14034,1,13,1,1,0.0,0.0,0.0,F,F,F,F,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7675,19449,2,12,3,3,0.0,0.0,0.0,F,F,F,F,F
7676,19449,2,12,4,4,0.0,0.0,0.0,F,F,F,F,F
7677,19449,2,13,1,1,0.0,0.0,0.0,F,F,F,F,F
7678,19449,2,13,2,2,0.0,0.0,0.0,F,F,F,F,F


#### ABW_MCB_WEB_SHEAR_REINF_RANGE

In [114]:
df = get_table("BRIDGEWARE", "ABW_MCB_WEB_SHEAR_REINF_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,web_id,range_id,span_num,shear_reinf_def_id,start_dist,num_spaces,spacing
0,3496,1,1,1,1,1,None,1,0.050810,4,273.050
1,3496,1,1,1,1,2,None,1,0.050810,1,0.000
2,4275,1,1,1,1,1,None,2,1.295000,1,0.000
3,4275,1,1,1,1,2,None,2,1.295000,30,150.000
4,4275,1,1,1,1,3,None,2,5.795000,164,149.604
...,...,...,...,...,...,...,...,...,...,...,...
192,4275,2,1,1,1,12,None,1,1.295000,94,150.000
193,4275,2,1,1,1,13,None,1,15.395000,75,198.560
194,4275,2,1,1,1,14,None,4,30.287000,43,150.000
195,4275,2,1,1,1,15,None,1,39.327439,1,0.000


#### ABW_MCELL_BOX_BDEF_SLAB_REINF

In [115]:
df = get_table("BRIDGEWARE", "ABW_MCELL_BOX_BDEF_SLAB_REINF")
df

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,reinf_id,reinf_type,overhang_type,cell_type,cell_num,stl_reinf_id,...,num_bars_left_web,rebar_id,clear_cover,measured_from_type,bar_spacing,side_cover,start_fully_developed_ind,end_fully_developed_ind,head_at_start_ind,head_at_end_ind
0,5236,1,1,1,27,55601,NaN,55801.0,NaN,1,...,0.0,14,79.375,55401,381.0,76.2,F,F,None,None
1,5236,1,1,1,28,55602,NaN,55801.0,NaN,1,...,0.0,14,53.975,55402,381.0,76.2,F,F,None,None
2,5236,1,1,1,29,55603,55701.0,NaN,NaN,1,...,NaN,14,79.375,55401,381.0,76.2,F,F,None,None
3,5236,1,1,1,30,55603,55701.0,NaN,NaN,1,...,NaN,14,361.950,55401,381.0,76.2,F,F,None,None
4,3496,1,1,1,1,55601,NaN,55801.0,NaN,1,...,6.5,13,88.900,55401,444.5,50.8,F,F,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,4275,2,1,1,50,55602,NaN,55801.0,NaN,1,...,NaN,5,40.000,55402,300.0,60.0,T,T,None,None
65,4275,2,1,1,51,55602,NaN,55801.0,NaN,1,...,NaN,5,25.000,55401,300.0,60.0,T,T,None,None
66,4275,2,1,1,52,55602,NaN,55801.0,NaN,1,...,NaN,5,40.000,55402,300.0,60.0,T,T,None,None
67,4275,2,1,1,53,55603,55701.0,NaN,NaN,1,...,NaN,3,65.000,55401,450.0,60.0,T,T,None,None


#### ABW_MULTI_CELL_BOX_BDEF_WEB_FK

In [116]:
df = get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_BDEF_WEB_FK")
df

,bridge_id,struct_def_id,spng_mbr_def_id,segment_id,web_id,same_as_web_id
0,5236,1,1,1,1,NaN
1,5236,1,1,1,2,NaN
2,5236,1,1,1,3,NaN
3,5236,1,1,1,4,NaN
4,5236,1,1,1,5,NaN
5,5236,1,1,1,6,NaN
6,5236,1,1,1,7,NaN
7,5236,1,1,1,8,NaN
8,5236,1,1,1,9,NaN
9,5236,1,1,1,10,NaN


#### ABW_RESULTS_CRIT_LOAD_LFR

In [117]:
df = get_table("BRIDGEWARE", "ABW_RESULTS_CRIT_LOAD_LFR")
df

,spng_mbr_alt_event_id,point_id,crit_load_lfr_id,load_group_id,stage_id,moment_max,moment_max_cvehicle_id,moment_min,moment_min_cvehicle_id,axial_max,...,shear_min,shear_min_cvehicle_id,reaction_max,reaction_max_cvehicle_id,reaction_min,reaction_min_cvehicle_id,y_defl_max,y_defl_max_cvehicle_id,y_defl_min,y_defl_min_cvehicle_id
0,105,649,742,1,3,6861.055421,5.0,1281.403088,5.0,0.0,...,-518.064704,5.0,NaN,NaN,NaN,NaN,147.409725,5.0,60.879546,5.0
1,105,650,743,1,3,5701.508627,5.0,144.848060,5.0,0.0,...,-723.122709,5.0,NaN,NaN,NaN,NaN,131.286772,5.0,46.941334,5.0
2,105,651,744,1,3,3729.623280,5.0,-1455.980379,5.0,0.0,...,-932.392563,5.0,NaN,NaN,NaN,NaN,102.977826,5.0,28.447419,5.0
3,105,652,745,1,3,955.224757,5.0,-3540.309247,5.0,0.0,...,-1158.150825,5.0,NaN,NaN,NaN,NaN,67.446505,5.0,11.251136,5.0
4,105,653,746,1,3,-2339.803576,5.0,-6575.267126,5.0,0.0,...,-1388.869083,5.0,NaN,NaN,NaN,NaN,30.975973,5.0,0.262746,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16870,60,5027,15442,1,3,3358.866935,NaN,8.567322,NaN,0.0,...,215.630436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16871,60,5028,15443,1,3,3157.667217,NaN,174.670283,NaN,0.0,...,172.781258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16872,60,5029,15444,1,3,3078.051511,NaN,184.159844,NaN,0.0,...,172.413795,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16873,60,5030,15445,1,3,2588.648859,NaN,242.492758,NaN,0.0,...,157.289587,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### ABW_SUPER_STRUCT_ALT

In [118]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_ALT")
df

,bridge_id,bridge_design_alt_id,super_struct_id,super_struct_alt_id,name,descr,struct_def_id,last_modified_event_id,creation_event_id,comment_id,last_change_timestamp
0,5710,1,1,1,Existing,None,1.0,970398.0,454132.0,None,2020-10-08 20:25:28.000000
1,610,1,1,1,Ex,None,2.0,1016908.0,16283.0,None,2020-10-08 22:48:15.000000
2,620,1,1,1,Ex,None,2.0,1016830.0,16406.0,None,2020-10-08 22:47:52.000000
3,621,1,1,1,BARS Structure Alt,Defined by BARS Import,1.0,1016822.0,16415.0,None,2020-10-08 22:47:50.000000
4,3732,1,1,1,Existing PSBB,None,1.0,NaN,454182.0,None,2019-01-11 21:39:19.000000
...,...,...,...,...,...,...,...,...,...,...,...
13636,11803,4,1,1,WOO-75-3038,None,3.0,NaN,NaN,None,2023-11-30 20:36:42.279631
13637,18230,1,1,1,LAW-00141-1298,None,1.0,NaN,NaN,None,2024-05-15 17:26:32.053000
13638,18231,1,1,1,Bridge,None,1.0,NaN,NaN,None,2024-07-11 20:26:45.147000
13639,16929,1,1,1,PRE-127-19.11,None,1.0,NaN,NaN,None,2021-10-05 12:52:16.000000


#### ABW_GUSSET_PARTIAL_SHEAR_PLANE

In [119]:
df = get_table("BRIDGEWARE", "ABW_GUSSET_PARTIAL_SHEAR_PLANE")
df

,bridge_id,struct_def_id,gusset_def_id,gusset_plate_def_id,gusset_partial_shear_plane_id,gusset_plate_member_def_id,shear_plane_direction_type,length,thickness,advanced_options_ind,override_angle
0,3192,1,1,1,1,4,58701,818.642,NaN,None,None
1,3192,1,1,1,2,4,58702,422.910,NaN,None,None
2,3192,1,2,1,1,6,58701,736.600,15.875,None,None
3,3192,1,2,1,2,6,58702,533.400,15.875,None,None
4,3192,1,2,1,3,4,58701,558.800,15.875,None,None
...,...,...,...,...,...,...,...,...,...,...,...
72,13078,1,8,1,1,4,58702,752.602,NaN,None,None
73,13078,1,9,1,1,4,58702,645.160,NaN,None,None
74,13078,1,10,1,1,4,58702,491.744,NaN,None,None
75,13078,1,11,1,1,4,58702,377.444,NaN,None,None


#### ABW_SUPER_STRUCT_DL_REACT_SUB

In [120]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_DL_REACT_SUB")
df

,bridge_id,sub_struct_def_id,dl_reaction_set_id,span_location_type,up_to_date_ind,results_timestamp,override_ind
0,5564,3,1,42403,T,2018-11-27 17:35:10,F
1,5564,4,1,42403,T,2018-11-27 18:22:31,F
2,5564,5,1,42403,T,2018-11-27 18:27:11,F
3,5564,6,1,42403,T,2018-11-27 18:32:23,F
4,13291,4,1,42403,F,NaT,F


#### ABW_SUPER_STRUCT_LL_REACT_SUB

In [121]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_LL_REACT_SUB")
df

,bridge_id,sub_struct_def_id,ll_reaction_set_id,span_location_type,up_to_date_ind,results_timestamp
0,5564,3,1,42403,None,2018-11-27 17:35:49
1,5564,4,1,42403,None,2018-11-27 18:22:50
2,5564,5,1,42403,None,2018-11-27 18:27:31
3,5564,6,1,42403,None,2018-11-27 18:32:37
4,13291,4,1,42403,None,NaT


#### ABW_SUPER_STRUCT_MBR

In [122]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_MBR")
df

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_mbr_type,last_modified_event_id,creation_event_id,comment_id,name,descr,last_change_timestamp
0,1020,1,1,25710,1013093.0,33957.0,None,S01,None,2000-01-01 12:00:00.000
1,18838,2,9,25705,NaN,NaN,None,G9,None,2024-10-14 00:10:27.387
2,18838,2,10,25705,NaN,NaN,None,G10,None,2024-10-17 13:36:28.137
3,1040,1,2,25705,1051130.0,35278.0,None,G2,None,2000-01-01 12:00:00.000
4,1040,1,3,25705,1051131.0,35279.0,None,G3,None,2000-01-01 12:00:00.000
...,...,...,...,...,...,...,...,...,...,...
85377,19778,1,3,25705,NaN,NaN,None,G3,None,2025-02-10 17:56:56.337
85378,19778,1,4,25705,NaN,NaN,None,G4,None,2025-02-11 03:16:20.453
85379,19779,2,1,25706,NaN,NaN,None,S1,None,2025-02-04 15:17:37.803
85380,19780,1,1,25706,NaN,NaN,None,S1,None,2025-02-12 15:04:56.107


#### ABW_LIB_BOLT

In [123]:
df = get_table("BRIDGEWARE", "ABW_LIB_BOLT")
df

,lib_bolt_id,name,descr,library_type,si_or_us_type,lfd_bearing_shear_threads_inc,lfd_bearing_shear_threads_exc,lfd_slip_class_a_std,lfd_slip_class_a_oversize,lfd_slip_class_a_long_trans,...,asd_slip_class_c_oversize,asd_slip_class_c_long_trans,asd_slip_class_c_long_parallel,asd_bearing_shear_threads_inc,asd_bearing_shear_threads_exc,lrfd_ks_class_d,lrfd_kc_class_a,lrfd_kc_class_b,lrfd_kc_class_c,lrfd_kc_class_d
0,13,ASTM F3148 Grade 144,ASTM F3148 Grade 144,22901,10401,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.45,NaN,NaN,NaN,NaN
1,1,AASHTO M 164 (US),AASHTO M 164 (ASTM A 325),22901,10401,241.316495,296.474551,144.789897,124.105626,103.421355,...,89.631841,75.842327,62.052813,131.000383,163.750479,NaN,1.0,1.0,0.8,1.0
2,2,AASHTO M 253 (US),AASHTO M 253 (ASTM A 490),22901,10401,296.474551,365.422121,179.263682,151.684654,124.105626,...,110.316112,89.631841,75.842327,165.474168,206.842710,NaN,1.0,1.0,0.8,1.0
3,3,AASHTO M 164 (SI),AASHTO M 164 (ASTM A 325M),22901,10402,241.318000,301.647500,144.790800,124.106400,103.422000,...,89.632400,75.842800,62.053200,131.001200,163.751500,NaN,1.0,1.0,0.8,1.0
4,4,AASHTO M 253 (SI),AASHTO M 253 (ASTM A 490M),22901,10402,296.476000,370.596000,179.264800,151.685600,124.106400,...,110.316800,89.632400,75.842800,165.480000,206.840000,NaN,1.0,1.0,0.8,1.0
5,5,ASTM A 307,ASTM A 307,22901,10401,124.105626,124.105626,NaN,NaN,NaN,...,NaN,NaN,NaN,75.842327,75.842327,NaN,1.0,1.0,0.8,1.0
6,6,ASTM A 502 - Grade 1,ASTM A 502 - Grade 1,22901,10401,172.368925,172.368925,NaN,NaN,NaN,...,NaN,NaN,NaN,93.079219,93.079219,NaN,1.0,1.0,0.8,1.0
7,7,ASTM A 502 - Grade 2,ASTM A 502 - Grade 2,22901,10401,206.842710,206.842710,NaN,NaN,NaN,...,NaN,NaN,NaN,137.895140,137.895140,NaN,1.0,1.0,0.8,1.0
8,10,Rivets Unknown origin After 1936,Table 6A.6.12.5.1.1,22902,10401,144.789897,144.789897,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,Rivets Unknown Origin Prior to 1936,Table 6A.6.12.5.1.1,22902,10401,124.105626,124.105626,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### ABW_SUPSTRUCTDEF_BRACING_MBR

In [124]:
df = get_table("BRIDGEWARE", "ABW_SUPSTRUCTDEF_BRACING_MBR")
df

,bridge_id,struct_def_id,bracing_id,mbr_id,name,lrfr_condition_factor_type,lrfr_field_meas_sect_prop_ind,last_change_timestamp
0,5858,1,1,1,AB,46101,F,2019-01-23 14:49:18.000
1,13938,1,1,1,AB,46101,F,2022-02-16 13:36:54.497
2,13938,1,1,2,CD,46101,F,2022-02-16 13:36:54.497
3,13938,1,1,3,AD,46101,F,2022-02-16 13:36:54.497
4,13938,1,1,4,CB,46101,F,2022-02-16 13:36:54.497
5,17598,1,1,1,AB,46101,F,2023-12-27 18:47:22.517
6,17598,1,1,2,CD,46101,F,2023-12-27 18:47:22.517
7,17598,1,1,3,AD,46101,F,2023-12-27 18:47:22.517
8,17598,1,1,4,CB,46101,F,2023-12-27 18:47:22.517


#### ABW_LIB_BOLT_DIAMETER

In [125]:
df = get_table("BRIDGEWARE", "ABW_LIB_BOLT_DIAMETER")
df

,lib_bolt_id,bolt_diameter_id,diameter,lrfd_tensile_strength,lrfd_required_tension
0,13,1,15.875,992.845008,106.757328
1,13,2,19.050,992.845008,155.687770
2,13,3,22.225,992.845008,217.962878
3,13,4,25.400,992.845008,284.686208
4,13,5,28.575,992.845008,355.857760
...,...,...,...,...,...
87,12,4,25.400,1034.214000,284.686200
88,12,5,28.575,1034.214000,355.857800
89,12,6,31.750,1034.214000,453.718700
90,12,7,34.925,1034.214000,538.234900


#### ABW_SYS_DB_MAINTENANCE

In [126]:
df = get_table("BRIDGEWARE", "ABW_SYS_DB_MAINTENANCE")
df

,maintenance_keyword,name,descr,development_timestamp,maintenance_type_flag,maintenance_source,version_applied
0,DB_MIG_740_TO_750,Migration from 7.4.0 to 7.5.0,Migration of BrDR database from version 7.4.0 ...,2023-02-03 12:00:00,MIGRATION,MigrateOracle_740_to_750.sql,7.4.0 (Build 3001)
1,DB_MIG_750_TO_760,Migration from 7.5.0 to 7.6.0,Migration of BrDR database from version 7.5.0 ...,2023-11-02 12:00:00,MIGRATION,MigrateOracle_750_to_760.sql,7.5.0 (Build 3001)
2,REMOVE_ORPHANBEAMDEFS_MIG_DB,Remove Orphaned Beam Defs,Remove Orphaned Beam Defs,2006-10-03 12:01:00,UTILITY PROGRAM,RemoveOrphanedBeamDefs.EXE,5.5.0 (Build 3001)
3,DB_MIG_560_TO_600,Migration from 5.6.0 to 6.0.0,Migration of Virtis/Opis database from version...,2008-07-23 12:00:00,MIGRATION,MigrateOracle_560_to_600.SQL,5.6.0 (Build 3001)
4,DB_MIG_520_TO_530,Migration from 5.2.0 to 5.3.0,Migration of Virtis/Opis database from version...,2005-04-08 12:00:00,MIGRATION,MigrateOracle_520_to_530.SQL,5.2.0 (Build 3001)
5,DB_MIG_520_TO_530_INC_5531,Incident 5531,Database Maintenance,2005-04-08 12:03:00,MIGRATION,MigrateOracle_Inc_5531_520_to_53,5.2.0 (Build 3001)
6,DB_MIG_530_TO_531,Migration from 5.3.0 to 5.3.1,Migration of Virtis/Opis database from version...,2005-06-14 12:00:00,MIGRATION,MigrateOracle_530_to_531.SQL,5.3.0 (Build 3001)
7,UPDATE_TIMESTAMPS_MIG_DB,Convert Timestamps to GMT,Convert data in timestamp fields from local ti...,2003-03-31 12:01:00,UTILITY PROGRAM,UpdateTimestamps.EXE,5.0.0 (Build 3001)
8,DB_MIG_510_TO_511,Migration from 5.1.0 to 5.1.1,Migration of Virtis/Opis database from version...,2004-01-08 12:00:00,MIGRATION,MigrateOracle_510_to_511.SQL,5.1.0 (Build 3001)
9,DB_MIG_511_TO_520,Migration from 5.1.1 to 5.2.0,Migration of Virtis/Opis database from version...,2004-11-23 12:00:00,MIGRATION,MigrateOracle_511_to_520.SQL,5.1.1 (Build 3002)


#### ABW_LIB_MATL_PS_STRAND

In [127]:
df = get_table("BRIDGEWARE", "ABW_LIB_MATL_PS_STRAND")
df

,matl_ps_strand_id,name,descr,library_type,si_or_us_type,strand_type,strand_diameter,strand_area,weight_per_length,ultimate_tensile_strength,yield_strength,mod_of_elast,transfer_length_lrfd,transfer_length_std,epoxy_coated_ind,checksum
0,30,"0.7"" (7W-270)LR","Low relaxation 0.700""/Seven Wire/fpu = 270",22901,10401,34301,17.7800,189.67704,1.488189,1861.584390,1675.425951,196500.5745,1066.80,889.000,F,None
1,1,"3/8"" (7W-270) LR","Low relaxation 3/8""/Seven Wire/fpu = 270",22901,10401,34301,9.5250,54.83860,0.431575,1861.584390,1675.425951,196500.5745,571.50,476.250,F,None
2,2,"7/16"" (7W-270) LR","Low relaxation 7/16""/Seven Wire/fpu = 270",22901,10401,34301,11.1125,74.19340,0.580394,1861.584390,1675.425951,196500.5745,666.75,555.625,F,None
3,3,"1/2"" (7W-270) LR","Low relaxation 1/2""/Seven Wire/fpu = 270",22901,10401,34301,12.7000,98.70948,0.773858,1861.584390,1675.425951,196500.5745,762.00,635.000,F,None
4,4,"9/16"" (7W-270) LR","Low relaxation 9/16""/Seven Wire/fpu = 270",22901,10401,34301,14.2875,123.87072,0.967323,1861.584390,1675.425951,196500.5745,857.25,714.375,F,None
5,5,"0.6"" (7W-270) LR","Low relaxation 0.600""/Seven Wire/fpu = 270",22901,10401,34301,15.2400,139.99972,1.101260,1861.584390,1675.425951,196500.5745,914.40,762.000,F,None
6,6,"1/4"" (7W-250) LR","Low relaxation 1/4""/Seven Wire/fpu = 250",22901,10401,34301,6.3500,23.22576,0.181559,1723.689250,1551.320325,196500.5745,381.00,317.500,F,None
7,7,"3/8"" (7W-250) LR","Low relaxation 3/8""/Seven Wire/fpu = 250",22901,10401,34301,9.5250,51.61280,0.404787,1723.689250,1551.320325,196500.5745,571.50,476.250,F,None
8,8,"7/16"" (7W-250) LR","Low relaxation 7/16""/Seven Wire/fpu = 250",22901,10401,34301,11.1125,69.67728,0.546165,1723.689250,1551.320325,196500.5745,666.75,555.625,F,None
9,9,"1/2"" (7W-250) LR","Low relaxation 1/2""/Seven Wire/fpu = 250",22901,10401,34301,12.7000,92.90304,0.729213,1723.689250,1551.320325,196500.5745,762.00,635.000,F,None


#### ABW_SYS_DB_MAINTENANCE_STAGE

In [128]:
df = get_table("BRIDGEWARE", "ABW_SYS_DB_MAINTENANCE_STAGE")
df

,maintenance_keyword,stage_keyword,name,descr,maintenance_stage_timestamp
0,DB_MIG_740_TO_750,START,Migration Started,Migration of BrDR 7.4.0 database started,2025-04-25 11:34:35.230670
1,DB_MIG_740_TO_750,FINISHED,Migration Finished,Migration Finished,2025-04-25 11:36:04.596541
2,DB_MIG_750_TO_760,START,Migration Started,Migration of BrDR 7.5.0 database started,2025-04-25 11:36:04.613388
3,DB_MIG_750_TO_760,FINISHED,Migration Finished,Migration Finished,2025-04-25 11:36:57.083509
4,REMOVE_ORPHANBEAMDEFS_MIG_DB,START,Started,Started,2006-12-11 21:18:49.000000
...,...,...,...,...,...
109,DB_MIG_710_TO_720,FINISHED,Migration Finished,Migration Finished,2022-04-27 08:48:26.846635
110,DB_MIG_720_TO_730,START,Migration Started,Migration of BrDR 7.2.0 database started,2023-07-06 12:40:29.579326
111,DB_MIG_720_TO_730,FINISHED,Migration Finished,Migration Finished,2023-07-06 12:48:28.570358
112,DB_MIG_730_TO_740,START,Migration Started,Migration of BrDR 7.3.0 database started,2023-07-06 12:49:01.021937


#### ABW_LIB_MATL_WEARING_SURFACE

In [129]:
df = get_table("BRIDGEWARE", "ABW_LIB_MATL_WEARING_SURFACE")
df

,lib_matl_wearing_surface_id,name,descr,library_type,si_or_us_type,density
0,1,Asphalt,Asphalt Wearing Surface,22902,10401,2322.716258
1,2,Microsilica,Microsilica Concrete Wearing Surface,22902,10401,2402.809922
2,3,Latex Modified Concrete,Latex Modified Conc Wearing Surface,22902,10401,2402.809922
3,4,Concrete,Concrete Wearing Surface,22902,10401,2402.809922


#### ABW_TIMBER_BEAM_SHAPE

In [130]:
df = get_table("BRIDGEWARE", "ABW_TIMBER_BEAM_SHAPE")
df

,bridge_id,timber_beam_shape_id,name,descr,si_or_us_type,timber_beam_shape_type,last_change_timestamp
0,20225,2,"Timber Panel 8.5""x 40.5"" Beams",None,10401,35901,2025-05-01 14:46:13.633
1,806,1,4 x 12 Beam,None,10401,35901,2010-11-16 20:19:25.000
2,25,1,6 x 18,None,10401,35901,2003-01-01 12:00:00.000
3,363,1,GL1S0101,Non-Detailed Description.,10401,35901,2008-11-26 21:28:51.000
4,1049,1,"Deck Plank (4""x12"")","Plank = 3.5"" x 12"" Rough",10401,35901,2011-09-01 12:06:56.000
5,1027,1,8.5 x 41.75,Deck Panel (GluLam flipped sideways),10401,35901,2011-08-11 13:07:43.000
6,1154,1,Deck Laminates Under Wheel,Actual Dimensions Deck Laminates Under Wheel (...,10401,35901,2011-12-28 13:10:17.000
7,1173,1,"Deck Planks (6"" Wide x 3.75"" Thick)",Actual Dimensions Deck Laminates Under Wheel (...,10401,35901,2012-01-11 13:01:53.000
8,1173,2,"Beams (9"" Wide x 10.5"" Deep)",Actual Dimensions Deck Laminates Under Wheel (...,10401,35901,2012-01-11 16:49:00.000
9,1263,1,"GluLam Net Thickness = 8.75""","One Panel 50.625"" x 8.75""",10401,35901,2012-04-04 14:42:08.000


#### ABW_LOAD_PALETTE_TEMPL_LOADING

In [131]:
df = get_table("BRIDGEWARE", "ABW_LOAD_PALETTE_TEMPL_LOADING")
df

,load_palette_template_id,loading_id,sys_lrfd_loading_id,use_ind,override_ind


#### ABW_FLNG_LAT_BEND_STRESS

In [132]:
df = get_table("BRIDGEWARE", "ABW_FLNG_LAT_BEND_STRESS")
df

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,flng_lat_bend_stress_id,load_case_name,load_case_description,stage_id,load_type,incl_analysis_line_girder_ind,incl_analysis_three_d_ind,consider_design_review_ind,consider_lrfr_rating_ind
0,16479,1,1,1,1,Overhang bracket dead load,None,6,29201,T,T,T,F
1,16479,1,1,1,2,Overhang bracket construction load,None,6,29239,T,T,T,F
2,16479,1,1,1,3,Skew effect,None,7,29240,T,F,T,T
3,16479,1,2,2,4,Skew effect,None,7,29240,T,F,T,T
4,16479,1,3,2,2,Skew effect,None,7,29240,T,F,T,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3972,20378,1,2,2,3,Skew effect,None,7,29240,T,F,T,T
3973,20379,1,2,1,1,Skew effect,None,7,29240,F,F,T,T
3974,20379,1,4,1,1,Skew effect,None,7,29240,F,F,T,T
3975,20379,1,5,1,1,Skew effect,None,7,29240,F,F,T,T


#### ABW_SYS_BRM_SYNC_SETTING

In [133]:
df = get_table("BRIDGEWARE", "ABW_SYS_BRM_SYNC_SETTING")
df

,brm_sync_setting_id,brdr_bridge_column,brdr_bridge_column_name,sync_enabled_ind,override_enabled_enabled_ind,override_enabled_ind,brm_data_table,brm_data_column,brm_default_data_table,brm_default_data_column
0,1,agency_code,Bridge ID,T,F,F,None,None,bridge,bridge_id
1,2,struct_num,NBI Structure ID,T,F,F,None,None,bridge,struct_num
2,3,name,Name,T,T,F,None,None,bridge,strucname
3,4,descr,Description,F,T,F,None,None,bridge,notes
4,5,location,Location,T,F,F,None,None,bridge,location
5,6,facility,Facility carried,T,F,F,None,None,bridge,facility
6,7,featint,Feat. intersected,T,F,F,None,None,bridge,featint
7,8,yearbuilt,Year built,T,F,F,None,None,bridge,yearbuilt
8,9,length,Length,T,F,F,None,None,bridge,length
9,10,routenum,Route number,T,F,F,None,None,roadway,routenum


#### ABW_DETAILED_TRUSS_DEF

In [134]:
df = get_table("BRIDGEWARE", "ABW_DETAILED_TRUSS_DEF")
df

,bridge_id,struct_def_id,spng_mbr_def_id,detailed_truss_def_type,even_num_panels_ind,symmetrical_ind,asr_inv_axialtens_gross_factor,asr_opr_axialtens_gross_factor,asr_inv_axialtens_net_factor,asr_opr_axialtens_net_factor,...,lfr_spec_edition_choice_type,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,lrfr_field_meas_sect_prop_ind,lrfr_system_fact_override_ind,lrfr_system_fact_override
0,14631,2,2,24123,F,F,None,None,None,None,...,50101,50101,50101,10870BD9-A122-4B1C-853D-163B591C4505,None,70A01C5A-21D9-4C12-9A23-1FD3431D583B,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E,None,F,None
1,14631,2,8,24123,F,F,None,None,None,None,...,50101,50101,50101,10870BD9-A122-4B1C-853D-163B591C4505,None,70A01C5A-21D9-4C12-9A23-1FD3431D583B,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E,None,F,None
2,14631,2,9,24123,F,F,None,None,None,None,...,50101,50101,50101,10870BD9-A122-4B1C-853D-163B591C4505,None,70A01C5A-21D9-4C12-9A23-1FD3431D583B,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E,None,F,None
3,14631,2,10,24123,F,F,None,None,None,None,...,50101,50101,50101,10870BD9-A122-4B1C-853D-163B591C4505,None,70A01C5A-21D9-4C12-9A23-1FD3431D583B,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E,None,F,None
4,14631,2,11,24123,F,F,None,None,None,None,...,50101,50101,50101,10870BD9-A122-4B1C-853D-163B591C4505,None,70A01C5A-21D9-4C12-9A23-1FD3431D583B,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E,None,F,None
5,14631,2,12,24123,F,F,None,None,None,None,...,50101,50101,50101,10870BD9-A122-4B1C-853D-163B591C4505,None,70A01C5A-21D9-4C12-9A23-1FD3431D583B,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E,None,F,None
6,14631,2,13,24123,F,F,None,None,None,None,...,50101,50101,50101,10870BD9-A122-4B1C-853D-163B591C4505,None,70A01C5A-21D9-4C12-9A23-1FD3431D583B,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E,None,F,None
7,14631,2,14,24123,F,F,None,None,None,None,...,50101,50101,50101,10870BD9-A122-4B1C-853D-163B591C4505,None,70A01C5A-21D9-4C12-9A23-1FD3431D583B,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E,None,F,None
8,14631,2,15,24123,F,F,None,None,None,None,...,50101,50101,50101,10870BD9-A122-4B1C-853D-163B591C4505,None,70A01C5A-21D9-4C12-9A23-1FD3431D583B,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E,None,F,None
9,14631,2,16,24123,F,F,None,None,None,None,...,50101,50101,50101,10870BD9-A122-4B1C-853D-163B591C4505,None,70A01C5A-21D9-4C12-9A23-1FD3431D583B,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E,None,F,None


#### ABW_SYS_METAL_BOX_COLUMN_LABEL

In [135]:
df = get_table("BRIDGEWARE", "ABW_SYS_METAL_BOX_COLUMN_LABEL")
df

,label_id,label_value
0,1,2.8194
1,2,3.5560
2,3,4.3180
3,4,4.7752
4,5,5.5372
5,6,6.3246
6,7,7.1120
7,8,8.0772
8,9,9.5758
9,10,3.1750


#### ABW_EVENT_VEHICLE

In [136]:
df = get_table("BRIDGEWARE", "ABW_EVENT_VEHICLE")
df

,event_id,event_vehicle_id,vehicle_id,inventory_ind,operating_ind,design_ind,permit_ind,fatigue_ind,scale_factor,single_lane_ind,...,lrfr_operating_ind,legal_pair_ind,lrfr_fatigue_ind,legal_live_load_override_ind,legal_ll_override_factor,lrfr_legal_haul_ind,asr_lfr_permit_inventory_ind,asr_lfr_permit_operating_ind,asr_lfr_legal_inventory_ind,asr_lfr_legal_operating_ind
0,65300,351,12,T,T,T,F,F,0.0,F,...,F,F,F,None,NaN,F,None,None,None,None
1,65343,352,3,T,T,F,F,F,1.0,F,...,F,F,F,None,NaN,F,None,None,None,None
2,65343,353,10,T,T,F,F,F,1.0,F,...,F,F,F,None,NaN,F,None,None,None,None
3,65343,354,15,F,T,F,F,F,1.0,F,...,F,F,F,None,NaN,F,None,None,None,None
4,65343,355,16,F,T,F,F,F,1.0,F,...,F,F,F,None,NaN,F,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1524,1165319,10,8,F,F,F,F,F,1.0,None,...,F,None,F,None,NaN,F,F,F,F,F
1525,1165319,11,1075,F,F,F,F,F,1.0,None,...,F,None,F,None,NaN,T,F,F,F,F
1526,1165319,12,1076,F,F,F,F,F,1.0,None,...,F,None,F,None,NaN,T,F,F,F,F
1527,1165319,13,1077,F,F,F,F,F,1.0,None,...,F,None,F,None,NaN,T,F,F,F,F


#### ABW_TIMBER_RECT_SAWN_BEAM_DEF

In [137]:
df = get_table("BRIDGEWARE", "ABW_TIMBER_RECT_SAWN_BEAM_DEF")
df

,bridge_id,struct_def_id,spng_mbr_def_id,adj_beam_flat_use_factor,adj_beam_rep_use_factor,lrfd_adj_incise_flex_shear,lrfd_adj_incise_bearing,lrfd_adj_incise_modulus
0,20225,1,1,1.0,1.00,1.0,1.0,1.0
1,25,1,1,1.0,1.00,NaN,NaN,NaN
2,363,1,1,NaN,NaN,NaN,NaN,NaN
3,806,7,1,1.0,1.15,NaN,NaN,NaN
4,1049,2,1,1.0,1.15,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
58,5921,1,7,NaN,NaN,NaN,NaN,NaN
59,20016,1,1,1.0,1.00,1.0,1.0,1.0
60,20521,1,1,1.0,1.00,1.0,1.0,1.0
61,20522,1,1,1.0,1.00,1.0,1.0,1.0


#### ABW_PARAMETER

In [138]:
df = get_table("BRIDGEWARE", "ABW_PARAMETER")
df

,parameter_id,table_name,field_name,parmvalue,shortdesc,longdesc
0,160,bridge,adminarea,5,Private,None
1,161,bridge,adminarea,6,Border State,None
2,162,bridge,adminarea,7,Other,None
3,1,bridge,adminarea,-1,Unknown,Unknown
4,2,bridge,adminarea,-2,Not Applicable,Not Applicable
...,...,...,...,...,...,...
157,155,roadway,nhs_ind,0,0 Not on NHS,Inventory route is not on the NHS
158,156,roadway,nhs_ind,1,1 On the NHS,Inventory route is on the NHS
159,157,roadway,nhs_ind,@,Unknown (P),Unknown
160,158,roadway,nhs_ind,Y,On NHS,None


#### ABW_ADV_CONC_XSECTION

In [139]:
df = get_table("BRIDGEWARE", "ABW_ADV_CONC_XSECTION")
df

,bridge_id,struct_def_id,spng_mbr_def_id,xsection_id,xsection_type,name,top_flng_conc_id,top_flng_cj,top_flng_stress_limit_id,other_parts_stress_limit_id,other_parts_conc_id,computed_wt1,computed_wt2,area,ixx,iyy,j
0,18358,1,1,1,61202,Modified Bulb T,2,177.8,1.0,1.0,2,203.2,203.2,794191.96,5.497864e+11,4.191735e+10,1.680884e+10
1,18358,1,3,1,61202,Modified Bulb T,2,177.8,1.0,1.0,2,203.2,203.2,794191.96,5.497864e+11,4.191735e+10,1.680884e+10
2,13878,6,1,1,61201,Max. Depth Girder,1,1524.0,NaN,NaN,1,482.6,482.6,1314190.92,6.120392e+11,4.064133e+10,1.142545e+11
3,13878,6,1,2,61201,Min. Depth Girder,1,1524.0,NaN,NaN,1,482.6,482.6,1314190.92,6.120392e+11,4.064133e+10,1.142545e+11
4,19736,1,1,1,61202,"WF I-Girder - 48"" Deep",3,NaN,1.0,1.0,3,177.8,177.8,504353.83,9.597168e+10,2.707603e+10,1.314643e+10
5,19736,1,2,1,61202,"WF I-Girder - 48"" Deep",3,NaN,1.0,1.0,3,177.8,177.8,504353.83,9.597168e+10,2.707603e+10,1.314643e+10
6,18765,1,1,1,61202,Modified Bulb T,2,177.8,1.0,1.0,2,203.2,203.2,794191.96,5.497864e+11,4.191735e+10,1.680884e+10
7,18765,1,2,1,61202,Modified Bulb T,2,177.8,1.0,1.0,2,203.2,203.2,794191.96,5.497864e+11,4.191735e+10,1.680884e+10
8,18795,3,1,1,61202,Bulb T,2,155.0,1.0,1.0,2,200.0,200.0,560875.00,1.369819e+11,2.588761e+10,1.519024e+10
9,18795,3,2,1,61202,Bulb T,2,155.0,1.0,1.0,2,200.0,200.0,560875.00,1.369819e+11,2.588761e+10,1.519024e+10


#### ABW_BMDEF_DECK_REINF_RANGE

In [140]:
df = get_table("BRIDGEWARE", "ABW_BMDEF_DECK_REINF_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,deck_reinf_range_id,dist,length,stl_reinf_id,num_bars,rebar_id,vert_dist,vert_dist_reference_type
0,20117,1,1,1,0.000000,169.165587,1,14.0,13,69.8500,21801
1,20117,1,1,2,0.000000,169.165587,1,8.0,14,49.2125,21802
2,20117,1,1,3,19.654837,21.031200,1,13.0,13,69.8500,21801
3,20117,1,1,4,18.740438,21.031200,1,13.0,13,69.8500,21801
4,20117,1,1,5,56.235600,21.031200,1,13.0,13,69.8500,21801
...,...,...,...,...,...,...,...,...,...,...,...
207,19659,1,8,2,0.000000,53.492400,1,8.0,13,25.4000,21802
208,19659,1,8,3,12.039600,7.924800,1,3.0,13,127.0000,21801
209,19659,1,8,4,12.954000,7.924800,1,3.0,13,127.0000,21801
210,19659,1,8,5,32.613600,7.924800,1,3.0,13,127.0000,21801


#### ABW_BMDEF_SHEAR_CONN_RANGE

In [141]:
df = get_table("BRIDGEWARE", "ABW_BMDEF_SHEAR_CONN_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,shear_conn_range_id,dist,length,num_spaces,num_across,shear_connector_id,spacing_across,cluster_pitch,number_rows_in_cluster,number_of_clusters,stud_pitch
0,20117,1,1,1,0.000000,20.586700,None,None,None,None,None,None,None,None
1,20117,1,1,2,39.162038,18.091150,None,None,None,None,None,None,None,None
2,20117,1,1,3,75.539600,18.091150,None,None,None,None,None,None,None,None
3,20117,1,1,4,111.906050,18.091150,None,None,None,None,None,None,None,None
4,20117,1,1,5,148.578888,20.586700,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,12858,5,11,1,0.000000,135.940800,None,None,None,None,None,None,None,None
88,13157,1,3,1,0.000000,12.167616,None,None,None,None,None,None,None,None
89,16040,8,2,1,0.000000,20.040600,None,None,None,None,None,None,None,None
90,16742,3,2,1,0.000000,5.334000,None,None,None,None,None,None,None,None


#### ABW_BMDEF_STL_FATIGUE_LOC

In [142]:
df = get_table("BRIDGEWARE", "ABW_BMDEF_STL_FATIGUE_LOC")
df

,bridge_id,struct_def_id,spng_mbr_def_id,anal_pt_id,stl_fatigue_loc_id,vert_dist,lfd_fatigue_category_type,lfd_allow_fatigue_stress,lrfd_fatigue_category_type,lrfd_nom_fatigue_resistance,vert_dist_ref_type


#### ABW_BOLT_DEF

In [143]:
df = get_table("BRIDGEWARE", "ABW_BOLT_DEF")
df

,bridge_id,bolt_def_id,bolt_designation,name,descr,bolt_diameter,hole_diameter,connection_type,hole_size_type,load_direction_type,...,lfd_slip_resistance,lrfd_min_tensile_strength,lrfd_required_tension,lrfd_kh_factor,lrfd_ks_factor,exclude_threads_ind,last_change_timestamp,grip_length,hole_type,lrfd_kc_factor
0,18126,1,AASHTO M 164 (SI),Place Holder,None,25.400,28.5750,30401,30501,30601,...,144.790800,830.000000,NaN,1.00,0.33,F,2017-07-20 15:00:28.000000,NaN,58401.0,1.0
1,18313,1,ASTM F3125 Grade A325,Bolt-Typ,None,0.000,23.8125,30401,30501,30601,...,144.789897,827.370840,84.516218,1.00,0.30,F,2024-07-23 15:08:59.213000,NaN,58402.0,1.0
2,1959,1,AASHTO M 164 (US),Splice Bolts,None,28.575,31.7500,30401,30501,30601,...,126.691160,723.949485,NaN,1.00,0.33,F,2015-11-24 18:16:11.000000,NaN,58401.0,1.0
3,3093,1,AASHTO M 164 (US),"5/8"" Dia. A325 Bolts (US)",5/8' Dia. A325 Galvanized Bolts,15.875,17.4625,30401,30502,30601,...,124.105626,827.370840,NaN,0.85,0.33,T,2015-05-01 15:01:39.000000,NaN,NaN,0.8
4,3192,1,AASHTO M 164 (US),"7/8"" Bolts",Gusset Plate Bolts,22.225,25.4000,30401,30501,30601,...,144.789897,827.370840,NaN,1.00,0.33,F,2015-08-27 14:20:58.000000,NaN,58402.0,0.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,15337,1,ASTM F3125 Grade A325,BOLT CONNECTOR,BOLT FOR SPLICE,25.400,26.9875,30401,30501,30601,...,144.789897,827.370840,226.859322,1.00,0.30,F,2022-10-12 13:34:14.203000,NaN,58401.0,1.0
246,20246,1,ASTM F3125 Grade A325,Splice Bolt,"1 1/8"" Splice Bolt",28.575,30.1625,30401,30501,30601,...,144.789897,827.370840,284.686208,1.00,0.30,F,2019-10-07 20:49:23.000000,NaN,58402.0,1.0
247,19864,1,ASTM F3125 Grade A325,Splice Bolt,"1 1/8"" Splice Bolt",28.575,30.1625,30401,30501,30601,...,144.789897,827.370840,284.686208,1.00,0.30,F,2025-05-20 19:04:14.353426,NaN,58402.0,1.0
248,20555,1,ASTM F3125 Grade A325,"1"" dia. A325 bolt",None,25.400,26.9875,30401,30501,30601,...,144.789897,827.370840,226.859322,1.00,0.30,F,2025-02-03 12:25:05.970000,NaN,58401.0,1.0


#### ABW_BRIDGE_DESIGN_PARAM

In [144]:
df = get_table("BRIDGEWARE", "ABW_BRIDGE_DESIGN_PARAM")
df

,bridge_id,long_force_load_dist_type,trans_force_load_dist_type,consider_induced_vert_reac_ind,horz_roadway_design_speed,horz_roadway_radius_curvature,curve_direction_type,cap_top_reinf_min_rebar_id,cap_top_reinf_max_rebar_id,cap_top_reinf_min_bar_spacing,...,geo_ftg_min_length,geo_ftg_max_length,geo_ftg_length_increment,geo_ftg_min_aspect_ratio,geo_ftg_max_aspect_ratio,geo_ftg_thick_increment,geo_ftg_spread_min_thick,geo_ftg_spread_max_thick,geo_ftg_pile_min_thick,geo_ftg_pile_max_thick
0,3197,41402,41405,T,None,None,40201,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,3198,41402,41405,T,None,None,40201,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,3199,41402,41405,T,None,None,40201,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,3204,41402,41405,T,None,None,40201,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,3212,41402,41405,T,None,None,40201,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13185,20261,41402,41405,T,None,None,40201,None,None,None,...,None,None,None,None,None,None,None,None,None,None
13186,19700,41402,41405,T,None,None,40201,None,None,None,...,None,None,None,None,None,None,None,None,None,None
13187,19701,41402,41405,T,None,None,40201,None,None,None,...,None,None,None,None,None,None,None,None,None,None
13188,19703,41402,41405,T,None,None,40201,None,None,None,...,None,None,None,None,None,None,None,None,None,None


#### ABW_BRIDGE_DIAPHRAGM_DEF

In [145]:
df = get_table("BRIDGEWARE", "ABW_BRIDGE_DIAPHRAGM_DEF")
df

,bridge_id,diaphragm_def_id,name,diaphragm_type,material_type,conc_id,conc_thick,conc_height,num_elements_fixed_members,member_connection_bolt_def_id,tension_only_diag_sys_ind
0,3803,1,Interm Diaph,51801,51901,NaN,NaN,NaN,1,NaN,None
1,3807,1,Interm X-Frames,51801,51901,NaN,NaN,NaN,1,NaN,None
2,3811,1,Interm X-Frame,51801,51901,NaN,NaN,NaN,1,NaN,None
3,3812,1,Interm X-Frame,51801,51901,NaN,NaN,NaN,1,NaN,None
4,3834,1,Crossframe,51801,51901,NaN,NaN,NaN,1,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...
9401,20413,2,Semi-Integral Ends,51804,51902,2.0,1066.8,1676.4,1,NaN,None
9402,20413,4,(New) Intermediate Diaphragm,51801,51901,2.0,NaN,NaN,1,NaN,None
9403,20414,1,End Diaphragm,51804,51902,1.0,635.0,1828.8,1,NaN,None
9404,20414,2,Intermediate Diaphragm,51801,51901,1.0,NaN,NaN,1,NaN,None


#### ABW_TRUSS_DEF_ELEMENT

In [146]:
df = get_table("BRIDGEWARE", "ABW_TRUSS_DEF_ELEMENT")
df

,bridge_id,struct_def_id,spng_mbr_def_id,truss_def_element_id,element_xsection_id,begin_panel_point_id,end_panel_point_id,name,x_axis_unbraced_length,y_axis_unbraced_length,k_value,end_connection_type
0,14631,2,2,1,1,1,2,L0L1,4.953000,4.953000,1.0,48305
1,14631,2,2,2,1,2,3,L1L2,4.953000,4.953000,1.0,48305
2,14631,2,2,3,1,3,4,L2L3,4.953000,4.953000,1.0,48305
3,14631,2,2,4,1,4,5,L3L4,4.953000,4.953000,1.0,48305
4,14631,2,2,5,1,6,7,U0U1,2.476500,2.476500,1.0,48305
...,...,...,...,...,...,...,...,...,...,...,...,...
537,14631,3,18,16,2,9,3,U3L2,2.837136,2.837136,1.0,48305
538,14631,3,18,17,2,3,11,L2U5,2.837136,2.837136,1.0,48305
539,14631,3,18,18,2,11,4,U5L3,2.837136,2.837136,1.0,48305
540,14631,3,18,19,2,4,13,L3U7,2.837136,2.837136,1.0,48305


#### ABW_TRUSS_DEF_PANEL_POINT

In [147]:
df = get_table("BRIDGEWARE", "ABW_TRUSS_DEF_PANEL_POINT")
df

,bridge_id,struct_def_id,spng_mbr_def_id,panel_point_id,name,x_coord,y_coord,location_type
0,14631,3,6,13,U7,17.3355,1.495503,48602
1,14631,3,6,14,U8,19.8120,1.495503,48602
2,14631,3,7,1,L0,0.0000,0.000000,48601
3,14631,3,7,2,L1,4.9530,0.000000,48601
4,14631,3,7,3,L2,9.9060,0.000000,48601
...,...,...,...,...,...,...,...,...
367,14631,3,6,8,U2,4.9530,1.495503,48602
368,14631,3,6,9,U3,7.4295,1.495503,48602
369,14631,3,6,10,U4,9.9060,1.495503,48602
370,14631,3,6,11,U5,12.3825,1.495503,48602


#### ABW_TRUSS_DEF_SPAN

In [148]:
df = get_table("BRIDGEWARE", "ABW_TRUSS_DEF_SPAN")
df

,bridge_id,struct_def_id,spng_mbr_def_id,span_id,length,dist,cantilever_ind
0,14631,2,2,42,4.953,0.000,F
1,14631,2,2,43,4.953,4.953,F
2,14631,2,2,44,4.953,9.906,F
3,14631,2,2,45,4.953,14.859,F
4,14631,2,8,29,4.953,0.000,F
...,...,...,...,...,...,...,...
94,14631,3,18,3,4.953,9.906,F
95,14631,3,18,4,4.953,14.859,F
96,4034,1,5,1,9.144,0.000,F
97,4034,2,5,1,9.144,0.000,F


#### ABW_SUPER_LL_REACT_SUB_DETAIL

In [149]:
df = get_table("BRIDGEWARE", "ABW_SUPER_LL_REACT_SUB_DETAIL")
df

,bridge_id,sub_struct_def_id,ll_reaction_set_id,live_load_id,vehicle_id,vehicle_type,min_single_lane_reaction,min_impact_factor_override,max_single_lane_reaction,max_impact_factor_override
0,5564,3,1,1,2,32802,None,None,312.591268,None
1,5564,3,1,2,2,32806,None,None,402.563473,None
2,5564,3,1,3,2,32801,None,None,263.751329,None
3,5564,3,1,4,2,32804,None,None,222.121061,None
4,5564,5,1,6,2,32806,None,None,307.990751,None
5,5564,5,1,7,2,32801,None,None,205.209151,None
6,5564,5,1,8,2,32804,None,None,226.824926,None
7,5564,6,1,10,2,32806,None,None,284.029440,None
8,5564,6,1,11,2,32801,None,None,161.016347,None
9,5564,6,1,12,2,32804,None,None,222.692498,None


#### ABW_SUPER_LOAD_CASE

In [150]:
df = get_table("BRIDGEWARE", "ABW_SUPER_LOAD_CASE")
df

,bridge_id,struct_def_id,load_case_id,name,stage_id,descr,load_type,load_application_time
0,16787,1,3,DW,2,None,29201,NaN
1,416,1,1,SDL (RC),1,Superimposed Dead Loads (Reinforced Concrete),29201,NaN
2,425,1,1,SDL (SS),1,Superimposed Dead Loads (Structural Steel),29201,NaN
3,20442,1,1,DC1,1,DC acting on non-composite section,29201,NaN
4,20442,1,2,DC2,2,DC acting on long-term composite section,29201,NaN
...,...,...,...,...,...,...,...,...
48369,12801,1,2,DC2,2,DC acting on long-term composite section,29201,NaN
48370,12801,1,3,DW,2,DW acting on long-term composite section,29202,NaN
48371,12801,1,4,SIP Forms,1,Weight due to stay-in-place forms,29201,NaN
48372,10494,3,1,DC1,1,DC acting on non-composite section,29201,NaN


#### ABW_SUPER_MBR_ENVIRONMENT_LOAD

In [151]:
df = get_table("BRIDGEWARE", "ABW_SUPER_MBR_ENVIRONMENT_LOAD")
df

,bridge_id,sub_struct_def_id,mbr_environmental_load_id,spng_mbr_bearing_loc_id,wso_vert_reaction,tu_long_force_temp_rise,tu_long_force_temp_fall,sh_long_force,cr_long_force,wso_vert_reaction_ul,tu_long_force_temp_rise_ul,tu_long_force_temp_fall_ul,sh_long_force_ul,cr_long_force_ul,wso_str_iii_vert_reaction,wso_serv_iv_vert_reaction,wso_str_iii_vert_reaction_ul,wso_serv_iv_vert_reaction_ul
0,5564,3,1,1,None,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None
1,5564,3,2,2,None,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None
2,5564,3,3,3,None,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None
3,5564,3,4,4,None,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None
4,5564,3,5,5,None,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None
5,5564,4,1,1,None,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None
6,5564,4,2,2,None,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None
7,5564,4,3,3,None,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None
8,5564,4,4,4,None,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None
9,5564,4,5,5,None,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None


#### ABW_SUPER_PED_LL_SUB_DETAIL

In [152]:
df = get_table("BRIDGEWARE", "ABW_SUPER_PED_LL_SUB_DETAIL")
df

,bridge_id,sub_struct_def_id,ped_ll_set_id,ped_ll_react_detail_id,super_struct_def_id,curb_id,min_calc_reaction,min_calc_uniform_load,min_ovr_uniform_load,max_calc_reaction,max_calc_uniform_load,max_ovr_uniform_load
0,13291,4,1,1,2,1,None,None,None,None,None,0.0


#### ABW_BOT_FLANGE_LAT_BRACING_DEF

In [153]:
df = get_table("BRIDGEWARE", "ABW_BOT_FLANGE_LAT_BRACING_DEF")
df

,bridge_id,lateral_bracing_def_id,name,stl_shape_type,stl_shape_id,stl_structural_id,connection_type,angle_vert_leg_type,member_connection_type,member_connection_bolt_def_id,num_bolt_lines,num_bolts_per_line,bolt_line_trans_spacing,bolt_line_long_spacing,start_work_point_offset,end_work_point_offset,girder_attach_area_deduction
0,17143,1,Lateral Bracing,57402,3,1,58001,58101,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,18281,1,Lateral Bracing,57402,2,1,58002,58101,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,18370,1,Lateral,57404,4,1,58001,58101,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,18674,1,Lateral Frame,57404,2,1,58001,58101,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20326,1,Lateral Bracing,57404,3,1,58002,58101,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,19661,1,Lat Bracing,57404,5,1,58001,58101,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
118,19668,1,Low Lateral Bracing,57402,1,1,58001,58101,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
119,17747,1,Lateral,57402,3,1,58001,58101,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120,17758,1,Lateral Bracing,57404,3,1,58001,58101,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### ABW_ANAL_PT_REINF_CONC

In [154]:
df = get_table("BRIDGEWARE", "ABW_ANAL_PT_REINF_CONC")
df

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,anal_pt_id,lrfd_shear_comp_method_type,override_shear_reinf_ind,percent_shear,shear_dist,shear_beta,...,horz_shear_rebar_id,horz_shear_num_legs,horz_shear_reinf_area,horz_shear_angle_alpha,horz_shear_reinf_spacing,lfr_ignore_shear_ind,override_reinf_develop_ind,lrfr_ignore_shear_dsnleg_ind,lrfr_ignore_shear_permit_ind,lrfr_cons_prm_tens_stlstrs_ind
0,509,1,1,1,1,34403.0,T,100.0,None,None,...,None,None,None,None,None,T,None,None,None,None
1,509,1,1,1,2,34403.0,T,100.0,None,None,...,None,None,None,None,None,T,None,None,None,None
2,509,1,1,1,3,34403.0,T,100.0,None,None,...,None,None,None,None,None,T,None,None,None,None
3,509,1,1,1,4,34403.0,T,100.0,None,None,...,None,None,None,None,None,T,None,None,None,None
4,509,1,1,1,5,34403.0,T,100.0,None,None,...,None,None,None,None,None,T,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2902,16058,1,1,1,58,34403.0,None,100.0,None,None,...,None,None,None,None,None,None,None,T,T,None
2903,16058,1,1,1,59,34403.0,None,100.0,None,None,...,None,None,None,None,None,None,None,T,T,None
2904,16058,1,1,1,60,34403.0,None,100.0,None,None,...,None,None,None,None,None,None,None,T,T,None
2905,16058,1,1,1,61,34403.0,None,100.0,None,None,...,None,None,None,None,None,None,None,T,T,None


#### ABW_BEAM_DEF

In [155]:
df = get_table("BRIDGEWARE", "ABW_BEAM_DEF")
df

,bridge_id,struct_def_id,spng_mbr_def_id,beam_def_type,xsection_based_ind,beam_use_type,beam_def_units_type,flrbm_cantilever_ind,flrbm_left_cantilever_length,flrbm_right_cantilever_length,...,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,lrfr_field_meas_sect_prop_ind,lrfr_system_fact_override_ind,lrfr_system_fact_override,self_load_case_id,self_loadcase_engineassign_ind,default_top_flng_matl_conc_id,shear_conn_input_method_type
0,12392,1,1,24112,F,37901,NaN,None,None,None,...,None,None,None,None,None,NaN,NaN,None,NaN,64501
1,11860,1,1,24112,F,37901,NaN,None,None,None,...,None,None,None,F,F,0.0,NaN,T,NaN,64501
2,13966,1,4,24112,F,37901,10401.0,None,None,None,...,10870BD9-A122-4B1C-853D-163B591C4505,70A01C5A-21D9-4C12-9A23-1FD3431D583B,631DD3F1-11EF-4ABE-93A2-45E9EA9E601E,F,F,NaN,NaN,T,NaN,64501
3,11473,1,1,24107,T,37901,NaN,None,None,None,...,None,None,None,None,None,NaN,NaN,None,NaN,64501
4,6344,1,1,24116,F,37907,NaN,None,None,None,...,None,None,None,F,F,0.0,NaN,None,NaN,64501
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49037,20556,12,2,24106,F,37901,NaN,None,None,None,...,None,None,None,F,F,NaN,NaN,T,NaN,64501
49038,20556,12,3,24106,F,37901,NaN,None,None,None,...,None,None,None,F,F,NaN,NaN,T,NaN,64501
49039,20556,12,4,24106,F,37901,NaN,None,None,None,...,None,None,None,F,F,NaN,NaN,T,NaN,64501
49040,20556,13,1,24106,F,37901,NaN,None,None,None,...,None,None,None,F,F,NaN,NaN,T,NaN,64501


#### ABW_TEMP_LOAD

In [156]:
df = get_table("BRIDGEWARE", "ABW_MULTIMEDIA")
df

,bridge_id,multimedia_id,context,fileloc,fileref,filetype,agency_type_param_id,status,reportflag,userkey1,userkey2,userkey3,userkey4,userkey5,notes,creation_event_id,last_modified_event_id
0,20530,2,BrDR,R:\BridgeManagement\LRDMS\Data Files\0804002\P...,P3290304.JPG,JPG File,137,None,N,None,None,None,None,None,None,1205562.0,1205563.0
1,16912,1,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,BrR Library Ohio 2022-10-19.brlx,BRLX File,137,None,N,None,None,None,None,None,None,NaN,NaN
2,16912,2,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,2022 BrR Rating Templates 2022-10.brsx,BRSX File,137,None,N,None,None,None,None,None,None,NaN,NaN
3,16912,3,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,BrDR System Data 2022-9-27.brsx,BRSX File,137,None,N,None,None,None,None,None,None,NaN,NaN
4,20530,1,BrDR,R:\BridgeManagement\LRDMS\Data Files\0804002\P...,P3290238.JPG,JPG File,137,None,N,None,None,None,None,None,None,1205560.0,1205561.0
5,85,1,Virtis/Opis,X:\POR-43-1375\,DSC_1690.JPG,JPG,137,None,N,None,None,None,None,None,None,NaN,NaN
6,16731,1,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,BrR Library Ohio 2022-10-19.brlx,BRLX File,137,None,N,None,None,None,None,None,None,NaN,NaN
7,16731,2,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,2022 BrR Rating Templates 2022-10.brsx,BRSX File,137,None,N,None,None,None,None,None,None,NaN,NaN
8,16731,3,BrDR,C:\Users\rcline\OneDrive - Thrasher Engineerin...,BrDR System Data 2022-9-27.brsx,BRSX File,137,None,N,None,None,None,None,None,None,NaN,NaN


#### ABW_TENDON_PROFILE_ANCH_REINF

In [157]:
df = get_table("BRIDGEWARE", "ABW_TENDON_PROFILE_ANCH_REINF")
df

,bridge_id,struct_def_id,spng_mbr_def_id,tendon_profile_id,anchorage_reinf_id,num_legs,rebar_id,start_distance,num_spaces,spacing


#### ABW_TENDON_PROFILE_DEF

In [158]:
df = get_table("BRIDGEWARE", "ABW_TENDON_PROFILE_DEF")
df

,bridge_id,struct_def_id,spng_mbr_def_id,tendon_profile_id,name,start_dist_into_start,end_dist_from_end,inflection_point_method_type,profile_assigned_to_type,pt_ps_matl_ps_strand_id,...,anch_spalling_reinf_num_bars,anch_spalling_rebar_id,starting_span,ending_span,sl_std_prior_to_seating,sl_std_anchor_and_couplers,sl_std_elsewhere_length,sl_std_service_limit,distribute_jforce_equal_ind,stage_id
0,18358,1,1,1,T1,0.0000,0.0000,56902,57001,1,...,None,None,1,1,1507.883356,1303.109073,1390.603539,1340.340761,T,1
1,18358,1,1,3,T2,0.0000,0.0000,56902,57001,1,...,None,None,1,1,1507.883356,1303.109073,1390.603539,1340.340761,T,1
2,18358,1,1,4,T3,0.0000,0.0000,56902,57001,1,...,None,None,1,1,1507.883356,1303.109073,1390.603539,1340.340761,T,2
3,18358,1,1,9,Pre - Bottom,0.0000,0.0000,56901,57001,1,...,None,None,1,1,1507.883356,1303.109073,1390.603539,1340.340761,T,1
4,18358,1,3,1,T1,0.0000,0.0000,56902,57001,1,...,None,None,1,1,1507.883356,1303.109073,1390.603539,1340.340761,T,1
5,18358,1,3,2,T2,0.0000,0.0000,56902,57001,1,...,None,None,1,1,1507.883356,1303.109073,1390.603539,1340.340761,T,1
6,18358,1,3,3,T3,0.0000,0.0000,56902,57001,1,...,None,None,1,1,1507.883356,1303.109073,1390.603539,1340.340761,T,2
7,18358,1,3,7,Pre - Bottom,0.0000,0.0000,56901,57001,1,...,None,None,1,1,1507.883356,1303.109073,1390.603539,1340.340761,T,1
8,19736,1,1,1,T1,0.0000,0.0000,56901,57001,1,...,None,None,1,5,1507.883356,1303.109073,1390.603539,1340.340761,T,2
9,19736,1,1,2,T2,0.0000,0.0000,56901,57001,1,...,None,None,1,5,1507.883356,1303.109073,1390.603539,1340.340761,T,1


#### ABW_TENDON_PROFILE_DEF_DETAIL

In [159]:
df = get_table("BRIDGEWARE", "ABW_TENDON_PROFILE_DEF_DETAIL")
df

,bridge_id,struct_def_id,spng_mbr_def_id,tendon_profile_id,detail_id,profile_type,inflect_point_left_percent,inflect_point_low_percent,inflect_point_right_percent,inflect_point_left_dist,...,vert_offset_left_end_dist,vert_offset_le_meas_from_type,vert_offset_left_dist,vert_offset_l_meas_from_type,vert_offset_low_dist,vert_offset_lo_meas_from_type,vert_offset_right_dist,vert_offset_r_meas_from_type,vert_offset_right_end_dist,vert_offset_re_meas_from_type
0,18358,1,1,1,2,57102,NaN,NaN,NaN,None,...,762.00000,57202,None,57201.0,88.90000,57202,None,57201,762.00000,57202
1,18358,1,1,3,2,57102,NaN,NaN,NaN,None,...,1143.00000,57202,None,57201.0,241.30000,57202,None,57201,1143.00000,57202
2,18358,1,1,4,2,57102,NaN,NaN,NaN,None,...,1524.00000,57202,None,57201.0,393.70000,57202,None,57201,1524.00000,57202
3,18358,1,1,9,1,57101,0.0,50.0,0.0,None,...,127.00000,57202,None,57202.0,127.00000,57202,None,57201,127.00000,57202
4,18358,1,3,1,1,57102,NaN,NaN,NaN,None,...,762.00000,57202,None,57201.0,88.90000,57202,None,57201,762.00000,57202
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,18796,3,1,6,1,57101,0.0,50.0,0.0,None,...,131.00000,57202,None,57202.0,131.00000,57202,None,57201,131.00000,57202
68,18796,3,2,1,1,57102,NaN,NaN,NaN,None,...,1000.00054,57201,None,57201.0,1250.00004,57201,None,57201,1000.00054,57201
69,18796,3,2,2,1,57102,NaN,NaN,NaN,None,...,650.00124,57201,None,57201.0,1100.00009,57201,None,57201,650.00124,57201
70,18796,3,2,3,1,57102,NaN,NaN,NaN,None,...,299.99940,57201,None,57201.0,970.00060,57201,None,57201,299.99940,57201


#### ABW_TIMBER_BEAM_DEF

In [160]:
df = get_table("BRIDGEWARE", "ABW_TIMBER_BEAM_DEF")
df

,bridge_id,struct_def_id,spng_mbr_def_id,timber_beam_def_type,overhang_left,overhang_right,adj_beam_shear_factor,adj_beam_flex_wet_service,adj_beam_shear_wet_service,adj_beam_compr_wet_service,...,lrfd_adj_size_modulus,lrfd_adj_flat_use,lrfd_adj_bearing,lrfd_adj_deck,lrfd_adj_time_effect_str_i,lrfd_adj_time_effect_str_ii,timber_beam_shape_id,timber_id,asd_adj_beam_stability,lrfd_adj_beam_stability
0,20225,1,1,24118,158.75,158.75,1.0,1.00,1.00,1.00,...,1.0,1.1,None,None,0.8,1.0,2,1,None,None
1,25,1,1,24118,0.00,0.00,1.1,0.96,1.00,0.67,...,NaN,NaN,None,None,NaN,NaN,1,1,None,None
2,363,1,1,24118,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,None,None,NaN,NaN,1,1,None,None
3,806,7,1,24118,76.20,76.20,1.0,0.85,0.97,0.67,...,NaN,NaN,None,None,NaN,NaN,1,2,None,None
4,1049,2,1,24118,152.40,152.40,1.0,0.85,0.97,0.67,...,NaN,NaN,None,None,NaN,NaN,1,1,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,5921,1,7,24118,152.40,152.40,NaN,NaN,NaN,NaN,...,NaN,NaN,None,None,NaN,NaN,1,1,None,None
59,20016,1,1,24118,120.65,120.65,1.0,1.00,1.00,1.00,...,1.0,1.1,None,None,0.8,1.0,2,1,None,None
60,20521,1,1,24118,127.00,127.00,1.0,1.00,1.00,1.00,...,1.0,1.1,None,None,0.8,1.0,2,1,None,None
61,20522,1,1,24118,127.00,127.00,1.0,1.00,1.00,1.00,...,1.0,1.1,None,None,0.8,1.0,2,1,None,None


#### ABW_LIB_MATL_STL_STRUCTURAL

In [161]:
df = get_table("BRIDGEWARE", "ABW_LIB_MATL_STL_STRUCTURAL")
df

,stl_structural_id,name,descr,library_type,si_or_us_type,yield_strength,tens_strength,toughness,thermal_exp_coeff,density,mod_of_elast,checksum
0,63,Grade 70W - Fu = 85 ksi,AASHTO M270 Grade 70W - Fu = 85 ksi,22901,10401,482.633060,586.054345,None,0.000012,7849.179078,199947.982,1.0
1,1,Grade 250,AASHTO M270M Grade 250,22901,10402,250.000000,400.000000,None,0.000012,7849.000000,199948.000,1.0
2,2,Grade 345,AASHTO M270M Grade 345,22901,10402,345.000000,450.000000,None,0.000012,7849.000000,199948.000,1.0
3,3,Grade 345W,AASHTO M270M Grade 345W,22901,10402,345.000000,485.000000,None,0.000012,7849.000000,199948.000,1.0
4,4,Grade 485W,AASHTO M270M Grade 485W,22901,10402,485.000000,620.000000,None,0.000012,7849.000000,199948.000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
58,58,ASTM A373 32 ksi,32 ksi circa 1950 +,22902,10401,220.632224,399.895906,None,0.000012,7849.179078,199947.982,NaN
59,60,Fy 36000,Based on year of construction,22902,10401,248.211252,NaN,None,0.000012,7849.179078,199947.953,NaN
60,61,Fy 33000,Based on year of construction,22902,10401,227.526981,NaN,None,0.000012,7849.179078,199947.953,NaN
61,59,ASTM A709,ASTM A 709 50 ksi,22902,10401,344.737850,448.159205,None,0.000012,7849.179078,199947.982,1.0


#### ABW_CULVERT_STRUCT_FK

In [162]:
df = get_table("BRIDGEWARE", "ABW_CULVERT_STRUCT_FK")
df

,bridge_id,bridge_design_alt_id,culvert_id,current_culvert_struct_alt_id,as_built_culvert_struct_alt_id
0,1730,1,1,1.0,1.0
1,1724,1,1,1.0,1.0
2,2012,1,1,1.0,1.0
3,2141,1,1,1.0,1.0
4,2573,1,1,1.0,1.0
...,...,...,...,...,...
1220,20436,1,1,1.0,1.0
1221,20440,1,1,1.0,1.0
1222,20441,1,1,1.0,1.0
1223,20516,1,1,1.0,1.0


#### ABW_GENERAL_PREFERENCE

In [163]:
df = get_table("BRIDGEWARE", "ABW_GENERAL_PREFERENCE")
df

,general_preference_id,name,keyword,preference_cat_id,preference_xml
0,67,Shear Computation Method,PREF_0067,22,"<Controls><Control controltype=""50401"" keyword..."
1,68,Loss and Stress Calculations,PREF_0068,22,"<Controls><Control controltype=""50401"" keyword..."
2,69,Multi-span analysis,PREF_0069,22,"<Controls><Control controltype=""50401"" keyword..."
3,70,Ignore design and legal load shear,PREF_0070,22,"<Controls><Control controltype=""50403"" keyword..."
4,71,Ignore permit load shear,PREF_0071,22,"<Controls><Control controltype=""50403"" keyword..."
...,...,...,...,...,...
526,405,Simplified Wind Loading,PREF_0405,72,"<Controls><Control controltype=""50402"" keyword..."
527,17,Steel Girder,PREF_0017,5,"<Controls><Control controltype=""50404"" keyword..."
528,19,Steel Stringer,PREF_0019,5,"<Controls><Control controltype=""50404"" keyword..."
529,20,PS,PREF_0020,6,"<Controls><Control controltype=""50404"" keyword..."


#### ABW_GENERAL_PREFERENCE_CAT

In [164]:
df = get_table("BRIDGEWARE", "ABW_GENERAL_PREFERENCE_CAT")
df

,preference_cat_id,parent_preference_cat_id,name
0,1,NaN,Bridge
1,2,NaN,Superstructure
2,3,NaN,Member
3,4,NaN,Substructure
4,5,3.0,Analysis Module - ASR
...,...,...,...
91,88,86.0,LRFR
92,89,12.0,LRFR
93,90,NaN,Deck
94,91,90.0,Analysis Module - ASR


#### ABW_RC_CULVERT_DEF

In [165]:
df = get_table("BRIDGEWARE", "ABW_RC_CULVERT_DEF")
df

,bridge_id,struct_def_id,culvert_def_id,rc_culvert_def_type
0,1724,1,1,50603
1,1730,1,1,50603
2,2012,1,1,50603
3,2141,1,1,50603
4,2573,1,1,50603
...,...,...,...,...
1243,20256,1,1,50603
1244,20436,1,1,50603
1245,20440,2,1,50603
1246,20516,1,1,50603


#### ABW_TRUSS_DEF_SUPPORT

In [166]:
df = get_table("BRIDGEWARE", "ABW_TRUSS_DEF_SUPPORT")
df

,bridge_id,struct_def_id,spng_mbr_def_id,support_id,panel_point_id,x_translation_type,y_translation_type,z_rotation_type
0,14631,2,2,1,6,25502,25502,25501
1,14631,2,2,2,8,25501,25502,25501
2,14631,2,2,3,10,25501,25502,25501
3,14631,2,2,4,12,25501,25502,25501
4,14631,2,2,5,14,25502,25502,25501
...,...,...,...,...,...,...,...,...
247,4034,2,5,4,6,25501,25502,25501
248,4034,2,6,1,7,25502,25502,25501
249,4034,2,6,2,12,25501,25502,25501
250,4034,2,6,3,1,25501,25502,25501


#### ABW_SUPER_STRUCT_DEF_FK

In [167]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_DEF_FK")
df

,bridge_id,struct_def_id,current_load_scenario_id,deck_load_case_id,deck_loadcase_engineassign_ind
0,355,1,1,NaN,T
1,356,1,1,NaN,T
2,357,1,1,NaN,T
3,395,1,1,NaN,T
4,17281,1,1,NaN,T
...,...,...,...,...,...
14826,20536,2,1,NaN,T
14827,20537,2,1,NaN,T
14828,20538,2,1,NaN,T
14829,20540,1,1,NaN,T


#### ABW_SUPER_STRUCT_FK

In [168]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_FK")
df

,bridge_id,bridge_design_alt_id,super_struct_id,current_super_struct_alt_id,as_built_super_struct_alt_id
0,355,1,1,1.0,1.0
1,356,1,1,1.0,1.0
2,357,1,1,1.0,1.0
3,16616,1,1,1.0,1.0
4,19169,1,1,1.0,1.0
...,...,...,...,...,...
13548,20091,1,2,1.0,1.0
13549,20417,1,1,1.0,1.0
13550,20419,1,1,1.0,1.0
13551,20420,1,1,1.0,1.0


#### ABW_SUPER_STRUCT_LL_DIST_SUB

In [169]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_LL_DIST_SUB")
df

,bridge_id,sub_struct_def_id,ll_dist_set_id,span_location_type,distribution_method_type,load_display_type,use_override_values_ind
0,5564,3,1,42403,42201,42101,F
1,5564,4,1,42403,42201,42101,F
2,5564,5,1,42403,42201,42101,F
3,5564,6,1,42403,42201,42101,F
4,13291,4,1,42403,42201,42101,F


#### ABW_SUPER_STRUCT_SPNG_MBR_FK

In [170]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_SPNG_MBR_FK")
df

,bridge_id,struct_def_id,super_struct_mbr_id,current_mbr_alt_id,as_built_mbr_alt_id,same_as_mbr_id
0,355,1,1,1.0,1.0,NaN
1,356,1,1,1.0,1.0,NaN
2,357,1,1,1.0,1.0,NaN
3,357,1,2,1.0,1.0,NaN
4,395,1,1,1.0,1.0,NaN
...,...,...,...,...,...,...
85377,16138,2,1,1.0,1.0,NaN
85378,16138,2,2,2.0,2.0,NaN
85379,16138,2,3,NaN,NaN,2.0
85380,16138,2,4,NaN,NaN,2.0


#### ABW_SUBSDEF_MODEL_SET_FIXITY

In [171]:
df = get_table("BRIDGEWARE", "ABW_SUBSDEF_MODEL_SET_FIXITY")
df

,bridge_id,struct_def_id,model_setting_id,fixity_condition_id,sub_struct_def_component_id,location_type,long_support_type,trans_support_type,long_rot_spring_constant,trans_rot_spring_constant
0,7533,2,1,1,3,44001,44101,44101,None,None
1,7533,2,1,2,4,44001,44101,44101,None,None
2,7533,2,1,3,5,44001,44101,44101,None,None
3,7533,2,1,4,6,44001,44101,44101,None,None
4,7533,2,1,5,7,44001,44101,44101,None,None
5,7533,2,1,6,8,44001,44101,44101,None,None
6,7534,3,1,1,3,44001,44101,44101,None,None
7,7534,3,1,2,4,44001,44101,44101,None,None
8,7534,3,1,3,5,44001,44101,44101,None,None
9,7534,3,1,4,6,44001,44101,44101,None,None


#### ABW_SYS_DESIGN_WATER_LEVEL

In [172]:
df = get_table("BRIDGEWARE", "ABW_SYS_DESIGN_WATER_LEVEL")
df

,sys_design_water_level_id,name,descr
0,1,Low,Low
1,2,Mean,Mean
2,3,Design Flood,Design Flood
3,4,Check Flood,Check Flood


#### ABW_SYS_LRFD_LOADING

In [173]:
df = get_table("BRIDGEWARE", "ABW_SYS_LRFD_LOADING")
df

,sys_lrfd_loading_id,name,descr,sort_order
0,1,DC,Dead load from components and at,1
1,2,DW,Dead load from wearing surface a,2
2,3,LL,Vehicular live load,3
3,4,CE,Centrifugal force,4
4,5,BR,Braking force,5
5,6,PL,Pedestrian live load,6
6,7,LS,Live load surcharge,7
7,8,WA,Water load and stream pressure,8
8,9,WS,Wind load on structure,9
9,10,WL,Wind on live load,10


#### ABW_FLOORSYS_STRUCT_DEF

In [174]:
df = get_table("BRIDGEWARE", "ABW_FLOORSYS_STRUCT_DEF")
df

,bridge_id,struct_def_id,floorsys_struct_def_type,main_member_config_type,deck_type,member_spacing_display_type,main_members_support_deck_ind,stringer_frame_into_flrbm_ind,num_stringer_units,modular_ratio_sustained_factor,...,wearing_surface_desc,wearing_surface_density,wearing_surface_thick,truck_traffic_frac_single_lane,num_lanes_avail_truck,override_truck_traffic_ind,dl_distribution1_type,dl_distribution2_type,deck_exposure_factor,thick_field_mea_ind
0,18126,2,25427,38102,36204,27601,F,T,None,NaN,...,None,NaN,NaN,None,NaN,F,33201,33301,None,F
1,20117,1,25415,38101,36201,27601,F,T,None,3.0,...,None,NaN,NaN,None,4.0,F,33201,33301,None,F
2,20129,1,25415,38101,36201,27601,F,T,None,3.0,...,None,NaN,NaN,None,5.0,F,33201,33301,None,F
3,20416,1,25413,38101,36201,27601,T,T,None,3.0,...,None,NaN,NaN,None,NaN,F,33201,33301,None,F
4,20416,12,25413,38101,36201,27601,T,F,None,3.0,...,None,NaN,NaN,None,4.0,F,33201,33301,None,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,18315,2,25413,38101,36201,27601,T,T,None,3.0,...,Concrete Overlay,2402.809922,78.74,None,NaN,F,33201,33301,None,F
114,19659,1,25413,38101,36201,27601,T,F,None,3.0,...,MSC overlay per 2009 rehab plans,2402.809922,95.25,None,2.0,F,33201,33301,None,F
115,20260,3,25413,38102,36201,27601,F,F,None,3.0,...,None,NaN,NaN,None,3.0,F,33201,33301,None,F
116,20260,6,25413,38102,36201,27601,F,F,None,3.0,...,None,NaN,NaN,None,3.0,F,33201,33301,None,F


#### ABW_LIB_SUB_LRFD_DS_VEH_LOAD

In [175]:
df = get_table("BRIDGEWARE", "ABW_LIB_SUB_LRFD_DS_VEH_LOAD")
df

,lrfd_design_setting_id,design_setting_vehicle_info_id,vehicle_loading_info_id,vehicle_type,axle_load
0,1,1,1,32802,142.343104
1,1,1,2,32804,111.205550
2,1,1,3,32801,0.000000
3,1,1,4,32806,142.343104
4,2,1,1,32802,145.000000
5,2,1,2,32804,110.000000
6,2,1,3,32801,0.000000
7,2,1,4,32806,145.000000
8,3,1,1,32802,142.343104
9,3,1,2,32804,111.205550


#### ABW_LIB_SUB_LRFD_DSGN_SETTING

In [176]:
df = get_table("BRIDGEWARE", "ABW_LIB_SUB_LRFD_DSGN_SETTING")
df

,lrfd_design_setting_id,name,descr,library_type,si_or_us_type,lrfd_factor_id,preliminary_design_setting_ind,final_design_setting_ind,dynamic_load_allow_method_type,dla_simple_fatigue_frac_impact,dla_simple_other_ls_impact,vehicle_summary_display_type,last_change_timestamp,limit_num_dsgn_load_combo_ind,cw_num_loadcombo_axial_bending,ftg_num_loadcombo_brg_capacity,lrfd_analysis_module_guid,lrfd_spec_edition_guid
0,1,Preliminary Design Setting (US),Preliminary Design Setting (US),22901,10401,12,T,F,42301,15.0,33.0,43502,2005-01-01 12:00:00,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,F9E28C1B-6942-488E-AB06-CDFA396512FE
1,2,Preliminary Design Setting (SI),Preliminary Design Setting (SI),22901,10402,12,T,F,42301,15.0,33.0,43502,2005-01-01 12:00:00,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,F9E28C1B-6942-488E-AB06-CDFA396512FE
2,3,Final Design Setting (US),Final Design Setting (US),22901,10401,12,F,T,42301,15.0,33.0,43502,2005-01-01 12:00:00,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,F9E28C1B-6942-488E-AB06-CDFA396512FE
3,4,Final Design Setting (SI),Final Design Setting (SI),22901,10402,12,F,T,42301,15.0,33.0,43502,2005-01-01 12:00:00,None,None,None,553254C5-C4CC-43B8-A6E6-FF99FB6330B3,F9E28C1B-6942-488E-AB06-CDFA396512FE


#### ABW_LIB_TIMBER_RECT_BEAM_SHAPE

In [177]:
df = get_table("BRIDGEWARE", "ABW_LIB_TIMBER_RECT_BEAM_SHAPE")
df

,timber_beam_shape_id,width,height,glulam_ind,num_glulam_laminations,nominal_weight,nominal_height,nominal_width,area,ixx,cg_bot,sxx_top,sxx_bot
0,2,304.8,317.5,None,None,None,None,None,96774.0,8.129520e+08,158.75,5120957.5,5120957.5


#### ABW_LIB_VEHICLE

In [178]:
df = get_table("BRIDGEWARE", "ABW_LIB_VEHICLE")
df

,vehicle_id,name,descr,library_type,si_or_us_type,uniform_lane_load,moment_conc_load,shear_conc_load,constant_gage_dist,constant_contact_width,...,std_design_ind,tandem_load,tandem_spacing,tandem_trans_spacing,notional_ind,last_change_timestamp,vehicle_gage_type,lrfr_rating_ind,private_ind,owner_id
0,17,OH-4F1,4 Axle Legal-27 tons,22902,10401,NaN,NaN,NaN,None,None,...,F,NaN,NaN,NaN,F,2025-01-15 15:46:23.133872,45601,F,None,NaN
1,1458,Permit BPN 6,None,22904,10401,NaN,NaN,NaN,None,None,...,T,NaN,NaN,NaN,F,2025-05-08 17:17:01.248941,45601,T,T,70.0
2,1450,AY Permit Vehicle-NS 3,1554066 1767275,22904,10401,NaN,NaN,NaN,None,None,...,None,NaN,NaN,NaN,None,2024-02-08 17:50:32.763702,45602,T,None,86.0
3,1452,Permit GJA NSG 1,Permit #2362418 (BOX BEAMS INCLUDED),22904,10401,NaN,NaN,NaN,None,None,...,T,NaN,NaN,NaN,F,2025-06-18 12:10:44.727305,45601,T,T,90.0
4,1453,Permit 1560804,"Edwards Moving & Rigging, Inc. - [GJA]",22904,10401,NaN,NaN,NaN,None,None,...,None,NaN,NaN,NaN,None,2024-02-13 19:50:44.896560,45602,T,T,65.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
410,1444,Permit GJA PROMILES TEST,Intel,22904,10401,NaN,NaN,NaN,None,None,...,F,NaN,NaN,NaN,F,2023-12-19 18:59:38.575421,45601,T,T,90.0
411,1446,Permit 1528000,"Edwards Moving & Rigging, Inc. - [GJA]",22904,10401,NaN,NaN,NaN,None,None,...,None,NaN,NaN,NaN,None,2024-01-17 14:37:33.275892,45602,T,T,65.0
412,1448,AY Permit Vehicle 026~1,1536356 DUAL LANE,22904,10401,NaN,NaN,NaN,None,None,...,T,NaN,NaN,NaN,F,2024-01-24 15:57:58.894588,45601,T,None,86.0
413,1449,Permit 1552053,"Edwards Moving & Rigging, Inc. - [GJA]",22904,10401,NaN,NaN,NaN,None,None,...,None,NaN,NaN,NaN,None,2024-02-06 20:21:29.151395,45602,T,T,65.0


#### ABW_LIB_VEHICLE_AXLE

In [179]:
df = get_table("BRIDGEWARE", "ABW_LIB_VEHICLE_AXLE")
df

,vehicle_id,axle_id,axle_weight,min_axle_spacing,max_axle_spacing,gage_dist,contact_width,nsg_dist_cl_first_wheel
0,642,10,NaN,1.524000,1.524000,NaN,NaN,-1.2192
1,642,11,NaN,13.563600,13.563600,NaN,NaN,3.0480
2,642,12,NaN,1.524000,1.524000,NaN,NaN,3.0480
3,642,13,NaN,1.524000,1.524000,NaN,NaN,3.0480
4,642,14,NaN,4.291584,4.291584,NaN,NaN,3.0480
...,...,...,...,...,...,...,...,...
5360,1449,25,NaN,1.498600,NaN,NaN,NaN,-1.2700
5361,1449,26,NaN,1.498600,NaN,NaN,NaN,-1.2700
5362,1449,27,NaN,1.498600,NaN,NaN,NaN,-1.2700
5363,1297,7,80.067996,1.524000,NaN,1.8288,1117.6,NaN


In [180]:
print(df.columns.to_list())
print(df.shape)

['vehicle_id', 'axle_id', 'axle_weight', 'min_axle_spacing', 'max_axle_spacing', 'gage_dist', 'contact_width', 'nsg_dist_cl_first_wheel']
(5365, 8)


#### ABW_BMDEF_HAUNCH_RANGE

In [181]:
df = get_table("BRIDGEWARE", "ABW_BMDEF_HAUNCH_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,haunch_range_id,dist,length,y1,y2,y3,z1,z2,z3,z4
0,20117,1,1,1,0.0,169.165587,136.525,136.525,None,50.8,187.325,50.8,187.325
1,20117,1,7,1,0.0,169.165587,136.525,136.525,None,50.8,187.325,50.8,187.325
2,20117,1,12,1,0.0,169.165587,136.525,136.525,None,50.8,187.325,50.8,187.325
3,20117,1,13,1,0.0,169.165587,136.525,136.525,None,50.8,187.325,50.8,187.325
4,20117,1,14,1,0.0,169.165587,136.525,136.525,None,50.8,187.325,50.8,187.325
...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,18315,3,4,1,0.0,4.724400,38.100,38.100,None,76.2,304.800,76.2,304.800
236,18315,3,5,1,0.0,4.724400,50.800,50.800,None,76.2,304.800,914.4,914.400
237,18315,3,6,1,0.0,4.724400,38.100,38.100,None,76.2,304.800,76.2,304.800
238,19659,1,3,1,0.0,53.492400,50.800,50.800,None,0.0,61.722,0.0,61.722


#### ABW_BRIDGE_DIAPH_DEF_LOC_CON

In [182]:
df = get_table("BRIDGEWARE", "ABW_BRIDGE_DIAPH_DEF_LOC_CON")
df

,bridge_id,diaphragm_def_id,location_id,location_connection_id,connection_id,member_connection_type,num_long_bolt_lines,num_bolts_per_line,bolt_line_trans_spacing,bolt_line_long_spacing,work_point_offset,trans_edge_distance,long_edge_distance
0,3920,1,1,1,1,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3920,1,1,2,2,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3920,1,2,1,3,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3920,1,2,2,4,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3920,1,3,1,1,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
63453,19774,3,10,1,13,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63454,19774,3,10,2,14,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63455,19774,3,11,1,13,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63456,19774,3,11,2,15,59401,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### ABW_ADV_CONC_BDEF_FLEX_REINF

In [183]:
df = get_table("BRIDGEWARE", "ABW_ADV_CONC_BDEF_FLEX_REINF")
df

,bridge_id,struct_def_id,spng_mbr_def_id,reinf_id,reinf_type,stl_reinf_id,ref_point_type,support_midspan_num,direction_type,start_distance,length,num_bars,rebar_id,clear_cover,bar_spacing,side_cover,start_fully_developed_ind,end_fully_developed_ind,head_at_start_ind,head_at_end_ind
0,18358,1,1,1,61701,1,55501,1,55301,0.00000,48.88230,8.0,16,60.325,190.5,76.20,F,F,None,None
1,18358,1,3,1,61701,1,55501,1,55301,0.00000,48.88230,8.0,16,60.325,190.5,76.20,F,F,None,None
2,13878,6,1,1,61702,1,55502,1,55302,-0.69851,16.63699,1.0,20,149.225,711.2,69.85,T,T,None,None
3,13878,6,1,2,61702,1,55502,1,55302,1.60020,12.03960,1.0,20,149.225,711.2,165.10,T,T,None,None
4,13878,6,1,3,61702,1,55502,1,55302,2.43840,10.36320,1.0,19,149.225,711.2,260.35,T,T,None,None
5,13878,6,1,4,61702,1,55502,1,55302,4.34340,6.55320,1.0,19,149.225,711.2,355.60,T,T,None,None
6,13878,6,1,5,61702,1,55502,1,55302,2.43840,10.36320,1.0,20,149.225,711.2,450.85,T,T,None,None
7,13878,6,1,6,61702,1,55502,1,55302,1.60020,12.03960,1.0,20,149.225,711.2,546.10,T,T,None,None
8,13878,6,1,7,61702,1,55502,1,55302,-0.69851,16.63699,1.0,20,149.225,711.2,641.35,T,T,None,None
9,13878,6,1,8,61702,1,55502,1,55302,-0.69851,16.63699,1.0,20,66.675,711.2,69.85,T,T,None,None


#### ABW_BMDEF_DISTRIB_LOAD

In [184]:
df = get_table("BRIDGEWARE", "ABW_BMDEF_DISTRIB_LOAD")
df

,bridge_id,struct_def_id,spng_mbr_def_id,distr_load_id,direction_type,load_case_id,dist,length,load_start,load_end,local_axis_ind
0,20416,12,3,1,22701,5,0.0,23.520380,11.923230,11.923230,None
1,20416,12,11,1,22701,8,0.0,33.531536,0.208693,0.208693,None
2,20416,12,13,1,22701,8,0.0,33.531536,0.208693,0.208693,None
3,20416,12,14,1,22701,9,0.0,33.531536,0.290419,0.290419,None
4,20416,12,16,1,22701,8,0.0,33.529585,0.208693,0.208693,None
...,...,...,...,...,...,...,...,...,...,...,...
97,18315,3,5,1,22701,2,0.0,4.724400,0.306472,0.306472,None
98,18315,3,6,1,22701,2,0.0,4.724400,0.335660,0.335660,None
99,19659,1,8,1,22701,2,0.0,53.492400,0.875635,0.875635,None
100,20260,3,21,1,22701,7,0.0,14.630400,18.942904,18.942904,None


#### ABW_BRIDGE_REF_LINE_HORZ

In [185]:
df = get_table("BRIDGEWARE", "ABW_BRIDGE_REF_LINE_HORZ")
df

,bridge_id,bridge_ref_line_id,horz_seg_id,horz_seg_type,infinite_horz_line_ind,horz_seg_length,radius,delta,direction_type
0,18047,2,9,55002,None,NaN,317.527000,0.095803,31102
1,18368,1,161,55002,None,NaN,167.640000,1.222655,31101
2,18368,1,162,55001,None,62.212728,NaN,0.000000,31103
3,18369,2,285,55001,None,34.600000,NaN,0.000000,31103
4,18369,2,286,55002,None,NaN,140.000000,1.147143,31101
...,...,...,...,...,...,...,...,...,...
239,17745,1,149,55002,None,NaN,74.932032,1.353397,31101
240,17758,1,124,55002,None,NaN,79.382112,1.378436,31102
241,19149,1,161,55002,None,NaN,349.276416,0.772322,31101
242,20446,2,146,55002,None,NaN,174.028100,0.389148,31102


#### ABW_BRIDGE_ENVIRONMENTAL_PARAM

In [186]:
df = get_table("BRIDGEWARE", "ABW_BRIDGE_ENVIRONMENTAL_PARAM")
df

,bridge_id,environmental_param_id,name,descr,default_param_ind,si_or_us_type,base_design_wind_velocity,wind_opencountry_fric_velocity,wind_opencountry_fric_length,wind_suburban_fric_velocity,...,gust_ww_dcoeff_bridge_sub,gust_lw_dcoeff_bridge_sub,gust_ww_dcoeff_sound_barrier,gust_lw_dcoeff_sound_barrier,gust_strength_iii_vertuw_press,gust_service_iv_vertuw_press,gust_smplfd_trns_super_pctcomp,gust_smplfd_long_super_pcttrns,gust_smplfd_trns_ll,gust_smplfd_long_ll
0,667,2,US Parameters,US Environmental Parameters,T,10401,160.9347,13.196645,70.104,17.541882,...,1.6,None,1.2,None,0.000958,0.000479,100.0,25.0,1.459392,0.583757
1,676,1,SI Parameters,SI Environmental Parameters,None,10402,160.0000,13.200000,70.000,17.600000,...,NaN,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN
2,676,2,US Parameters,US Environmental Parameters,T,10401,160.9347,13.196645,70.104,17.541882,...,1.6,None,1.2,None,0.000958,0.000479,100.0,25.0,1.459392,0.583757
3,614,1,SI Parameters,SI Environmental Parameters,None,10402,160.0000,13.200000,70.000,17.600000,...,NaN,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN
4,614,2,US Parameters,US Environmental Parameters,T,10401,160.9347,13.196645,70.104,17.541882,...,1.6,None,1.2,None,0.000958,0.000479,100.0,25.0,1.459392,0.583757
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26371,19742,1,SI Parameters,SI Environmental Parameters,T,10402,160.0000,13.200000,70.000,17.600000,...,NaN,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN
26372,19742,2,US Parameters,US Environmental Parameters,F,10401,160.9347,13.196645,70.104,17.541882,...,1.6,None,1.2,None,0.000958,0.000479,100.0,25.0,1.459392,0.583757
26373,19743,1,SI Parameters,SI Environmental Parameters,None,10402,160.0000,13.200000,70.000,17.600000,...,NaN,None,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN
26374,19743,2,US Parameters,US Environmental Parameters,T,10401,160.9347,13.196645,70.104,17.541882,...,1.6,None,1.2,None,0.000958,0.000479,100.0,25.0,1.459392,0.583757


#### ABW_LRFD_LS

In [187]:
df = get_table("BRIDGEWARE", "ABW_LRFD_LS")
df

,bridge_id,lrfd_factor_id,ls_id,name,descr,ductility,redundancy,importance,min_product,max_product,limit_state_type,consider_other_adj_vehicle_ind,cons_rc_ind,cons_ps_ind,cons_steel_ind,cons_timber_ind
0,1690,1,7,SERVICE-II,SERVICE-II,1.0,1.0,1.0,1.000,1.000,NaN,None,F,F,T,F
1,1690,1,8,SERVICE-III,SERVICE-III,1.0,1.0,1.0,1.000,1.000,NaN,None,F,T,F,F
2,1690,1,9,SERVICE-IV,SERVICE-IV,1.0,1.0,1.0,1.000,1.000,NaN,None,T,T,T,F
3,1690,1,10,FATIGUE-I,FATIGUE-I,1.0,1.0,1.0,1.000,1.000,NaN,None,F,F,F,F
4,1690,1,11,FATIGUE-II,FATIGUE-II,1.0,1.0,1.0,1.000,1.000,NaN,None,F,F,F,F
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36708,19751,1,17,STRENGTH-IV,STRENGTH-IV,1.0,1.0,1.0,0.857,1.158,43401.0,None,F,F,F,F
36709,19751,1,18,STRENGTH-V,STRENGTH-V,1.0,1.0,1.0,0.857,1.158,43401.0,None,F,F,T,F
36710,19751,1,19,SERVICE-I,SERVICE-I,1.0,1.0,1.0,1.000,1.000,43401.0,None,T,T,F,F
36711,19751,1,20,SERVICE-II,SERVICE-II,1.0,1.0,1.0,1.000,1.000,43401.0,None,F,F,T,F


#### ABW_LRFR_FACTOR

In [188]:
df = get_table("BRIDGEWARE", "ABW_LRFR_FACTOR")
df

,bridge_id,lrfr_factor_id,name,descr,last_change_timestamp,wood_flex,wood_shear,wood_comp_parallel,wood_comp_perpend,wood_tens_parallel,...,alum_axial_compression,alum_axial_tension_rupture,alum_axial_tension_yield,alum_pin_bearing_conn_parts,alum_bolts_bearing_conn_parts,alum_block_shear_rupture,alum_web_crippling,alum_weld_base_metal_at_welds,stl_gusset_basic_corner_check,stl_stability_bracing
0,3153,1,2011 (2014 Interim) AASHTO LRFR Spec.,"AASHTO Manual for Bridge Evaluation, 2nd Editi...",NaT,0.85,0.75,0.9,0.9,0.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3156,1,2011 (2014 Interim) AASHTO LRFR Spec.,"AASHTO Manual for Bridge Evaluation, 2nd Editi...",NaT,0.85,0.75,0.9,0.9,0.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3157,1,2011 (2014 Interim) AASHTO LRFR Spec.,"AASHTO Manual for Bridge Evaluation, 2nd Editi...",NaT,0.85,0.75,0.9,0.9,0.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3000,1,2013 Interims AASHTO LRFR Culvert Spec.,Includes only the culvert factors as approved ...,NaT,0.85,0.75,0.9,0.9,0.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3001,1,2013 Interims AASHTO LRFR Culvert Spec.,Includes only the culvert factors as approved ...,NaT,0.85,0.75,0.9,0.9,0.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8599,20552,1,2018 AASHTO LRFR Specifications,"AASHTO Manual for Bridge Evaluation, 3rd Editi...",2025-07-16 14:23:27.281344,0.85,0.75,0.9,0.9,0.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8600,20554,1,2018 (2022 Interim) AASHTO LRFR Spec.,"AASHTO Manual for Bridge Evaluation, 3rd Editi...",2025-01-19 19:16:18.557000,0.85,0.75,0.9,0.9,0.8,...,0.9,0.75,0.9,0.75,0.75,0.75,0.8,0.75,NaN,NaN
8601,20555,1,2018 AASHTO LRFR Specifications,"AASHTO Manual for Bridge Evaluation, 3rd Editi...",NaT,0.85,0.75,0.9,0.9,0.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8602,20556,1,2018 AASHTO LRFR Specifications,"AASHTO Manual for Bridge Evaluation, 3rd Editi...",NaT,0.85,0.75,0.9,0.9,0.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### ABW_LRFR_LEGAL_LOADS

In [189]:
df = get_table("BRIDGEWARE", "ABW_LRFR_LEGAL_LOADS")
df

,bridge_id,lrfr_factor_id,legal_load_id,traffic_volume_type,load_factor
0,19016,1,2,46303,1.45
1,12286,1,1,46301,1.45
2,12286,1,2,46303,1.45
3,12286,1,3,46306,1.30
4,16282,1,5,46303,1.45
...,...,...,...,...,...
26767,20556,1,2,46303,1.45
26768,20556,1,3,46306,1.30
26769,20557,1,1,46301,1.45
26770,20557,1,2,46303,1.45


#### ABW_LRFR_LEGAL_LOADS_HAUL

In [190]:
df = get_table("BRIDGEWARE", "ABW_LRFR_LEGAL_LOADS_HAUL")
df

,bridge_id,lrfr_factor_id,legal_load_id,traffic_volume_type,load_factor
0,1945,1,1,46301,1.60
1,1945,1,2,46303,1.60
2,1945,1,3,46304,1.40
3,1945,1,4,46305,1.15
4,1848,1,1,46301,1.60
...,...,...,...,...,...
26711,3677,2,2,46303,1.45
26712,3677,2,3,46306,1.30
26713,36,2,1,46301,1.45
26714,36,2,2,46303,1.45


#### ABW_LRFR_LF_TABLE_VALUE

In [191]:
df = get_table("BRIDGEWARE", "ABW_LRFR_LF_TABLE_VALUE")
df

,bridge_id,lrfr_factor_id,load_factor_table_id,load_factor_value_id,component_type,lrfr_load_factor_value
0,3381,1,1,1,60401,0.8
1,3388,1,1,1,60401,0.8
2,3389,1,1,1,60401,0.8
3,3390,1,1,1,60401,0.8
4,3391,1,1,1,60401,0.8
...,...,...,...,...,...,...
18195,19860,3,1,2,60402,0.8
18196,19861,2,1,1,60401,1.0
18197,19861,2,1,2,60402,0.8
18198,19862,2,1,1,60401,1.0


#### ABW_LRFR_LOAD_FACTOR

In [192]:
df = get_table("BRIDGEWARE", "ABW_LRFR_LOAD_FACTOR")
df

,bridge_id,lrfr_factor_id,ls_id,load_factor_id,sys_lrfr_load_group_id,sys_lrfr_loading_id,load_factor,load_factor_text,load_factor_table_id
0,967,1,9,6,4,3,NaN,Table 6A.4.5.4.2a-1,NaN
1,967,1,10,1,5,1,1.00,None,NaN
2,967,1,10,2,5,2,1.00,None,NaN
3,967,1,10,3,1,3,NaN,None,NaN
4,967,1,10,4,2,3,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...
803083,19752,1,30,4,2,3,NaN,None,NaN
803084,19752,1,30,5,3,3,1.00,None,NaN
803085,19752,1,30,6,4,3,NaN,None,NaN
803086,19752,1,31,1,5,1,1.25,None,NaN


#### ABW_LRFR_LS

In [193]:
df = get_table("BRIDGEWARE", "ABW_LRFR_LS")
df

,bridge_id,lrfr_factor_id,ls_id,bridge_type,name,descr,consider_inv_vehicle_ind,consider_opr_vehicle_ind,consider_legal_vehicle_ind,consider_permit_vehicle_ind,...,ls_max,ls_min,eh_max,eh_min,ev_max,ev_min,es_max,es_min,aw_min,aw_max
0,19969,1,1,45801,STRENGTH I,STRENGTH I,T,T,T,F,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,19969,1,2,45801,STRENGTH II,STRENGTH II,F,F,F,T,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2,19969,1,3,45801,SERVICE II,SERVICE II,T,T,T,T,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
3,19969,1,4,45801,FATIGUE,FATIGUE,T,F,F,F,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
4,19969,1,5,45802,STRENGTH I,STRENGTH I,T,T,T,F,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133843,19781,1,11,45803,SERVICE III,SERVICE III,T,F,T,F,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
133844,19781,1,12,45804,STRENGTH I,STRENGTH I,T,T,T,F,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
133845,19781,1,13,45804,STRENGTH II,STRENGTH II,F,F,F,T,...,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
133846,19781,1,14,45805,STRENGTH I,STRENGTH I,T,T,T,F,...,None,0.0,1.35,0.9,1.3,0.9,1.5,0.75,0.0,None


#### ABW_LRFR_PERMIT_LIMITED

In [194]:
df = get_table("BRIDGEWARE", "ABW_LRFR_PERMIT_LIMITED")
df

,bridge_id,lrfr_factor_id,limited_permit_id,traffic_volume_type,loading_condition_type,frequency_type,load_factor
0,18857,2,4,46302,46001,45901,1.1
1,18857,2,5,46307,46002,45901,1.2
2,18857,2,6,46307,46002,45902,1.4
3,18858,2,1,46302,46001,45901,1.1
4,18858,2,2,46307,46002,45901,1.2
...,...,...,...,...,...,...,...
29654,20556,1,2,46307,46002,45901,1.2
29655,20556,1,3,46307,46002,45902,1.4
29656,20557,1,1,46302,46001,45901,1.1
29657,20557,1,2,46307,46002,45901,1.2


#### ABW_LRFR_PERMIT_ROUTINE

In [195]:
df = get_table("BRIDGEWARE", "ABW_LRFR_PERMIT_ROUTINE")
df

,bridge_id,lrfr_factor_id,routine_permit_id,traffic_volume_type,lf_permit_under_100_kips,lf_permit_over_150_kips,ratio_under_2_kipft,ratio_between_2_3_kipft,ratio_over_3_kipft
0,2248,1,1,46303,NaN,NaN,1.40,1.35,1.30
1,2248,1,2,46304,NaN,NaN,1.35,1.25,1.20
2,2248,1,3,46305,NaN,NaN,1.30,1.20,1.15
3,2449,1,1,46303,1.8,1.3,NaN,NaN,NaN
4,2449,1,2,46304,1.6,1.2,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
25808,20529,1,5,46304,NaN,NaN,1.35,1.25,1.20
25809,20529,1,6,46305,NaN,NaN,1.30,1.20,1.15
25810,20531,1,1,46303,NaN,NaN,1.40,1.35,1.30
25811,20531,1,2,46304,NaN,NaN,1.35,1.25,1.20


#### ABW_STL_CHANNEL

In [196]:
df = get_table("BRIDGEWARE", "ABW_STL_CHANNEL")
df

,bridge_id,stl_shape_id,channel_type,nominal_weight_or_mass,area,depth,web_thick_tw,flng_width_bf,avg_flng_thick_tf,dist_k,grip,max_flng_fastener,x_bar,eo,ixx,iyy
0,127,1,23502,63.545670,8129.0160,457.2,11.4300,100.3300,15.8750,36.5760,NaN,NaN,22.2758,24.6126,2.305922e+08,5.952109e+06
1,14989,5,23501,13.393701,1703.2224,127.0,8.2550,47.8790,8.1280,19.0500,NaN,NaN,12.1412,10.8458,3.704460e+06,2.630583e+05
2,15036,6,23501,44.645670,5690.3112,304.8,12.9540,80.5180,12.7254,28.5750,NaN,NaN,17.1196,15.6972,6.742949e+07,2.139430e+06
3,15038,9,23501,50.449607,6425.7936,381.0,10.1600,86.3600,16.5100,36.5125,NaN,NaN,19.9898,22.7584,1.311129e+08,3.383961e+06
4,15043,2,23501,30.805512,3929.0244,304.8,7.1628,74.7268,12.7254,28.5750,NaN,NaN,17.7292,22.0980,5.369385e+07,1.614978e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
568,13190,8,23501,44.645670,5683.8596,304.8,12.9540,80.5180,12.7254,28.7020,NaN,NaN,17.1196,15.6972,6.742949e+07,2.131105e+06
569,13233,2,23502,63.545670,8129.0160,457.2,11.4300,100.3300,15.8750,36.5760,NaN,NaN,22.2758,24.6126,2.305922e+08,5.952109e+06
570,13234,4,23502,63.545670,8129.0160,457.2,11.4300,100.3300,15.8750,36.5760,NaN,NaN,22.2758,24.6126,2.305922e+08,5.952109e+06
571,14979,3,23501,15.625984,1993.5444,152.4,7.9756,51.6636,8.7122,20.6375,NaN,NaN,12.6746,12.3444,6.326718e+06,3.604564e+05


#### ABW_STL_COVER_PLATE

In [197]:
df = get_table("BRIDGEWARE", "ABW_STL_COVER_PLATE")
df

,bridge_id,struct_def_id,stl_component_id,bolt_def_id,side_weld_id,end_weld_id,length,width_start,width_end,relative_position,num_bolts,bolt_hole_pitch,bolt_hole_gage,development_length_start,percent_developed_start,development_length_end,percent_developed_end
0,5125,1,22,NaN,NaN,NaN,4.4196,266.7,266.7,1,NaN,NaN,NaN,0.0,100.0,0.0,100.0
1,5125,1,23,NaN,NaN,NaN,4.4196,342.9,342.9,-1,NaN,NaN,NaN,0.0,100.0,0.0,100.0
2,5125,1,27,NaN,NaN,NaN,4.4196,342.9,342.9,-1,NaN,NaN,NaN,0.0,100.0,0.0,100.0
3,5125,1,28,NaN,NaN,NaN,4.4196,266.7,266.7,1,NaN,NaN,NaN,0.0,100.0,0.0,100.0
4,5125,1,29,NaN,NaN,NaN,5.8928,266.7,266.7,1,NaN,NaN,NaN,0.0,100.0,0.0,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50744,20261,2,10,NaN,NaN,NaN,1.5240,292.1,292.1,-1,NaN,NaN,NaN,0.0,100.0,0.0,100.0
50745,20261,2,12,NaN,NaN,NaN,1.5240,241.3,241.3,1,NaN,NaN,NaN,0.0,100.0,0.0,100.0
50746,20261,2,13,NaN,NaN,NaN,1.5240,241.3,241.3,1,NaN,NaN,NaN,0.0,100.0,0.0,100.0
50747,20261,2,14,NaN,NaN,NaN,1.5240,292.1,292.1,-1,NaN,NaN,NaN,0.0,100.0,0.0,100.0


#### ABW_STL_COVER_PLATE_LOSS_RANGE

In [198]:
df = get_table("BRIDGEWARE", "ABW_STL_COVER_PLATE_LOSS_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,range_id,stl_cover_plat_loss_range_type,percent_width_loss,percent_thick_loss
0,20416,1,11,6,36508,NaN,11.1
1,20416,2,1,5,36508,NaN,33.3
2,20416,2,1,6,36508,NaN,16.7
3,20416,2,1,7,36508,NaN,16.7
4,4623,1,7,2,36508,0.0,10.0
...,...,...,...,...,...,...,...
199,19689,2,13,11,36507,0.0,5.0
200,19689,2,16,4,36507,0.0,5.0
201,19689,2,16,5,36507,0.0,5.0
202,19689,2,16,6,36507,0.0,5.0


#### ABW_STL_FATIGUE_LOC

In [199]:
df = get_table("BRIDGEWARE", "ABW_STL_FATIGUE_LOC")
df

,bridge_id,struct_def_id,super_struct_mbr_id,super_struct_spng_mbr_alt_id,anal_pt_id,stl_fatigue_loc_id,vert_dist,lfd_fatigue_category_type,lfd_allow_fatigue_stress,lrfd_fatigue_category_type,lrfd_nom_fatigue_resistance,vert_dist_ref_type
0,20555,2,1,1,4,1,61.9760,28901,None,29004,68.94757,27801
1,20555,2,1,1,4,2,57.2135,28901,None,29004,68.94757,27802
2,20555,2,1,1,5,1,61.9760,28901,None,29004,68.94757,27801
3,20555,2,1,1,5,2,57.2135,28901,None,29004,68.94757,27802
4,20555,2,2,1,4,1,61.9760,28901,None,29004,68.94757,27801
5,20555,2,2,1,4,2,57.2135,28901,None,29004,68.94757,27802
6,20555,2,2,1,5,1,61.9760,28901,None,29004,68.94757,27801
7,20555,2,2,1,5,2,57.2135,28901,None,29004,68.94757,27802
8,20555,2,8,1,4,1,61.9760,28901,None,29004,68.94757,27801
9,20555,2,8,1,4,2,57.2135,28901,None,29004,68.94757,27802


#### ABW_STL_FLNG_ANGLE_LOSS_RANGE

In [200]:
df = get_table("BRIDGEWARE", "ABW_STL_FLNG_ANGLE_LOSS_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,range_id,vert_angle_leg_ind,percent_leg_length_loss,percent_leg_thick_loss
0,20416,1,11,2,F,NaN,14.3
1,20416,1,11,3,T,NaN,7.1
2,20416,1,11,5,F,NaN,16.7
3,20416,1,12,2,F,NaN,11.1
4,20416,1,12,3,T,NaN,5.6
...,...,...,...,...,...,...,...
71,5103,1,9,12,F,NaN,15.0
72,5103,1,9,13,F,NaN,15.0
73,5103,1,9,14,F,NaN,30.0
74,5103,1,9,15,T,NaN,30.0


#### ABW_STL_FLNG_FLAT_LOSS_RANGE

In [201]:
df = get_table("BRIDGEWARE", "ABW_STL_FLNG_FLAT_LOSS_RANGE")
df

,bridge_id,struct_def_id,spng_mbr_def_id,range_id,percent_width_loss,percent_thick_loss
0,16413,1,5,9,50.0,31.6
1,16413,1,5,10,10.4,31.6
2,16413,1,8,1,10.4,31.6
3,16413,1,8,2,10.4,31.6
4,16762,3,1,2,NaN,5.0
...,...,...,...,...,...,...
1125,20049,2,6,1,NaN,63.0
1126,20049,2,1,2,NaN,45.0
1127,20049,2,6,2,NaN,41.0
1128,20052,2,1,2,NaN,25.0


#### ABW_STL_FLNG_PLATE

In [202]:
df = get_table("BRIDGEWARE", "ABW_STL_FLNG_PLATE")
df

,bridge_id,struct_def_id,stl_component_id,length,width_start,width_end,top_location_ind,end_weld_id,web_weld_id
0,19654,2,6,26.9748,406.4,406.4,T,NaN,NaN
1,19654,2,7,17.9832,406.4,406.4,T,NaN,NaN
2,19654,2,8,26.9748,406.4,406.4,T,NaN,NaN
3,19654,2,9,26.9748,457.2,457.2,F,NaN,NaN
4,19654,2,10,17.9832,457.2,457.2,F,NaN,NaN
...,...,...,...,...,...,...,...,...,...
114577,2394,3,238,4.2672,406.4,406.4,F,NaN,NaN
114578,2394,3,239,9.1440,406.4,406.4,F,NaN,NaN
114579,2394,3,240,17.6784,406.4,406.4,F,NaN,NaN
114580,2394,3,241,6.0960,406.4,406.4,F,NaN,NaN


#### ABW_STL_ISHAPE

In [203]:
df = get_table("BRIDGEWARE", "ABW_STL_ISHAPE")
df

,bridge_id,stl_shape_id,shape_type,nominal_depth,nominal_weight_or_mass,jumbo_shape_ind,area,depth,web_thick_tw,flng_width_bf,flng_thick_tf,dist_k,dist_k1,ixx,iyy,zx,zy
0,1786,1,23701,914.4,446.456700,None,56967.628,933.196,24.003,423.037,42.672,71.4375,NaN,8.449498e+09,5.411009e+08,2.064770e+07,3.949281e+06
1,17473,3,23701,355.6,50.598426,None,6451.600,355.092,7.239,171.323,11.557,25.4000,NaN,1.415187e+08,9.698192e+06,8.947335e+05,1.737028e+05
2,1825,1,23701,914.4,223.228350,None,28516.072,910.590,15.875,304.165,23.876,47.6250,NaN,3.762732e+09,1.123825e+08,9.520882e+06,1.161843e+06
3,1825,2,23701,914.4,288.708666,None,36774.120,926.846,19.431,307.721,32.004,55.5625,NaN,5.036400e+09,1.560868e+08,1.256888e+07,1.601016e+06
4,1865,1,23701,838.2,328.889769,None,41935.400,861.822,19.685,401.447,32.385,52.3875,NaN,5.327762e+09,3.496344e+08,1.401094e+07,2.687478e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7925,17647,1,23701,914.4,200.905515,None,25741.884,904.240,15.240,304.800,20.066,39.1160,28.5750,3.246605e+09,9.365207e+07,8.341016e+06,9.783077e+05
7926,17647,2,23701,914.4,252.992130,None,32258.000,919.480,17.272,304.800,27.940,46.9900,30.1625,4.370430e+09,1.331941e+08,1.094656e+07,1.373236e+06
7927,17650,2,23701,914.4,200.905515,None,25612.852,902.970,15.240,303.530,20.066,42.8625,NaN,3.246605e+09,9.365207e+07,8.341014e+06,9.783075e+05
7928,17650,3,23701,914.4,223.228350,None,28516.072,910.590,15.875,304.165,23.876,47.6250,NaN,3.762732e+09,1.123825e+08,9.520882e+06,1.161843e+06


#### ABW_CULVERT_DEF_RCBOX_SEG_CELL

In [204]:
df = get_table("BRIDGEWARE", "ABW_CULVERT_DEF_RCBOX_SEG_CELL")
df

,bridge_id,struct_def_id,culvert_def_id,culvert_def_segment_id,cell_id,top_slab_thick,bottom_slab_thick
0,16829,1,1,2,1,304.8,NaN
1,16893,1,1,1,1,304.8,304.8
2,16894,1,1,1,1,304.8,304.8
3,16899,1,1,1,1,304.8,304.8
4,16900,1,1,1,1,304.8,304.8
...,...,...,...,...,...,...,...
1370,20497,3,1,1,2,203.2,203.2
1371,20499,3,1,1,1,254.0,254.0
1372,20499,3,1,1,2,254.0,254.0
1373,20506,2,1,1,1,304.8,304.8


#### ABW_CULVERT_DEF_RCBOX_SEG_WALL

In [205]:
df = get_table("BRIDGEWARE", "ABW_CULVERT_DEF_RCBOX_SEG_WALL")
df

,bridge_id,struct_def_id,culvert_def_id,culvert_def_segment_id,wall_id,thick
0,2878,1,1,1,2,304.8
1,2879,1,1,1,1,304.8
2,2879,1,1,1,2,304.8
3,2880,1,1,1,1,304.8
4,2880,1,1,1,2,304.8
...,...,...,...,...,...,...
2654,2877,1,1,1,1,304.8
2655,2877,1,1,1,2,304.8
2656,2878,1,1,1,1,304.8
2657,16409,1,1,3,1,609.6


#### ABW_DECK_CORR_METAL_PANEL

In [206]:
df = get_table("BRIDGEWARE", "ABW_DECK_CORR_METAL_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,plank_depth,plank_moment_of_inertia,plank_section_modulus,plank_load,plank_yield_strength,fill_weight,fill_thickness_above_plank,plank_thickness,plank_dimension_a,plank_dimension_b,plank_dimension_c,wheel_load_dist_parallel,wheel_load_dist_perpend,plank_panel_length,plank_phi
0,18126,2,1,76.20,7.129867e+06,1.748386e+05,0.699052,344.737850,2322.716258,120.650,5.35940,38.10000,76.20000,76.20000,254.0,508.0,8.534400,None
1,16926,1,1,76.20,7.009109e+06,1.671387e+05,0.646383,344.737850,2322.716258,127.000,5.30860,40.94480,98.42500,87.29980,254.0,508.0,3.048000,None
2,18835,1,1,76.20,2.533850e+08,2.958596e+05,0.564029,344.737850,2322.716258,0.000,5.30860,40.94480,71.98360,71.98360,254.0,457.2,7.315200,None
3,1904,1,1,0.00,2.133732e+07,6.720417e+05,0.498912,13.789514,0.000000,0.000,63.50000,38.10000,76.20000,76.20000,254.0,508.0,41.833800,None
4,5262,1,1,50.80,3.147385e+06,1.121789e+05,2.082791,227.526981,2322.716258,0.000,5.31368,25.40000,50.80000,50.80000,304.8,304.8,1.219200,None
5,5340,1,1,76.20,7.188322e+06,1.780480e+05,0.550623,275.790280,2322.716258,38.100,4.54660,15.87500,98.42500,98.42500,254.0,508.0,5.037734,None
6,5839,1,1,76.20,2.136024e+07,4.639748e+05,7.182039,344.737850,2322.716258,76.200,15.87500,38.10000,76.20000,76.20000,203.2,203.2,NaN,None
7,5927,1,1,76.20,5.538478e+06,1.378320e+05,0.502743,689.475700,2322.716258,63.500,4.16560,38.10000,76.20000,76.20000,254.0,508.0,1.243584,None
8,12774,1,1,76.20,7.570812e+06,1.802799e+05,0.564029,344.737850,2322.716258,76.200,5.30860,29.26842,94.78518,83.65998,254.0,508.0,6.096000,None
9,4303,1,1,76.20,8.404600e+06,2.044886e+05,0.703840,275.790280,2322.716258,76.200,5.30860,15.87500,98.42500,95.25000,254.0,508.0,3.657600,None


#### ABW_DECK_GENERIC_PANEL

In [207]:
df = get_table("BRIDGEWARE", "ABW_DECK_GENERIC_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,total_thick,density
0,17082,2,1,NaN,NaN
1,1329,2,1,0.0254,16.018733
2,5489,1,1,114.3000,1068.449479
3,2036,2,1,NaN,NaN
4,4348,2,1,NaN,NaN
5,4331,1,1,NaN,NaN
6,4369,1,1,0.0000,0.000000
7,5602,2,1,101.6000,720.842977
8,14968,3,1,127.0000,656.768045
9,14613,1,1,101.6000,2322.716258


#### ABW_LIB_PS_SHAPE_STRAND_GRID

In [208]:
df = get_table("BRIDGEWARE", "ABW_LIB_PS_SHAPE_STRAND_GRID")
df

,ps_shape_id,row_id,num_strands,vert_dist,horz_spacing
0,195,11,2,558.8,50.8
1,195,12,2,609.6,50.8
2,195,13,2,660.4,50.8
3,195,14,2,711.2,50.8
4,195,15,2,762.0,50.8
...,...,...,...,...,...
1320,195,6,2,304.8,50.8
1321,195,7,2,355.6,50.8
1322,195,8,2,406.4,50.8
1323,195,9,2,457.2,50.8


#### ABW_LIB_PS_TEE_BEAM

In [209]:
df = get_table("BRIDGEWARE", "ABW_LIB_PS_TEE_BEAM")
df

,ps_shape_id,nominal_depth,depth,left_flng_width,right_flng_width,flng_thick,flng_haunch_flng_height,flng_haunch_web_width,flng_haunch_web_height,top_web_thick,...,dist_y_to_cg,ixx,sxx_top,sxx_bot,half_depth_area_neg_flex,half_depth_area_pos_flex,st_venant_torsional_constant,volume_surface_ratio,nominal_weight_or_mass,checksum


#### ABW_LIB_REBAR

In [210]:
df = get_table("BRIDGEWARE", "ABW_LIB_REBAR")
df

,rebar_id,name,descr,library_type,si_or_us_type,weight_per_length,nominal_diameter,xsection_area,perimeter,checksum,lib_rebar_type
0,23,"0.25"" sq.",0.2500 inch square rebar,22901,10401,NaN,7.16534,40.322500,25.4000,None,64102
1,24,"0.375"" sq.",0.3750 inch square rebar,22901,10401,NaN,10.74674,90.709496,38.1000,None,64102
2,25,"0.5"" sq.",0.5000 inch square rebar,22901,10401,NaN,14.33068,161.290000,50.8000,None,64102
3,26,"0.625"" sq.",0.6250 inch square rebar,22901,10401,NaN,17.91208,251.999496,63.5000,None,64102
4,27,"0.75"" sq.",0.7500 inch square rebar,22901,10401,NaN,21.49602,362.902500,76.2000,None,64102
5,28,"0.875"" sq.",0.8750 inch square rebar,22901,10401,NaN,25.07742,493.934496,88.9000,None,64102
6,29,"1"" sq.",1.0000 inch square rebar,22901,10401,NaN,28.66136,645.160000,101.6000,None,64102
7,30,"1.125"" sq.",1.1250 inch square rebar,22901,10401,NaN,32.24276,816.514496,114.3000,None,64102
8,31,"1.25"" sq.",1.2500 inch square rebar,22901,10401,NaN,35.82670,1008.062500,127.0000,None,64102
9,32,"1.375"" sq.",1.3750 inch square rebar,22901,10401,NaN,39.40810,1219.739496,139.7000,None,64102


#### ABW_LIB_STAY_IN_PLACE_FORM

In [211]:
df = get_table("BRIDGEWARE", "ABW_LIB_STAY_IN_PLACE_FORM")
df

,sip_form_id,name,descr,library_type,si_or_us_type,stay_in_place_form_weight
0,1,Standard,standard metal sip form,22901,10401,0.015


#### ABW_LIB_STL_ANGLE

In [212]:
df = get_table("BRIDGEWARE", "ABW_LIB_STL_ANGLE")
df

,stl_shape_id,angle_size_1,angle_size_2,angle_thick,nominal_weight_or_mass,k,area,ixx,yxx,iyy,xyy,rzz,zztantheta,checksum
0,713,152.4,101.6,19.0500,35.121260,None,4477.4104,1.019767e+07,52.8320,3.612889e+06,27.4320,21.8440,NaN,None
1,714,152.4,101.6,15.8750,29.763780,None,3780.6376,8.782483e+06,51.5620,3.130060e+06,26.1620,21.9456,NaN,None
2,715,152.4,101.6,14.2875,26.936221,None,3425.7996,8.033267e+06,51.0540,2.876159e+06,25.6540,21.9964,NaN,None
3,716,152.4,101.6,12.7000,24.108662,None,3064.5100,7.242427e+06,50.5460,2.609771e+06,25.0698,22.0980,NaN,None
4,717,152.4,101.6,11.1125,21.281103,None,2696.7688,6.451587e+06,49.7840,2.330896e+06,24.4856,22.1742,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,2124,76.2,63.5,12.7000,12.649607,None,1612.9000,8.615991e+05,25.2730,5.369385e+05,18.9484,13.1064,NaN,None
498,2125,76.2,63.5,11.1252,11.310236,None,1432.2552,7.783528e+05,24.6888,4.869908e+05,18.3896,13.1064,NaN,None
499,2126,76.2,63.5,9.5250,9.822047,None,1245.1588,6.867819e+05,24.1046,4.287184e+05,17.8054,13.1318,NaN,None
500,2127,76.2,63.5,7.9502,8.333858,None,1051.6108,5.868863e+05,23.4950,3.696135e+05,17.1958,13.1572,NaN,None


#### ABW_LIB_STL_CHANNEL

In [213]:
df =  get_table("BRIDGEWARE", "ABW_LIB_STL_CHANNEL")
df

,stl_shape_id,channel_type,nominal_weight_or_mass,area,depth,web_thick_tw,flng_width_bf,avg_flng_thick_tf,dist_k,grip,max_flng_fastener,x_bar,eo,ixx,iyy,checksum
0,3337,23501,63.545670,8129.0160,457.2,11.4300,100.330,15.8750,34.925,15.875,25.4,22.2758,24.6126,2.305922e+08,5.993733e+06,None
1,1952,23501,74.409450,9483.8520,381.0,18.1864,94.488,16.5100,36.576,NaN,NaN,20.2946,14.8082,1.681575e+08,4.578546e+06,None
2,1953,23501,59.527560,7612.8880,381.0,13.2080,89.408,16.5100,36.576,NaN,NaN,19.7612,19.4818,1.448485e+08,3.816842e+06,None
3,1954,23501,50.449607,6451.6000,381.0,10.1600,86.360,16.5100,36.576,NaN,NaN,20.0152,22.7584,1.311129e+08,3.358988e+06,None
4,1955,23501,44.645670,5683.8596,304.8,12.9540,80.518,12.7254,28.702,NaN,NaN,17.1196,15.6972,6.742949e+07,2.131105e+06,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
262,2873,23502,17.900000,2280.0000,152.0,7.8700,63.500,9.5300,22.200,NaN,NaN,17.9000,18.4000,7.780000e+06,7.700000e+05,None
263,2874,23502,10.400000,1350.0000,152.0,4.5500,47.800,7.3900,19.100,NaN,NaN,12.7000,14.8000,4.750000e+06,2.510000e+05,None
264,2875,23502,9.700000,1260.0000,152.0,3.9400,47.000,7.3900,19.100,NaN,NaN,13.0000,15.5000,4.580000e+06,2.350000e+05,None
265,2876,23502,20.500000,2600.0000,102.0,12.7000,63.500,12.7000,25.400,NaN,NaN,21.6000,16.3000,3.680000e+06,8.870000e+05,None


#### ABW_LIB_STL_ISHAPE

In [214]:
df =  get_table("BRIDGEWARE", "ABW_LIB_STL_ISHAPE")
df

,stl_shape_id,shape_type,nominal_depth,nominal_weight_or_mass,jumbo_shape_ind,area,depth,web_thick_tw,flng_width_bf,flng_thick_tf,dist_k,dist_k1,ixx,iyy,zx,zy,checksum
0,162,23701,406.4,38.692914,F,4954.8288,398.526,6.350,139.700,8.763,26.9875,NaN,1.252857e+08,3.991659e+06,7.243081e+05,8.980109e+04,None
1,163,23701,355.6,1202.456712,T,152902.9200,580.136,94.996,471.424,130.048,147.6375,NaN,6.659703e+09,2.293435e+09,3.005387e+07,1.519080e+07,None
2,164,23701,355.6,1086.377970,T,138709.4000,569.468,77.978,454.406,124.714,141.2875,NaN,5.952109e+09,1.964612e+09,2.720252e+07,1.337184e+07,None
3,165,23701,355.6,989.645685,T,126451.3600,549.656,71.882,448.310,114.808,131.7625,NaN,5.161270e+09,1.735685e+09,2.425285e+07,1.196255e+07,None
4,166,23701,355.6,900.354345,T,114838.4800,531.368,65.913,442.341,105.664,122.2375,NaN,4.495299e+09,1.531732e+09,2.163092e+07,1.068436e+07,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1319,2681,23701,310.0,143.000000,F,18200.0000,323.000,14.000,310.000,22.900,38.1000,28.6,3.470000e+08,1.120000e+08,2.410000e+06,1.110000e+06,None
1320,2682,23701,310.0,129.000000,F,16500.0000,318.000,13.100,307.000,20.600,35.8000,27.0,3.080000e+08,1.000000e+08,2.160000e+06,9.900000e+05,None
1321,2683,23701,310.0,117.000000,F,15000.0000,315.000,11.900,307.000,18.700,33.8000,27.0,2.760000e+08,8.990000e+07,1.950000e+06,8.900000e+05,None
1322,2684,23701,310.0,107.000000,F,13600.0000,312.000,10.900,305.000,17.000,32.3000,27.0,2.480000e+08,8.120000e+07,1.770000e+06,8.060000e+05,None


#### ABW_LIB_STL_RAILING

In [215]:
df =  get_table("BRIDGEWARE", "ABW_LIB_STL_RAILING")
df

,stl_railing_id,name,descr,library_type,si_or_us_type,railing_matl_type,eff_wind_height,width,weight_per_length,dist_to_cg,checksum
0,2,Twin Steel Tube Bridge Railing,TST-1-99 (10-17-03),22902,10401,28801,800.1,254.000,1.163865,NaN,None
1,1,VPF,Vandal Protection Fence,22902,10401,28801,NaN,NaN,0.729696,NaN,None
2,3,Rail,Pedestrian Rail,22902,10401,28801,NaN,NaN,0.291878,NaN,None
3,4,Type 5 with Tubular Backup,GR-2.2,22902,10401,28801,736.6,440.944,0.802665,76.2,None
4,5,Deep Beam Rail (not for PS Box),Std Dwg DBR-2-73,22902,10401,28801,596.9,431.800,0.437818,NaN,None
5,7,Retrofit Deep Beam Rail (not for PS Box),Std Dwgs DBR-2-73 and DBR-3-11,22902,10401,28801,596.9,431.800,0.875635,NaN,None
6,6,Deep Beam Rail (max for PS Box),Std Dwg DBR-2-73,22902,10401,28801,711.2,635.000,0.467005,NaN,None
7,8,Bridge Rail CS-2-54,Sidewalk not included,22902,10401,28801,NaN,838.200,0.437818,NaN,None


#### ABW_LIB_STL_STRUCT_TEE

In [216]:
df =  get_table("BRIDGEWARE", "ABW_LIB_STL_STRUCT_TEE")
df

,stl_shape_id,shape_type,nominal_weight_or_mass,area,tee_depth_d,stem_thick_tw,stem_area,flng_width_bf,flng_thick_tf,dist_k,dist_y_to_cg,ixx,iyy,checksum
0,392,23701,69.944883,8903.20800,341.884,12.4460,None,253.746,18.9230,36.5125,86.6140,9.947931e+07,2.580635e+07,None
1,393,23701,62.503938,7999.98400,339.217,11.6840,None,252.984,16.2560,34.9250,88.3920,8.990599e+07,2.197702e+07,None
2,394,23701,366.094494,46451.52000,376.555,50.0380,None,358.521,89.9160,109.5375,103.3780,4.703415e+08,3.483857e+08,None
3,395,23701,303.590556,38387.02000,362.458,41.9100,None,350.520,75.9460,95.2500,94.9960,3.637863e+08,2.742965e+08,None
4,396,23701,249.271658,31741.87200,349.504,35.0520,None,343.408,62.9920,82.5500,86.8680,2.851185e+08,2.135267e+08,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1229,2432,23702,4.836614,618.70844,101.600,3.4290,None,57.912,4.8006,14.3002,29.9720,6.534833e+05,7.825151e+04,None
1230,2433,23702,4.613386,587.74076,101.600,3.2766,None,57.912,4.4958,11.1252,29.9720,6.243471e+05,7.325673e+04,None
1231,2434,23702,3.274016,417.41852,76.200,2.8956,None,46.736,4.3434,9.5250,21.3614,2.409980e+05,3.733596e+04,None
1232,2435,23702,2.753150,351.61220,75.184,2.4892,None,50.800,3.2766,7.9502,21.0058,2.010398e+05,3.592077e+04,None


ABW_LIB_SUB_LRFD_DS_LS


In [217]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_CULVERT_DEF_RC_SEGMENT

In [218]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_CULVERT_DEF_RC_SEGMENT


In [219]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_ADV_CONC_PRECAST_XSECTION


In [220]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_ADV_CONC_XSECT_RANGE


In [221]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_AGENCY_WIND_EFFECT


In [222]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_BMDEF_SPAN


In [223]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_FSYS_STRGRP_FLOOR_BEAM


In [224]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_FSYS_SUPPORT


In [225]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_GIRDER_SYS_FRAME_CONN


In [226]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_GIRDER_SYS_INT_DIAPH


In [227]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_GIRDER_SYS_STRUCT_DEF


In [228]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_GLINE_MBR


In [229]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_BMDEF_DIAPH_LOC


In [230]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_GLINE_SUPPORT


In [231]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_LIB_VEHICLE_AXLE_WHEEL


In [232]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_LIB_WELDED_WIRE_REINF


In [233]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_LL_DISTFACTOR_RANGE


In [234]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_LRFD_FACTOR


In [235]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_LRFD_LF_TABLE_VALUE


In [236]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_LRFD_LOAD_FACTOR


In [237]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_SYS_BRM_SYNC_SETTING_VALUE


In [238]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_BEAM_DEF_KNEE_BRACE


In [239]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_BEAM_HINGE_LOC


In [240]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_MCELL_BDEF_TRANS_REF_LINE


In [241]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_SYS_DATABASE


In [242]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_WEB_CONCENT_LOAD


In [243]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_BMDEF_ANAL_PT_COMPONENT


In [244]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_CULVERT_COMPONENT


In [245]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_DECK_PANEL_COMPONENT


In [246]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_ABUTMENT_SUB_STRUCT


In [247]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_BEAM_DEF_FK


In [248]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_BEARING_ELASTOMERIC


In [249]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_BEARING_POT


In [250]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_BEARING_ROCKER


In [251]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_SUB_STRUCT_EARTH_LOAD


In [252]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_SUB_STRUCT_ICE_LOAD


In [253]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_SUB_STRUCT_PILE_DEF


In [254]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_SUB_STRUCT_PS_PILE_DEF


In [255]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_BMDEF_CONCENT_LOAD


In [256]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

,bridge_id,struct_def_id,deck_panel_id,deck_panel_type,dist,length,width_left_start,width_right_start,width_left_end,width_right_end,...,lrfd_spec_edition_choice_type,lrfr_spec_edition_choice_type,asr_spec_edition_guid,lfr_spec_edition_guid,lrfd_spec_edition_guid,lrfr_spec_edition_guid,override_asr_factor_id,override_lfr_factor_id,override_lrfd_factor_id,override_lrfr_factor_id
0,1750,4,1,21501,0.0,22.860000,-6.09600,6.09600,-6.09600,6.09600,...,NaN,NaN,None,None,None,None,None,None,None,None
1,2159,2,1,21501,0.0,20.574000,-11.43000,7.77240,-11.43000,7.77240,...,NaN,NaN,None,None,None,None,None,None,None,None
2,2110,1,1,21501,0.0,39.255700,-7.16280,7.16280,-7.16280,7.16280,...,NaN,NaN,None,None,None,None,None,None,None,None
3,2146,1,1,21501,0.0,24.663502,-1.82880,1.82880,-1.82880,1.82880,...,NaN,NaN,None,None,None,None,None,None,None,None
4,2114,7,1,21501,0.0,23.602950,-7.77240,11.43000,-7.77240,11.43000,...,NaN,NaN,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13251,20555,2,1,21501,0.0,62.484000,-7.67080,9.72820,-7.67080,9.72820,...,NaN,NaN,None,None,None,None,None,None,None,None
13252,20556,7,1,21501,0.0,52.428800,-7.01040,6.07060,-7.01040,6.07060,...,NaN,NaN,None,None,None,None,None,None,None,None
13253,20556,12,1,21501,0.0,52.428800,-6.07060,7.01040,-6.07060,7.01040,...,NaN,NaN,None,None,None,None,None,None,None,None
13254,20557,1,1,21501,0.0,5.791200,-14.02080,14.02080,-14.02080,14.02080,...,NaN,NaN,None,None,None,None,None,None,None,None


ABW_LIB_TIMBER_SIZE_CLASS


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_TIMBER_SPECIES


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_VEHICLE_LFD


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_ADV_CONC_CIP_XSECTION


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LOADCOMBO_SETTING_TEMPLATE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LOAD_PALETTE_TEMPLATE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_MBRALT_SCHCPLT_LOSS_RANGE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_STL_XSEC_CPLATE_LOSS_RANGE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_STRINGER_MBR


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_STRUCT_DEF_STRINGER_DLCASE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUBALT_ANALYSIS_OPTIONS


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUBALT_ANALYSIS_OPTIONS_LS


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUBALT_ANAL_OPTION_VEHICLE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUBSALT_LOADCOMBO_SETTING


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUBSALT_LOAD_PALETTE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUBSALT_LRFD_LOAD_COMBO


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUBSALT_WATERLEVEL_SETTING


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUBSDEF_FOUND


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUPPORT_LINE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUPSTRUCTDEF_BRAC_MBR_LOSS


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SYS_DATA_DICTIONARY


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SYS_UNIT


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_TIMBER_COMMERCIALGRADE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_TIMBER_GRAD_RULE_AGNCY


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_MBRALT_XSCCPLT_LOSS_RANGE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_MBR_ALT_IMPORT_MESSAGE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_SHAPE_STRAND_GRID


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_MBR_CONN_DEF


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_MODIFICATION_EVENT


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_TEE_BEAM


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PARTIAL_SUPER_STRUCT_DEF


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PERSON


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PIER_CAP_BOT_COLUMN_BAY


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PIER_CAP_COL


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PIER_CAP_ENCASEMENT_WALL


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PIER_CAP_PILE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PIER_COL_CIRC_XSEC


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PIER_COL_FOUNDATION


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PIER_COL_RECT_XSEC


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_UBEAM


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_STL_BEAM_LOSS_RANGE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_GENPREF_TEMPLATE_ITEM


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_CHECK_IN_OUT_EVENT


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_COMMENT


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_COMMENT_ITEM


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_CONC_LEG_DEF


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_STL_BEARING_STIFF_LOC


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_CREATION_EVENT


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUPER_STRUCT


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_MULTI_CELL_BOX_BEAM_DEF


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_FLOORSYS_STRINGER_MBR


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_CONC_RAILING


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_CORR_METAL_PANEL


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_LRFD_FACTOR


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_LRFD_LF_TABLE_VALUE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_LRFD_LOAD_FACTOR


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_LRFD_LS


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RESULTS_CRIT_STRESSES_ASR


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_CORR_MP_CULV_ITEM


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_CORRUGATED_MP_CULV


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_LIB_SPRL_RIB_MP_CULV_ITEM


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RATING_TOOL_PERM_SCENARIO


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RC_BOX_CULVERT_DEF


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RC_BOX_CULVERT_DEF_CELL


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RC_LEG_CIRC_XSECT_REINF


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RC_LEG_RECT_XSECT_REINF


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RC_LEG_XSECTION


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RC_LEG_XSECTION_RANGE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RC_VARIABLE_XSECTION_RANGE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RC_XSECTION


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RC_XSECTION_RANGE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_RC_XSECTION_REINF


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_BOX_BEAM


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_BOX_INT_DIAPH_LOC


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_CONC_STRESS_LIMIT


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_CONC_STRESS_LIMIT_RANGE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_IBEAM


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_PRECAST_BEAM_CONT_REBAR


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_PRECAST_BEAM_SPAN


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_PRECAST_MILD_STL_LAYOUT


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_PRECAST_STRAND_LAYOUT


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_PROPERTIES


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_PS_SHAPE_MILD_STEEL


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_CHECKED_OUT_STRUCT_DEF


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_BMDEF_LL_DISTFACTOR_RANGE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_BF_LAT_BRACING_PROPERTIES


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_CHECKOUT_AUTHORIZATION


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_BRIDGE_SUB_LRFD_DSVEH_LOAD


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_BRIDGE_WATER_LEVEL


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SYS_TABLE_PROPERTIES


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_CONC_BMDEF_EFF_SUPPORT


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_CONC_BMDEF_FRAME_CONN


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_CONC_BMDEF_REINF_PROFILE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SYS_BRDR_TABLE


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_BRIDGE_ALT


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUB_DR_SHAFT_QW


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUB_DR_SHAFT_REINF_BAR


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUB_DR_SHAFT_SHEAR_REINF


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

ABW_SUB_DR_SHAFT_TZ


In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

#### ABW_DECK_PANEL

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_PANEL")
df

#### ABW_SYS_TYPE

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_SYS_TYPE")
df

#### ABW_DECK_TIMBER_PANEL

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_DECK_TIMBER_PANEL")
df

#### ABW_GROUP_BRIDGE

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_SLAB_SYS_STRUCT_DEF")
df

#### ABW_GROUP_USER

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_GROUP_USER")
df

#### ABW_GROUP_ITEM_USER

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_GROUP_ITEM_USER")
df

#### ABW_GROUP_ITEM_BRIDGE

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_GROUP_ITEM_BRIDGE")
df

#### ABW_STL_TRANS_STIFF

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_STL_TRANS_STIFF")
df

#### ABW_GUSSET_PLATE_DEF

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_GUSSET_PLATE_DEF")
df

#### ABW_GUSSET_PLATE_MEMBER_DEF

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_GUSSET_PLATE_MEMBER_DEF")
df

#### ABW_LANE_POSITION

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_LANE_POSITION")
df

#### ABW_BRIDGE_EXCHANGE

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_BRIDGE_EXCHANGE")
df

#### ABW_PIER_DEF_CAP_FLEX_REINF

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_PIER_DEF_CAP_FLEX_REINF")
df

#### ABW_PIER_DEF_CAP_SHEAR_REINF

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_PIER_DEF_CAP_SHEAR_REINF")
df

#### ABW_IMPORT_EVENT

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_IMPORT_EVENT")
df

#### ABW_IMPORT_MESSAGE

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_IMPORT_MESSAGE")
df

#### ABW_SHEAR_CONNECTOR

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_SHEAR_CONNECTOR")
df

#### ABW_SHEAR_REINF_DEF

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_SHEAR_REINF_DEF")
df

#### ABW_SLAB_SYS_STRUCT_DEF

In [ ]:
df =  get_table("BRIDGEWARE", "ABW_SLAB_SYS_STRUCT_DEF")
df

#### IABW_PIER_RC_COL_RECT_XSEC

In [ ]:
# df =  get_table("BRIDGEWARE", "IABW_PIER_RC_COL_RECT_XSEC")
# df
# //TODO - This one doesn't seem to actually exist

#### ABW_PIER_SUB_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_SUB_STRUCT_DEF")

#### ABW_PIER_SUBSDEF_COLUMN

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_SUBSDEF_COLUMN")

#### ABW_PIER_SUBSDEF_COLUMN_SEG

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_SUBSDEF_COLUMN_SEG")

#### ABW_PIER_TOP_CAP_SEG

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_TOP_CAP_SEG")

#### ABW_PT_LOSS_PROPERTIES

In [ ]:
get_table("BRIDGEWARE", "ABW_PT_LOSS_PROPERTIES")

#### ABW_RATING_RESULTS

In [ ]:
get_table("BRIDGEWARE", "ABW_RATING_RESULTS")

#### ABW_RATING_RESULTS_SUMMARY

In [ ]:
get_table("BRIDGEWARE", "ABW_RATING_RESULTS_SUMMARY")

#### ABW_SUBSDEF_FOUND_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_FOUND_DEF")

#### ABW_ADV_CONC_RC_BEAM_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_ADV_CONC_RC_BEAM_DEF")

#### ABW_ACCESS_PRIVILEGE

In [ ]:
get_table("BRIDGEWARE", "ABW_ACCESS_PRIVILEGE")

#### ABW_ANALYSIS_REPORTS_TEMPLATE

In [ ]:
get_table("BRIDGEWARE", "ABW_ANALYSIS_REPORTS_TEMPLATE")

#### ABW_ANAL_PT_CONC_REINF

In [ ]:
get_table("BRIDGEWARE", "ABW_ANAL_PT_CONC_REINF")

#### ABW_SUPPORT

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPPORT")

#### ABW_GUSSET_PLATE_PANEL_POINT

In [ ]:
get_table("BRIDGEWARE", "ABW_GUSSET_PLATE_PANEL_POINT")

#### ABW_FSYS_STRGRP_DEF_TEMPLATE

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_STRGRP_DEF_TEMPLATE")

#### ABW_LIB_LFR_LOADING

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LFR_LOADING")

#### ABW_FLOORSYS_FLOOR_BEAM_MBR

In [ ]:
get_table("BRIDGEWARE", "ABW_FLOORSYS_FLOOR_BEAM_MBR")

#### ABW_FLOORSYS_MBR_LOC

In [ ]:
get_table("BRIDGEWARE", "ABW_FLOORSYS_MBR_LOC")

#### ABW_SUPER_STRUCT_DEF_DL_DIST

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_DEF_DL_DIST")

#### ABW_SUPER_DL_REACT_SUB_DETAIL

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_DL_REACT_SUB_DETAIL")

#### ABW_SUPER_DLREACT_SUB_OVERRIDE

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_DLREACT_SUB_OVERRIDE")

#### ABW_LIB_LFR_FACT_SPEC_EDITION

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LFR_FACT_SPEC_EDITION")

#### ABW_LIB_LFR_FACTOR

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LFR_FACTOR")

#### ABW_RC_FRAME_PIER_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_RC_FRAME_PIER_STRUCT_DEF")

#### ABW_RC_I_BEAM_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_RC_I_BEAM_DEF")

#### ABW_RC_PIER_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_RC_PIER_STRUCT_DEF")

#### ABW_RC_PILEBENT_PIER_STR_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_RC_PILEBENT_PIER_STR_DEF")

#### ABW_RC_SLAB_BEAM_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_RC_SLAB_BEAM_DEF")

#### ABW_RC_TEE_BEAM_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_RC_TEE_BEAM_DEF")

#### ABW_MATL_ALUMINUM

In [ ]:
get_table("BRIDGEWARE", "ABW_MATL_ALUMINUM")

#### ABW_CULVERT_BAR_MARK_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_CULVERT_BAR_MARK_DEF")

#### ABW_GLINE_MBR_FRAME_CONN

In [ ]:
get_table("BRIDGEWARE", "ABW_GLINE_MBR_FRAME_CONN")

#### ABW_GLINE_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_GLINE_STRUCT_DEF")

#### ABW_MULTI_CELL_BOX_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_STRUCT_DEF")

#### ABW_SLAB_SYS_STRUCT_DEF_FK

In [ ]:
get_table("BRIDGEWARE", "ABW_SLAB_SYS_STRUCT_DEF_FK")

#### ABW_BMDEF_CONC_DECK_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_BMDEF_CONC_DECK_RANGE")

#### ABW_FLOORBEAM_MBR_ALT

In [ ]:
get_table("BRIDGEWARE", "ABW_FLOORBEAM_MBR_ALT")

#### ABW_LIB_LRFD_LOAD_FACTOR_TABLE

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFD_LOAD_FACTOR_TABLE")

#### ABW_CULVERT_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_CULVERT_STRUCT_DEF")

#### ABW_LIB_LRFR_LOAD_FACTOR_TABLE

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_LOAD_FACTOR_TABLE")

#### ABW_LRFD_LOAD_FACTOR_TABLE

In [ ]:
get_table("BRIDGEWARE", "ABW_LRFD_LOAD_FACTOR_TABLE")

#### ABW_FLOORLINE_FLOOR_BEAM_MBR

In [ ]:
get_table("BRIDGEWARE", "ABW_FLOORLINE_FLOOR_BEAM_MBR")

#### ABW_SYS_ANAL_MODULE_MBR_TYPE

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_ANAL_MODULE_MBR_TYPE")

#### ABW_FLOORLINE_MBR_SUPPORT

In [ ]:
get_table("BRIDGEWARE", "ABW_FLOORLINE_MBR_SUPPORT")

#### ABW_FLOORLINE_STRINGER_MBR

In [ ]:
get_table("BRIDGEWARE", "ABW_FLOORLINE_STRINGER_MBR")

#### ABW_FLOORLINE_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_FLOORLINE_STRUCT_DEF")

#### ABW_GIRDER_MBR_ALT

In [ ]:
get_table("BRIDGEWARE", "ABW_GIRDER_MBR_ALT")

#### ABW_TRUSS_MBR_ALT

In [ ]:
get_table("BRIDGEWARE", "ABW_TRUSS_MBR_ALT")

#### ABW_STRINGER_MBR_ALT

In [ ]:
get_table("BRIDGEWARE", "ABW_STRINGER_MBR_ALT")

#### ABW_STL_XSECTION_COVER_PLATE

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_XSECTION_COVER_PLATE")

#### ABW_STL_XSECTION_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_XSECTION_RANGE")

#### ABW_STL_XSECTION_REINF

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_XSECTION_REINF")

#### ABW_STLBD_TOPFL_LSUP_LOC_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_STLBD_TOPFL_LSUP_LOC_RANGE")

#### ABW_STLBD_TOPFLNG_LATSUP_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_STLBD_TOPFLNG_LATSUP_RANGE")

#### ABW_CULVERT_STRUCT_ALT

In [ ]:
get_table("BRIDGEWARE", "ABW_CULVERT_STRUCT_ALT")

#### ABW_EVENT

In [ ]:
get_table("BRIDGEWARE", "ABW_EVENT")

#### ABW_MULTI_CELL_BOX_BDEF_SEG

In [ ]:
get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_BDEF_SEG")

#### ABW_MULTI_CELL_BOX_INT_DIAPH

In [ ]:
get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_INT_DIAPH")

#### ABW_MCB_SEG_LANE_POSITION

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_SEG_LANE_POSITION")

#### ABW_DIAPH_PROPERTIES

In [ ]:
get_table("BRIDGEWARE", "ABW_DIAPH_PROPERTIES")

#### ABW_BRIDGE_STREAM_FLOW_EFFECT

In [ ]:
get_table("BRIDGEWARE", "ABW_BRIDGE_STREAM_FLOW_EFFECT")

#### ABW_BRIDGE_SUB_LRFD_DS_LS

In [ ]:
get_table("BRIDGEWARE", "ABW_BRIDGE_SUB_LRFD_DS_LS")

#### ABW_LIB_SPIRAL_RIB_MP_CULV

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_SPIRAL_RIB_MP_CULV")

#### ABW_LIB_STR_PL_MP_CULV_ITEM

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_STR_PL_MP_CULV_ITEM")

#### ABW_LIB_STRUCT_PLATE_MP_CULV

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_STRUCT_PLATE_MP_CULV")

#### ABW_LIB_METAL_PIPE_CULVERT

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_METAL_PIPE_CULVERT")

#### ABW_LIB_METAL_BOX_CULVERT_ITEM

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_METAL_BOX_CULVERT_ITEM")

#### ABW_LIB_MATL_TIMBER_GLULAM

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_TIMBER_GLULAM")

#### ABW_LIB_MATL_TMBR_GLULAM_ITEM

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_TMBR_GLULAM_ITEM")

#### ABW_RC_BEAM_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_RC_BEAM_DEF")

#### ABW_TEMP_GRADIENT_LOAD

In [ ]:
get_table("BRIDGEWARE", "ABW_TEMP_GRADIENT_LOAD")

#### ABW_STL_BEAM_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_BEAM_DEF")

#### ABW_SUPER_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_DEF")

#### ABW_SYS_LINE_GRD_ENG_DEFAULT

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_LINE_GRD_ENG_DEFAULT")

#### ABW_SYS_MODULE_SPEC_DEFAULT

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_MODULE_SPEC_DEFAULT")

#### ABW_SYS_THREE_D_ENG_DEFAULT

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_THREE_D_ENG_DEFAULT")

#### ABW_BMDEF_INT_SUPPORT_DETAIL

In [ ]:
get_table("BRIDGEWARE", "ABW_BMDEF_INT_SUPPORT_DETAIL")

#### ABW_BMDEF_INTERMEDIATE_SUPPORT

In [ ]:
get_table("BRIDGEWARE", "ABW_BMDEF_INTERMEDIATE_SUPPORT")

#### ABW_SYS_SPEC_EDITION_SOIL

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_SPEC_EDITION_SOIL")

#### ABW_FL_FLOORBEAM_STRINGER_SPAN

In [ ]:
get_table("BRIDGEWARE", "ABW_FL_FLOORBEAM_STRINGER_SPAN")

#### ABW_MBR_DISTRIB_LOAD

In [ ]:
get_table("BRIDGEWARE", "ABW_MBR_DISTRIB_LOAD")

#### ABW_BRIDGE_EXPLORER_SETTING

In [ ]:
get_table("BRIDGEWARE", "ABW_BRIDGE_EXPLORER_SETTING")

#### ABW_EVENT_COMPONENT

In [ ]:
get_table("BRIDGEWARE", "ABW_EVENT_COMPONENT")

#### ABW_EVENT_COMPONENT_TEMPLATE

In [ ]:
get_table("BRIDGEWARE", "ABW_EVENT_COMPONENT_TEMPLATE")

#### ABW_GROUP_ACCESS_PRIVILEGE

In [ ]:
get_table("BRIDGEWARE", "ABW_GROUP_ACCESS_PRIVILEGE")

#### ABW_LIB_MATL_TIMBER

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_TIMBER")

#### ABW_LIB_PS_SHAPE

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_PS_SHAPE")

#### ABW_LIB_STL_SHAPE

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_STL_SHAPE")

#### ABW_LIB_TIMBER_BEAM_SHAPE

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_TIMBER_BEAM_SHAPE")

#### ABW_STLTRUSS_ELEMENT_XSECTION

In [ ]:
get_table("BRIDGEWARE", "ABW_STLTRUSS_ELEMENT_XSECTION")

#### ABW_RATING_TOOL_ANAL_TEMPLATE

In [ ]:
get_table("BRIDGEWARE", "ABW_RATING_TOOL_ANAL_TEMPLATE")

#### ABW_CONC_BEAM_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_CONC_BEAM_DEF")

#### ABW_CULVERT_ALT

In [ ]:
get_table("BRIDGEWARE", "ABW_CULVERT_ALT")

#### ABW_CULVERT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_CULVERT_DEF")

#### ABW_CULVERT_DEF_SEG_ANAL_PT

In [ ]:
get_table("BRIDGEWARE", "ABW_CULVERT_DEF_SEG_ANAL_PT")

#### ABW_CULVERT_DEF_SEGMENT

In [ ]:
get_table("BRIDGEWARE", "ABW_CULVERT_DEF_SEGMENT")

#### ABW_HAUNCH_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_HAUNCH_RANGE")

#### ABW_COMMAND_TRUSS_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_COMMAND_TRUSS_DEF")

#### ABW_DETAILED_STL_TRUSS_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_DETAILED_STL_TRUSS_DEF")

#### ABW_STL_TRUSS_ROLLED_XSECTION

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_TRUSS_ROLLED_XSECTION")

#### ABW_FLOORSYS_STRUCT_DEF_FK

In [ ]:
df = get_table("BRIDGEWARE", "ABW_FLOORSYS_STRUCT_DEF_FK")
df

#### ABW_LIB_DEFAULT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_DEFAULT")
df

#### ABW_SUB_PILE_FOUND_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUB_PILE_FOUND_DEF")
df

#### ABW_SUB_PILE_FOUND_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUB_PILE_FOUND_DEF")
df

#### ABW_SUB_SPREAD_FOUND_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUB_SPREAD_FOUND_DEF")
df

#### ABW_SUB_STRUCT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUB_STRUCT")
df

#### ABW_SUB_STRUCT_ALT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUB_STRUCT_ALT")
df

#### ABW_SUB_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_DEF")

#### ABW_LRFD_FACTOR_SPEC_EDITION

In [ ]:
get_table("BRIDGEWARE", "ABW_LRFD_FACTOR_SPEC_EDITION")

#### ABW_LRFR_FACTOR_SPEC_EDITION

In [ ]:
get_table("BRIDGEWARE", "ABW_LRFR_FACTOR_SPEC_EDITION")

#### ABW_SYS_ANAL_MODULE_HELP_MENU

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_ANAL_MODULE_HELP_MENU")

#### ABW_SUB_PILE_FOUND_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_PILE_FOUND_DEF")

#### ABW_SYS_SPECIFICATION

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_SPECIFICATION")

#### ABW_SYS_SPECIFICATION_EDITION

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_SPECIFICATION_EDITION")

#### ABW_BRIDGE_DIAPHRAGM_DEF_LOC

In [ ]:
get_table("BRIDGEWARE", "ABW_BRIDGE_DIAPHRAGM_DEF_LOC")

#### ABW_CULVERT_DEF_RC_BOX_SEG

In [ ]:
get_table("BRIDGEWARE", "ABW_CULVERT_DEF_RC_BOX_SEG")

#### ABW_MULTIPLE_PRESENCE_FACTORS

In [ ]:
get_table("BRIDGEWARE", "ABW_MULTIPLE_PRESENCE_FACTORS")

#### ABW_NAIL_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_NAIL_DEF")

#### ABW_PIER_BOT_CAP_SEG

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_BOT_CAP_SEG")

#### ABW_PIER_COL_REINF

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_COL_REINF")

#### ABW_PIER_COL_REINF_PAT_DETAIL

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_COL_REINF_PAT_DETAIL")

#### ABW_PIER_COL_SHEAR_REINF

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_COL_SHEAR_REINF")

#### ABW_PIER_DEF_CAP

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_DEF_CAP")

#### ABW_MCB_SEG_DECK_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_SEG_DECK_RANGE")

#### ABW_BRIDGE_REF_LINE

In [ ]:
get_table("BRIDGEWARE", "ABW_BRIDGE_REF_LINE")

#### ABW_MCB_TENDON_PROFILE_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_MCB_TENDON_PROFILE_DEF")

#### ABW_STL_TRUSS_BUILTUP_XSECTION

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_TRUSS_BUILTUP_XSECTION")

#### ABW_MC_BOX_WEB_DISTFACT_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_MC_BOX_WEB_DISTFACT_RANGE")

#### ABW_MBRALT_STLFLNG_LOSS_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_MBRALT_STLFLNG_LOSS_RANGE")

#### ABW_MATL_STL_REINF

In [ ]:
get_table("BRIDGEWARE", "ABW_MATL_STL_REINF")

#### ABW_MATL_STL_STRUCTURAL

In [ ]:
get_table("BRIDGEWARE", "ABW_MATL_STL_STRUCTURAL")

#### ABW_MATL_TIMBER_SAWN

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MATL_TIMBER_SAWN")
df

#### ABW_BRIDGE_SUB_LRFD_DS_LS_VEH

In [ ]:
df = get_table("BRIDGEWARE", "ABW_BRIDGE_SUB_LRFD_DS_LS_VEH")
df

#### ABW_BRIDGE_SUB_LRFD_DS_VEHICLE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_BRIDGE_SUB_LRFD_DS_VEHICLE")
df

#### ABW_MULTI_CELL_BOX_XSEC_CELL

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_XSEC_CELL")
df

#### ABW_MULTI_CELL_BOX_XSECT_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_XSECT_RANGE")
df

#### ABW_FLNG_LAT_BEND_STRESS_LOC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_FLNG_LAT_BEND_STRESS_LOC")
df

#### ABW_MULTI_CELL_BOX_XSECTION

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_XSECTION")
df

#### ABW_TENDON_PROFILE_DEF_WEBDUCT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_TENDON_PROFILE_DEF_WEBDUCT")
df

#### ABW_CULVDEF_SEG_ANAL_PT_RC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_CULVDEF_SEG_ANAL_PT_RC")
df

#### ABW_PS_SHAPE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_PS_SHAPE")
df

#### ABW_DECK_CONC_PANEL

In [ ]:
df = get_table("BRIDGEWARE", "ABW_DECK_CONC_PANEL")
df

#### ABW_SPNG_MBR_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SPNG_MBR_DEF")
df

#### ABW_STL_SHAPE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_STL_SHAPE")
df

#### ABW_STRUCT_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_STRUCT_DEF")
df

#### ABW_SUBSDEF_FOUND_ALT

In [ ]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_FOUND_ALT")

#### ABW_STRIPED_LANE_POSITION

In [ ]:
df = get_table("BRIDGEWARE", "ABW_STRIPED_LANE_POSITION")
df

#### ABW_STRUCT_DEF_REF_LINE

In [ ]:
get_table("BRIDGEWARE", "ABW_STRUCT_DEF_REF_LINE")

#### ABW_STRUCT_DEF_REF_LINE_HORZ

In [ ]:
get_table("BRIDGEWARE", "ABW_STRUCT_DEF_REF_LINE_HORZ")

#### ABW_MBRALT_FLNGANGL_LOSS_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBRALT_FLNGANGL_LOSS_RANGE")
df

#### ABW_MBRALT_FLNGFLAT_LOSS_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBRALT_FLNGFLAT_LOSS_RANGE")
df

#### ABW_MBRALT_STL_TRUSS_ELEMENT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBRALT_STL_TRUSS_ELEMENT")
df

#### ABW_MBRALT_STLBM_LOSS_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBRALT_STLBM_LOSS_RANGE")
df

#### ABW_MBRALT_STLCPLT_LOSS_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBRALT_STLCPLT_LOSS_RANGE")
df

#### ABW_MBRALT_STLWEB_LOSS_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBRALT_STLWEB_LOSS_RANGE")
df

#### ABW_SUBSDEF_BEARING_LINE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUBSDEF_BEARING_LINE")
df

#### ABW_SUBSDEF_BEARING_LOC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUBSDEF_BEARING_LOC")
df

#### ABW_STL_COMPONENT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_STL_COMPONENT")
df

#### ABW_LRFR_LOAD_FACTOR_TABLE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LRFR_LOAD_FACTOR_TABLE")
df

#### ABW_SYS_BRDR_COLUMN

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SYS_BRDR_COLUMN")
df

#### ABW_SYS_BRDR_FOREIGN_KEY

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SYS_BRDR_FOREIGN_KEY")
df

#### ABW_SUPSTRUCTDEF_BRACING

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUPSTRUCTDEF_BRACING")
df

#### ABW_LIB_LRFD_RANGE_APP_VALUE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_LRFD_RANGE_APP_VALUE")
df

#### ABW_LIB_LRFR_FACTOR

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_LRFR_FACTOR")
df

#### ABW_LIB_LRFR_LEGAL_LOADS

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_LRFR_LEGAL_LOADS")
df

#### ABW_LIB_LRFR_LEGAL_LOADS_HAUL

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_LRFR_LEGAL_LOADS_HAUL")
df

#### ABW_LIB_LRFR_LF_TABLE_VALUE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_LRFR_LF_TABLE_VALUE")
df

#### ABW_LIB_LRFR_LOAD_FACTOR

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_LRFR_LOAD_FACTOR")
df

#### ABW_LIB_LRFR_LS

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_LRFR_LS")
df

#### ABW_LIB_LRFR_PERMIT_LIMITED

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_LRFR_PERMIT_LIMITED")
df

#### ABW_LIB_LRFR_PERMIT_ROUTINE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_LRFR_PERMIT_ROUTINE")
df

#### ABW_LIB_MATL_CONC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_MATL_CONC")
df

#### ABW_LIB_MATL_PS_BAR

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_MATL_PS_BAR")
df

#### ABW_LIB_MATL_STL_PS

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_MATL_STL_PS")
df

#### ABW_LIB_MATL_STL_REINF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_MATL_STL_REINF")
df

#### ABW_TIMBER_RECT_BEAM_SHAPE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_TIMBER_RECT_BEAM_SHAPE")
df

#### ABW_TOP_FLNG_LAT_SUP_LOC_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_TOP_FLNG_LAT_SUP_LOC_RANGE")
df

#### ABW_TOP_FLNG_LAT_SUPPORT_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_TOP_FLNG_LAT_SUPPORT_RANGE")
df

#### ABW_TRUSS_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_TRUSS_DEF")
df

#### ABW_SUPER_STRUCT_MBR_SPAN

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_MBR_SPAN")
df

#### ABW_SUPER_STRUCT_SPNG_MBR

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_SPNG_MBR")
df

#### ABW_MULTI_CELL_BOX_BDEF_WEB

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_BDEF_WEB")
df

#### ABW_MCB_SEG_CONC_RAILING_LOC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MCB_SEG_CONC_RAILING_LOC")
df

#### ABW_MCB_SEG_STL_RAILING_LOC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MCB_SEG_STL_RAILING_LOC")
df

#### ABW_SUPER_SUB_STRUCT_INTERFACE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUPER_SUB_STRUCT_INTERFACE")
df

#### ABW_SUPER_ENVIRONMENTAL_LOAD

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUPER_ENVIRONMENTAL_LOAD")
df

#### ABW_SUPER_LL_DIST_SUB_WDLA

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUPER_LL_DIST_SUB_WDLA")
df

#### ABW_SUPER_LL_DIST_SUB_WODLA

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUPER_LL_DIST_SUB_WODLA")
df

#### ABW_MBR_ALT_CONC_DECK_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBR_ALT_CONC_DECK_RANGE")
df

#### ABW_MBR_ALT_DECK_REINF_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBR_ALT_DECK_REINF_RANGE")
df

#### ABW_MBR_ALT_DIAPH_LOC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBR_ALT_DIAPH_LOC")
df

#### ABW_MBR_ALT_GENERIC_DECK_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBR_ALT_GENERIC_DECK_RANGE")
df

#### ABW_MBR_ALT_GLINE_SUPPORT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBR_ALT_GLINE_SUPPORT")
df

#### ABW_MBR_ALT_SHEAR_CONN_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBR_ALT_SHEAR_CONN_RANGE")
df

#### ABW_MBRALT_STLIBM_LOSS_RANGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MBRALT_STLIBM_LOSS_RANGE")
df

#### ABW_LIB_MATL_WELD

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_MATL_WELD")
df

#### ABW_LIB_PS_BOX_BEAM

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_PS_BOX_BEAM")
df

#### ABW_LIB_PS_IBEAM

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_PS_IBEAM")
df

#### PARAMTRS

In [ ]:
df = get_table("BRIDGEWARE", "PARAMTRS")
df

#### USERS

In [ ]:
df = get_table("BRIDGEWARE", "USERS")
df

#### ABW_AGENCY_STREAM_FLOW_EFFECT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_AGENCY_STREAM_FLOW_EFFECT")
df

#### ABW_SLAB_MBR

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SLAB_MBR")
df

#### ABW_SYS_LRFD_RANGE_OF_APP_ROW

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SYS_LRFD_RANGE_OF_APP_ROW")
df

#### ABW_SYS_LRFD_RANGE_APP_LABEL

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SYS_LRFD_RANGE_APP_LABEL")
df

#### ABW_SYS_SUPER_STRUCT_STAGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SYS_SUPER_STRUCT_STAGE")
df

#### ABW_SYS_TABLE_RELATIONSHIPS

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SYS_TABLE_RELATIONSHIPS")
df

#### ABW_SYS_TYPE_CATEGORY

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SYS_TYPE_CATEGORY")
df

#### ABW_MCB_TEND_PROF_DEF_WEBDUCT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MCB_TEND_PROF_DEF_WEBDUCT")
df

#### ABW_TFS_FLOORSYS_STRUCT_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_TFS_FLOORSYS_STRUCT_DEF")
df

#### ABW_TF_FLOORLINE_STRUCT_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_TF_FLOORLINE_STRUCT_DEF")
df

#### ABW_TF_FLOORSYS_STRUCT_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_TF_FLOORSYS_STRUCT_DEF")
df

#### ABW_BRIDGE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_BRIDGE")
df

#### ABW_TRUSS_DEF_COMMAND

In [ ]:
df = get_table("BRIDGEWARE", "ABW_TRUSS_DEF_COMMAND")
df

#### ABW_TRUSS_MBR

In [ ]:
df = get_table("BRIDGEWARE", "ABW_TRUSS_MBR")
df

#### ABW_STL_PLATE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_STL_PLATE")
df

#### ABW_STL_RAILING

In [ ]:
df = get_table("BRIDGEWARE", "ABW_STL_RAILING")
df

#### ABW_STL_RAILING_LOC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_STL_RAILING_LOC")
df

#### ABW_STL_RIVET_BOLT_CONN

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_RIVET_BOLT_CONN")

#### ABW_MULTICELL_BOX_TENDON_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_MULTICELL_BOX_TENDON_RANGE")

#### ABW_ANAL_PT_LRFD_OVERRIDE_CAP

In [ ]:
get_table("BRIDGEWARE", "ABW_ANAL_PT_LRFD_OVERRIDE_CAP")

#### ABW_FSYS_FLOOR_BEAM_MBR_LOC

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_FLOOR_BEAM_MBR_LOC")

#### ABW_FSYS_STRGRP_DIAPH

In [ ]:
get_table("BRIDGEWARE", "ABW_FSYS_STRGRP_DIAPH")

#### ABW_BEAM_SHEAR_REINF_LOC

In [ ]:
get_table("BRIDGEWARE", "ABW_BEAM_SHEAR_REINF_LOC")

#### ABW_SYS_ANALYSIS_MODULE

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_ANALYSIS_MODULE")

#### ABW_SYS_COMPONENT_ERRORS

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_COMPONENT_ERRORS")

#### ABW_BMDEF_ANALPT_LRFD_OVRD_CAP

In [ ]:
get_table("BRIDGEWARE", "ABW_BMDEF_ANALPT_LRFD_OVRD_CAP")

#### ABW_BMDEF_ANALPT_LRFR_OVRD_CAP

In [ ]:
get_table("BRIDGEWARE", "ABW_BMDEF_ANALPT_LRFR_OVRD_CAP")

#### ABW_LIB_LRFD_FACT_SPEC_EDITION

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFD_FACT_SPEC_EDITION")

#### ABW_LIB_LRFR_FACT_SPEC_EDITION

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_LRFR_FACT_SPEC_EDITION")

#### ABW_TRUSS_ELEMENT_XSECTION

In [ ]:
get_table("BRIDGEWARE", "ABW_TRUSS_ELEMENT_XSECTION")

#### ABW_SUBSDEF_SPNG_MBR_BRNG_LOC

In [ ]:
get_table("BRIDGEWARE", "ABW_SUBSDEF_SPNG_MBR_BRNG_LOC")

#### ABW_SUB_STRUCT_DEF_COMPONENT

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_DEF_COMPONENT")

#### ABW_SUB_STRUCT_DEF_PILE

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_DEF_PILE")

#### ABW_SUB_STRUCT_FK

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_FK")

#### ABW_SUB_STRUCT_STL_PILE_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_SUB_STRUCT_STL_PILE_DEF")

#### ABW_PIER_SUBSDEF_COLUMN_XSEC

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_SUBSDEF_COLUMN_XSEC")

#### ABW_PIER_SUB_STRUCT

In [ ]:
get_table("BRIDGEWARE", "ABW_PIER_SUB_STRUCT")

#### ABW_FS_FLOORLINE_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_FS_FLOORLINE_STRUCT_DEF")

#### ABW_FS_FLOORSYS_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_FS_FLOORSYS_STRUCT_DEF")

#### ABW_GFS_FLOORLINE_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_GFS_FLOORLINE_STRUCT_DEF")

#### ABW_GFS_FLOORSYS_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_GFS_FLOORSYS_STRUCT_DEF")

#### ABW_GF_FLOORSYS_STRUCT_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_GF_FLOORSYS_STRUCT_DEF")

#### ABW_GIRDER_MBR

In [ ]:
get_table("BRIDGEWARE", "ABW_GIRDER_MBR")

#### ABW_GIRDER_SYS_STRUCT_DEF_FKABW_GIRDER_SYS_STRUCT_DEF_FK

In [ ]:
get_table("BRIDGEWARE", "ABW_GIRDER_SYS_STRUCT_DEF_FK")

#### ABW_STL_FLNG_LOSS_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_FLNG_LOSS_RANGE")

#### ABW_STL_IBEAM_LOSS_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_IBEAM_LOSS_RANGE")

#### ABW_STL_NON_DETAILED_BEAM_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_NON_DETAILED_BEAM_DEF")

#### ABW_STL_PLATE_IBEAM_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_PLATE_IBEAM_DEF")

#### ABW_STL_ROLLED_IBEAM_DEF

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_ROLLED_IBEAM_DEF")

#### ABW_STL_SCH_CPLATE_LOSS_RANGE

In [ ]:
get_table("BRIDGEWARE", "ABW_STL_SCH_CPLATE_LOSS_RANGE")

#### ABW_GENPREF_TEMPLATE

In [ ]:
get_table("BRIDGEWARE", "ABW_GENPREF_TEMPLATE")

#### ABW_LIB_METAL_BOX_CULVERT

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_METAL_BOX_CULVERT")

#### ABW_BMDEF_SUPPORT

In [ ]:
get_table("BRIDGEWARE", "ABW_BMDEF_SUPPORT")

#### ABW_BRIDGE_FK

In [ ]:
get_table("BRIDGEWARE", "ABW_BRIDGE_FK")

#### ABW_SPNG_MBR_ALT_COMPONENT

In [ ]:
get_table("BRIDGEWARE", "ABW_SPNG_MBR_ALT_COMPONENT")

#### ABW_ANAL_PT_LRFR_OVERRIDE_CAP

In [ ]:
get_table("BRIDGEWARE", "ABW_ANAL_PT_LRFR_OVERRIDE_CAP")

#### ABW_LIB_MATL_TIMBER_SAWN

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_MATL_TIMBER_SAWN")

#### ABW_LIB_SPEC

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_SPEC")

#### ABW_LIB_SPEC_ARTICLE

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_SPEC_ARTICLE")

#### ABW_LIB_STANDARD_LOAD_GROUP

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_STANDARD_LOAD_GROUP")

#### ABW_LIB_SUB_LRFD_DS_LS_VEHICLE

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_SUB_LRFD_DS_LS_VEHICLE")

#### ABW_LIB_SUB_LRFD_DS_VEHICLE

In [ ]:
get_table("BRIDGEWARE", "ABW_LIB_SUB_LRFD_DS_VEHICLE")

#### ABW_SYS_ANAL_MOD_SPEC_EDITION

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_ANAL_MOD_SPEC_EDITION")

#### ABW_SYS_ANAL_MODULE_FEATURE

In [ ]:
get_table("BRIDGEWARE", "ABW_SYS_ANAL_MODULE_FEATURE")

#### ABW_BOT_FLANGE_LAT_BRACING_LOC

In [ ]:
get_table("BRIDGEWARE", "ABW_BOT_FLANGE_LAT_BRACING_LOC")

#### ABW_TENDON_DEF_WEB_JFORCE

In [ ]:
get_table("BRIDGEWARE", "ABW_TENDON_DEF_WEB_JFORCE")

#### ABW_LFR_FACTOR

In [ ]:
get_table("BRIDGEWARE", "ABW_LFR_FACTOR")

#### ABW_LFR_FACTOR_SPEC_EDITION

In [ ]:
get_table("BRIDGEWARE", "ABW_LFR_FACTOR_SPEC_EDITION")

#### ABW_LFR_LOADING

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LFR_LOADING")
df

#### ABW_ANAL_PT_TIMBER

In [ ]:
df = get_table("BRIDGEWARE", "ABW_ANAL_PT_TIMBER")
df

#### ABW_ANALYSIS_EVENT_TEMPLATE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_ANALYSIS_EVENT_TEMPLATE")
df

#### ABW_BAR_MARK_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_BAR_MARK_DEF")
df

#### ABW_ANAL_PT_STL

In [ ]:
df = get_table("BRIDGEWARE", "ABW_ANAL_PT_STL")
df

#### ABW_GUSSET_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_GUSSET_DEF")
df

#### ABW_FSYS_STRGRPDEF_TEMPLATE_FK

In [ ]:
df = get_table("BRIDGEWARE", "ABW_FSYS_STRGRPDEF_TEMPLATE_FK")
df

#### ABW_FSYS_STRINGER_UNIT_LAYOUT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_FSYS_STRINGER_UNIT_LAYOUT")
df

#### ABW_ANAL_PT_COMPONENT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_ANAL_PT_COMPONENT")
df

#### ABW_BMDEF_COMPONENT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_BMDEF_COMPONENT")
df

#### ABW_SUPER_STRUCT_SPNG_MBR_ALT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_SPNG_MBR_ALT")
df

#### ABW_SYS_ANAL_MODULE_COMPONENT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SYS_ANAL_MODULE_COMPONENT")
df

#### ABW_SYS_BRIDGE_EXPLORER_COLUMN

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SYS_BRIDGE_EXPLORER_COLUMN")
df

#### ABW_USER_PROFILE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_USER_PROFILE")
df

#### ABW_STL_ROLLED_SHAPE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_STL_ROLLED_SHAPE")
df

#### ABW_SYS_UNIT_CATEGORY

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SYS_UNIT_CATEGORY")
df

#### ABW_STL_SPLICE_LOC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_STL_SPLICE_LOC")
df

#### ABW_SUPER_STRUCT_COMPONENT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUPER_STRUCT_COMPONENT")
df

#### ABW_AGENCY_BRIDGE_DESIGN_PARAM

In [ ]:
df = get_table("BRIDGEWARE", "ABW_AGENCY_BRIDGE_DESIGN_PARAM")
df

#### ABW_ADV_CONC_BEAM_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_ADV_CONC_BEAM_DEF")
df

#### ABW_CULVDEF_RCBOX_SEG_REINPROF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_CULVDEF_RCBOX_SEG_REINPROF")
df

#### ABW_ADV_CONC_PT_BEAM_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_ADV_CONC_PT_BEAM_DEF")
df

#### ABW_ANAL_PT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_ANAL_PT")
df

#### ABW_ANAL_PT_PS_CONC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_ANAL_PT_PS_CONC")
df

#### ABW_BMDEF_ANAL_PT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_BMDEF_ANAL_PT")
df

#### ABW_UNIT_TOLERANCE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_UNIT_TOLERANCE")
df

#### ABW_MULTI_CELL_BOX_MBR

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MULTI_CELL_BOX_MBR")
df

#### ABW_MCELL_BOX_STRUCT_DEF_FK

In [ ]:
df = get_table("BRIDGEWARE", "ABW_MCELL_BOX_STRUCT_DEF_FK")
df

#### ABW_WELD_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_WELD_DEF")
df

#### ABW_WIND_LOAD

In [ ]:
df = get_table("BRIDGEWARE", "ABW_WIND_LOAD")
df

#### ABW_EVENT_LRFD_FACTOR

In [ ]:
df = get_table("BRIDGEWARE", "ABW_EVENT_LRFD_FACTOR")
df

#### ABW_EVENT_LRFD_LS

In [ ]:
df = get_table("BRIDGEWARE", "ABW_EVENT_LRFD_LS")
df

#### ABW_SPNG_MBR_ALT_EVENTS

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SPNG_MBR_ALT_EVENTS")
df

#### ABW_RESULTS_DL_CASE

In [ ]:
df = get_table("BRIDGEWARE", "ABW_RESULTS_DL_CASE")
df

#### ABW_INTEREST_PT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_INTEREST_PT")
df

#### ABW_SPNG_MBR_ALT_EVENTS_REPORT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SPNG_MBR_ALT_EVENTS_REPORT")
df

#### ABW_BRIDGE_WIND_EFFECT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_BRIDGE_WIND_EFFECT")
df

#### ABW_CONC_BMDEF_VOID_SEC_PAT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_CONC_BMDEF_VOID_SEC_PAT")
df

#### ABW_AGENCY_ENVIRONMENTAL_PARAM

In [ ]:
df = get_table("BRIDGEWARE", "ABW_AGENCY_ENVIRONMENTAL_PARAM")
df

#### ABW_FSYS_GIRDER_MBR_LOC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_FSYS_GIRDER_MBR_LOC")
df

#### ABW_FSYS_STRINGER_GROUP_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_FSYS_STRINGER_GROUP_DEF")
df

#### ABW_FSYS_STRINGER_MBR_LOC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_FSYS_STRINGER_MBR_LOC")
df

#### ABW_FSYS_TRUSS_MBR_LOC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_FSYS_TRUSS_MBR_LOC")
df

#### ABW_PIER_RC_COL_CIRC_XSEC

In [ ]:
df = get_table("BRIDGEWARE", "ABW_PIER_RC_COL_CIRC_XSEC")
df

#### ABW_COMMAND_STL_TRUSS_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_COMMAND_STL_TRUSS_DEF")
df

#### ABW_SUP_STR_THR_D_NODE_GEN_TOL

In [ ]:
df = get_table("BRIDGEWARE", "ABW_SUP_STR_THR_D_NODE_GEN_TOL")
df

#### ABW_PROJECT

In [ ]:
df = get_table("BRIDGEWARE", "ABW_PROJECT")
df

#### ABW_PROJECT_ENGINEER

In [ ]:
df = get_table("BRIDGEWARE", "ABW_PROJECT_ENGINEER")
df

#### ABW_PROJECT_GROUP

In [ ]:
df = get_table("BRIDGEWARE", "ABW_PROJECT_GROUP")
df

#### ABW_PROJECT_GROUP_DETAIL

In [ ]:
df = get_table("BRIDGEWARE", "ABW_PROJECT_GROUP_DETAIL")
df

#### ABW_PS_PRECAST_BOX_BEAM_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_PS_PRECAST_BOX_BEAM_DEF")
df

#### ABW_PS_PRECAST_IBEAM_DEF

In [ ]:
df = get_table("BRIDGEWARE", "ABW_PS_PRECAST_IBEAM_DEF")
df

#### ABW_LIB_LRFD_RANGE_OF_APP

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_LRFD_RANGE_OF_APP")
df

#### ABW_LIB_LRFD_RANGE_OF_APP_ROW

In [ ]:
df = get_table("BRIDGEWARE", "ABW_LIB_LRFD_RANGE_OF_APP_ROW")
df

## Appendix